In [1]:
# Cell 01 — Architecture-B configuration
# DECISION: ADD

import json
import hashlib
from pathlib import Path


# =========================================================================
# RUN MODE
# =========================================================================

RUN_MODE = "production"
# Change to "production" after the complete FAST pipeline is verified.


assert RUN_MODE in {
    "fast",
    "production",
}


# =========================================================================
# ARCHITECTURE B IDENTITY
# =========================================================================

EXPERIMENT_ID = (
    "CS781-P1-B-encoder-decoder-transformer-v1"
)

ARCHITECTURE_NAME = (
    "Architecture-B-Encoder-Decoder-Transformer"
)

ARCHITECTURE_VERSION = (
    "B-v1-dual-mode-early-stop"
)


# =========================================================================
# MODEL CONFIGURATION
# =========================================================================

if RUN_MODE == "fast":

    MODEL_CONFIG = {

        "vocab_size": 50_000,

        "src_context_length": 256,

        "tgt_context_length": 256,

        "d_model": 256,

        "n_encoder_layers": 2,

        "n_decoder_layers": 2,

        "n_heads": 4,

        "d_ff": 1_024,

        "dropout": 0.10,

        "activation": "gelu",

        "normalization": "layernorm",

        "pre_norm": True,

        "learned_positions": True,

        "tie_embeddings": True,

        "bias": True,
    }

else:

    MODEL_CONFIG = {

        "vocab_size": 50_000,

        "src_context_length": 1_024,

        "tgt_context_length": 1_024,

        "d_model": 768,

        "n_encoder_layers": 6,

        "n_decoder_layers": 6,

        "n_heads": 12,

        "d_ff": 3_072,

        "dropout": 0.10,

        "activation": "gelu",

        "normalization": "layernorm",

        "pre_norm": True,

        "learned_positions": True,

        "tie_embeddings": True,

        "bias": True,
    }


# =========================================================================
# TRAINING CONFIGURATION
# =========================================================================

if RUN_MODE == "fast":

    TRAIN_CONFIG = {

        "train_token_budget": 16_384,

        "validation_token_budget": 8_192,

        "micro_batch_size": 2,

        "gradient_accumulation_steps": 2,

        "eval_interval_steps": 5,

        "checkpoint_interval_steps": 5,

        "max_steps": 20,

        "early_stopping": True,

        "min_best_checkpoints": 2,

        "max_best_checkpoints": 4,

        "patience": 2,

        "min_delta": 0.01,

        "min_steps_before_plateau_stop": 5,
    }

else:

    TRAIN_CONFIG = {

        # Hard training ceiling.
        "train_token_budget": 100_000_000,

        "validation_token_budget": 5_000_000,

        "micro_batch_size": 1,

        "gradient_accumulation_steps": 8,

        "eval_interval_steps": 500,

        "checkpoint_interval_steps": 500,

        "max_steps": 0,   # computed after effective tokens/step

        "early_stopping": True,

        "min_best_checkpoints": 8,

        "max_best_checkpoints": 12,

        "patience": 4,

        "min_delta": 0.005,

        "min_steps_before_plateau_stop": 2_000,
    }


# =========================================================================
# OPTIMIZER CONFIGURATION
# =========================================================================

OPTIMIZER_CONFIG = {

    "name": "AdamW",

    "learning_rate": (
        3e-4
        if RUN_MODE == "fast"
        else 3e-4
    ),

    "min_learning_rate": (
        3e-5
        if RUN_MODE == "fast"
        else 3e-5
    ),

    "weight_decay": 0.10,

    "warmup_fraction": 0.05,

    "gradient_clip_norm": 1.0,
}


# =========================================================================
# DATA MIX
# =========================================================================

SOURCE_NAMES = (
    "acts",
    "high_courts",
    "supreme_court",
    "tribunals",
)

SOURCE_PROBABILITIES = {
    "acts": 0.25,
    "high_courts": 0.25,
    "supreme_court": 0.25,
    "tribunals": 0.25,
}


# =========================================================================
# OUTPUT DIRECTORIES
# =========================================================================

ARCHITECTURE_B_ROOT = Path(
    "/kaggle/working/architecture_B"
)

CHECKPOINTS_DIR = (
    ARCHITECTURE_B_ROOT / "checkpoints"
)

METRICS_DIR = (
    ARCHITECTURE_B_ROOT / "metrics"
)

CONFIGS_DIR = (
    ARCHITECTURE_B_ROOT / "configs"
)

PREDICTIONS_DIR = (
    ARCHITECTURE_B_ROOT / "predictions"
)


for directory in [
    ARCHITECTURE_B_ROOT,
    CHECKPOINTS_DIR,
    METRICS_DIR,
    CONFIGS_DIR,
    PREDICTIONS_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# =========================================================================
# COMPUTE MAX STEPS
# =========================================================================

effective_tokens_per_step = (
    TRAIN_CONFIG["micro_batch_size"]
    * TRAIN_CONFIG["gradient_accumulation_steps"]
)

# B uses source/target sequences, so the exact number of tokens
# consumed per optimizer step will be finalized in the data/model cell.
# For now, keep the explicit production ceiling and calculate a
# conservative step ceiling from target-context length.

if RUN_MODE == "production":

    tokens_per_sequence_pair = (
        MODEL_CONFIG["src_context_length"]
        + MODEL_CONFIG["tgt_context_length"]
    )

    effective_tokens_per_step = (
        TRAIN_CONFIG["micro_batch_size"]
        * TRAIN_CONFIG["gradient_accumulation_steps"]
        * tokens_per_sequence_pair
    )

    TRAIN_CONFIG["max_steps"] = (
        (
            TRAIN_CONFIG["train_token_budget"]
            + effective_tokens_per_step
            - 1
        )
        // effective_tokens_per_step
    )


# =========================================================================
# CONFIG FINGERPRINT
# =========================================================================

fingerprint_payload = {

    "experiment_id": EXPERIMENT_ID,

    "architecture_name": ARCHITECTURE_NAME,

    "architecture_version": (
        ARCHITECTURE_VERSION
    ),

    "run_mode": RUN_MODE,

    "model_config": MODEL_CONFIG,

    "train_config": TRAIN_CONFIG,

    "optimizer_config": OPTIMIZER_CONFIG,

    "source_probabilities": (
        SOURCE_PROBABILITIES
    ),
}


fingerprint_json = json.dumps(
    fingerprint_payload,
    sort_keys=True,
    ensure_ascii=False,
)

CONFIG_FINGERPRINT = hashlib.sha256(
    fingerprint_json.encode("utf-8")
).hexdigest()


# =========================================================================
# SAVE CONFIG
# =========================================================================

CONFIG_MANIFEST = {
    **fingerprint_payload,
    "config_fingerprint": CONFIG_FINGERPRINT,
}

CONFIG_PATH = (
    CONFIGS_DIR
    / "architecture_B_config.json"
)

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        CONFIG_MANIFEST,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# SUMMARY
# =========================================================================

print("✅ Architecture-B configuration ready.")
print()
print(f"Run mode             : {RUN_MODE}")
print(f"Experiment ID        : {EXPERIMENT_ID}")
print(f"Architecture         : {ARCHITECTURE_NAME}")
print(f"Version              : {ARCHITECTURE_VERSION}")
print()
print("Model:")
print(
    f"  d_model            : "
    f"{MODEL_CONFIG['d_model']}"
)
print(
    f"  encoder layers     : "
    f"{MODEL_CONFIG['n_encoder_layers']}"
)
print(
    f"  decoder layers     : "
    f"{MODEL_CONFIG['n_decoder_layers']}"
)
print(
    f"  heads              : "
    f"{MODEL_CONFIG['n_heads']}"
)
print(
    f"  FFN                : "
    f"{MODEL_CONFIG['d_ff']}"
)
print(
    f"  source context     : "
    f"{MODEL_CONFIG['src_context_length']}"
)
print(
    f"  target context     : "
    f"{MODEL_CONFIG['tgt_context_length']}"
)
print()
print("Training:")
print(
    f"  token budget       : "
    f"{TRAIN_CONFIG['train_token_budget']:,}"
)
print(
    f"  max steps          : "
    f"{TRAIN_CONFIG['max_steps']}"
)
print(
    f"  min best checkpoints: "
    f"{TRAIN_CONFIG['min_best_checkpoints']}"
)
print(
    f"  max best checkpoints: "
    f"{TRAIN_CONFIG['max_best_checkpoints']}"
)
print(
    f"  patience           : "
    f"{TRAIN_CONFIG['patience']}"
)
print(
    f"  min delta          : "
    f"{TRAIN_CONFIG['min_delta']}"
)
print()
print(
    f"Config fingerprint   : "
    f"{CONFIG_FINGERPRINT}"
)
print(
    f"Config file          : "
    f"{CONFIG_PATH}"
)

✅ Architecture-B configuration ready.

Run mode             : production
Experiment ID        : CS781-P1-B-encoder-decoder-transformer-v1
Architecture         : Architecture-B-Encoder-Decoder-Transformer
Version              : B-v1-dual-mode-early-stop

Model:
  d_model            : 768
  encoder layers     : 6
  decoder layers     : 6
  heads              : 12
  FFN                : 3072
  source context     : 1024
  target context     : 1024

Training:
  token budget       : 100,000,000
  max steps          : 6104
  min best checkpoints: 8
  max best checkpoints: 12
  patience           : 4
  min delta          : 0.005

Config fingerprint   : 664e5d82e753897da2727ead43b9294907235d514880e60bffa0d9cf7609cc9e
Config file          : /kaggle/working/architecture_B/configs/architecture_B_config.json


In [2]:
# Cell 02 — Runtime, device, precision, and reproducibility
# DECISION: ADD

import os
import random
import numpy as np
import torch


print("⚙️ Initializing Architecture-B runtime...")


# =========================================================================
# REPRODUCIBILITY
# =========================================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


if hasattr(torch, "mps") and torch.backends.mps.is_available():

    try:
        torch.mps.manual_seed(SEED)
    except Exception:
        pass


# =========================================================================
# DEVICE SELECTION
# =========================================================================

if torch.cuda.is_available():

    DEVICE = torch.device("cuda")

elif (
    hasattr(torch, "mps")
    and torch.backends.mps.is_available()
):

    DEVICE = torch.device("mps")

else:

    DEVICE = torch.device("cpu")


# =========================================================================
# PRECISION POLICY
# =========================================================================
# Preference:
#   CUDA + BF16 support  -> BF16 AMP
#   CUDA without BF16   -> FP16 AMP + GradScaler
#   MPS / CPU            -> FP32
#
# This keeps the exact same notebook compatible across Kaggle/Colab
# GPU environments without hard-coding T4-specific behavior.

USE_AMP = False
AMP_DTYPE = torch.float32
AMP_MODE = "fp32"
USE_GRAD_SCALER = False


if DEVICE.type == "cuda":

    bf16_supported = False

    try:
        bf16_supported = torch.cuda.is_bf16_supported(
            including_emulation=False
        )
    except TypeError:
        try:
            bf16_supported = torch.cuda.is_bf16_supported()
        except Exception:
            bf16_supported = False
    except Exception:
        bf16_supported = False


    if bf16_supported:

        USE_AMP = True
        AMP_DTYPE = torch.bfloat16
        AMP_MODE = "bf16"
        USE_GRAD_SCALER = False

    else:

        USE_AMP = True
        AMP_DTYPE = torch.float16
        AMP_MODE = "fp16"
        USE_GRAD_SCALER = True


elif DEVICE.type == "mps":

    # Keep MPS in FP32 for maximum compatibility.
    USE_AMP = False
    AMP_DTYPE = torch.float32
    AMP_MODE = "fp32-mps"
    USE_GRAD_SCALER = False


else:

    USE_AMP = False
    AMP_DTYPE = torch.float32
    AMP_MODE = "fp32-cpu"
    USE_GRAD_SCALER = False


# =========================================================================
# CUDA / PERFORMANCE SETTINGS
# =========================================================================

if DEVICE.type == "cuda":

    # Do not force deterministic CUDA algorithms.
    # Some deterministic kernels are significantly slower or unavailable.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

    # TF32 is useful on Ampere+ GPUs but harmlessly disabled on older GPUs.
    if hasattr(torch.backends.cuda.matmul, "allow_tf32"):

        gpu_major = torch.cuda.get_device_capability(
            DEVICE
        )[0]

        torch.backends.cuda.matmul.allow_tf32 = (
            gpu_major >= 8
        )

    if hasattr(torch.backends.cudnn, "allow_tf32"):

        gpu_major = torch.cuda.get_device_capability(
            DEVICE
        )[0]

        torch.backends.cudnn.allow_tf32 = (
            gpu_major >= 8
        )


# =========================================================================
# THREAD SETTINGS
# =========================================================================

if DEVICE.type == "cpu":

    cpu_count = os.cpu_count() or 4

    torch.set_num_threads(
        max(
            1,
            min(cpu_count, 8),
        )
    )


# =========================================================================
# CUDA DEVICE INFORMATION
# =========================================================================

GPU_NAME = None
GPU_MEMORY_GB = None
CUDA_CAPABILITY = None


if DEVICE.type == "cuda":

    GPU_INDEX = torch.cuda.current_device()

    GPU_NAME = torch.cuda.get_device_name(
        GPU_INDEX
    )

    GPU_MEMORY_GB = round(
        torch.cuda.get_device_properties(
            GPU_INDEX
        ).total_memory
        / (1024 ** 3),
        2,
    )

    CUDA_CAPABILITY = (
        torch.cuda.get_device_capability(
            GPU_INDEX
        )
    )


# =========================================================================
# RUNTIME MANIFEST
# =========================================================================

RUNTIME_MANIFEST = {

    "experiment_id":
        EXPERIMENT_ID,

    "architecture_name":
        ARCHITECTURE_NAME,

    "architecture_version":
        ARCHITECTURE_VERSION,

    "run_mode":
        RUN_MODE,

    "seed":
        SEED,

    "device":
        str(DEVICE),

    "device_type":
        DEVICE.type,

    "amp_enabled":
        USE_AMP,

    "amp_mode":
        AMP_MODE,

    "amp_dtype":
        str(AMP_DTYPE),

    "grad_scaler":
        USE_GRAD_SCALER,

    "gpu_name":
        GPU_NAME,

    "gpu_memory_gb":
        GPU_MEMORY_GB,

    "cuda_capability":
        CUDA_CAPABILITY,

    "pytorch_version":
        torch.__version__,
}


RUNTIME_MANIFEST_PATH = (
    CONFIGS_DIR
    / "architecture_B_runtime.json"
)

with open(
    RUNTIME_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    import json

    json.dump(
        RUNTIME_MANIFEST,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# SUMMARY
# =========================================================================

print()
print("✅ Architecture-B runtime initialized.")
print()

print(
    f"Run mode        : {RUN_MODE}"
)

print(
    f"Device          : {DEVICE}"
)

print(
    f"AMP enabled     : {USE_AMP}"
)

print(
    f"AMP mode        : {AMP_MODE}"
)

print(
    f"AMP dtype       : {AMP_DTYPE}"
)

print(
    f"GradScaler      : {USE_GRAD_SCALER}"
)

if GPU_NAME is not None:

    print(
        f"GPU             : {GPU_NAME}"
    )

    print(
        f"GPU memory      : {GPU_MEMORY_GB} GB"
    )

    print(
        f"CUDA capability : {CUDA_CAPABILITY}"
    )

print(
    f"PyTorch         : {torch.__version__}"
)

print(
    f"Seed            : {SEED}"
)

print(
    f"Runtime manifest: {RUNTIME_MANIFEST_PATH}"
)

⚙️ Initializing Architecture-B runtime...

✅ Architecture-B runtime initialized.

Run mode        : production
Device          : cuda
AMP enabled     : True
AMP mode        : fp16
AMP dtype       : torch.float16
GradScaler      : True
GPU             : Tesla T4
GPU memory      : 14.56 GB
CUDA capability : (7, 5)
PyTorch         : 2.10.0+cu128
Seed            : 42
Runtime manifest: /kaggle/working/architecture_B/configs/architecture_B_runtime.json


In [3]:
# Cell 03 — Resolve shared tokenizer and Phase-1 tokenized corpus
# DECISION: UPDATE

import json
import os
from pathlib import Path

import numpy as np
from tokenizers import Tokenizer


print("📚 Resolving Architecture-B shared tokenizer and tokenized corpus...")


# =========================================================================
# EXPECTED FILES
# =========================================================================

EXPECTED_SHARD_NAMES = {
    "acts_train.bin",
    "acts_val.bin",
    "high_courts_train.bin",
    "high_courts_val.bin",
    "supreme_court_train.bin",
    "supreme_court_val.bin",
    "tribunals_train.bin",
    "tribunals_val.bin",
}

EXPECTED_TRAIN_NAMES = {
    "acts_train.bin",
    "high_courts_train.bin",
    "supreme_court_train.bin",
    "tribunals_train.bin",
}

EXPECTED_VAL_NAMES = {
    "acts_val.bin",
    "high_courts_val.bin",
    "supreme_court_val.bin",
    "tribunals_val.bin",
}


# =========================================================================
# KAGGLE DATASET MOUNT
# =========================================================================

KAGGLE_INPUT_ROOT = Path(
    "/kaggle/input"
)

PREFERRED_DATASET_ROOT = (
    KAGGLE_INPUT_ROOT
    / "cs781-phase1-tokenized-corpus"
)


# =========================================================================
# HELPER — FIND DATASET STRUCTURE
# =========================================================================

def find_tokenized_dataset(root):
    """
    Search recursively for:

        tokenizer/tokenizer.json

    and all expected tokenized shards.

    Returns:
        dataset_root,
        tokenizer_path,
        tokenized_dir

    or None.
    """

    root = Path(root)

    if not root.exists():
        return None

    tokenizer_candidates = []

    # -------------------------------------------------------------
    # Recursive tokenizer discovery.
    # -------------------------------------------------------------
    for tokenizer_path in root.rglob(
        "tokenizer.json"
    ):

        if tokenizer_path.is_file():

            tokenizer_candidates.append(
                tokenizer_path
            )

    # -------------------------------------------------------------
    # Test each tokenizer candidate.
    # -------------------------------------------------------------
    for tokenizer_path in tokenizer_candidates:

        tokenizer_dir = (
            tokenizer_path.parent
        )

        # Most expected structure:
        #
        # .../
        #   tokenizer/tokenizer.json
        #   tokenized/*.bin
        #
        tokenized_dir = (
            tokenizer_dir.parent
            / "tokenized"
        )

        if not tokenized_dir.is_dir():

            # More defensive fallback:
            # search near the tokenizer for a directory
            # containing the expected shards.

            possible_tokenized_dirs = []

            search_root = (
                tokenizer_dir.parent
            )

            for candidate_dir in [
                search_root / "tokenized",
                search_root,
            ]:

                if candidate_dir.is_dir():

                    possible_tokenized_dirs.append(
                        candidate_dir
                    )

            # Also inspect immediate descendants.
            if search_root.is_dir():

                try:

                    possible_tokenized_dirs.extend(
                        [
                            p
                            for p in search_root.iterdir()
                            if p.is_dir()
                        ]
                    )

                except PermissionError:
                    pass

            found_candidate = None

            for candidate_dir in possible_tokenized_dirs:

                present = {
                    p.name
                    for p in candidate_dir.iterdir()
                    if p.is_file()
                }

                if EXPECTED_SHARD_NAMES.issubset(
                    present
                ):

                    found_candidate = candidate_dir
                    break

            if found_candidate is None:
                continue

            tokenized_dir = found_candidate

        # ---------------------------------------------------------
        # Check expected shards.
        # ---------------------------------------------------------
        present_shards = {
            p.name
            for p in tokenized_dir.iterdir()
            if p.is_file()
        }

        if EXPECTED_SHARD_NAMES.issubset(
            present_shards
        ):

            # Dataset root is the directory containing tokenizer/
            # and tokenized/ in the canonical layout.
            dataset_root = (
                tokenizer_dir.parent
            )

            return (
                dataset_root,
                tokenizer_path,
                tokenized_dir,
            )

    return None


# =========================================================================
# SEARCH PREFERRED DATASET FIRST
# =========================================================================

dataset_result = find_tokenized_dataset(
    PREFERRED_DATASET_ROOT
)


# =========================================================================
# FALLBACK — SEARCH ALL KAGGLE DATASETS
# =========================================================================

if dataset_result is None:

    assert KAGGLE_INPUT_ROOT.exists(), (
        f"Kaggle input directory not found: "
        f"{KAGGLE_INPUT_ROOT}"
    )

    print(
        "ℹ️ Direct dataset layout not found; "
        "searching recursively under /kaggle/input ..."
    )

    dataset_result = find_tokenized_dataset(
        KAGGLE_INPUT_ROOT
    )


assert dataset_result is not None, (
    "\nCould not locate the shared Phase-1 tokenized corpus.\n\n"
    "A Kaggle dataset is expected to contain:\n"
    "  tokenizer/tokenizer.json\n"
    "  tokenized/acts_train.bin\n"
    "  tokenized/acts_val.bin\n"
    "  tokenized/high_courts_train.bin\n"
    "  tokenized/high_courts_val.bin\n"
    "  tokenized/supreme_court_train.bin\n"
    "  tokenized/supreme_court_val.bin\n"
    "  tokenized/tribunals_train.bin\n"
    "  tokenized/tribunals_val.bin\n"
)


# =========================================================================
# RESOLVE PATHS
# =========================================================================

(
    TOKENIZED_DATASET_ROOT,
    TOKENIZER_PATH,
    TOKENIZED_DIR,
) = dataset_result


METADATA_DIR = (
    TOKENIZED_DATASET_ROOT / "metadata"
)

TOKENIZER_METADATA_PATH = (
    METADATA_DIR / "tokenizer_metadata.json"
)

DATASET_METADATA_PATH = (
    METADATA_DIR / "dataset_metadata.json"
)

TOKENIZATION_MANIFEST_PATH = (
    METADATA_DIR / "tokenization_manifest.json"
)


# =========================================================================
# VERIFY TOKENIZER
# =========================================================================

assert TOKENIZER_PATH.is_file(), (
    f"Tokenizer not found:\n"
    f"{TOKENIZER_PATH}"
)

TOKENIZER = Tokenizer.from_file(
    str(TOKENIZER_PATH)
)

TOKENIZER_VOCAB_SIZE = (
    TOKENIZER.get_vocab_size()
)

assert TOKENIZER_VOCAB_SIZE > 0, (
    "Tokenizer vocabulary is empty."
)

assert (
    TOKENIZER_VOCAB_SIZE
    == MODEL_CONFIG["vocab_size"]
), (
    "Tokenizer vocabulary size does not match "
    "Architecture-B model configuration.\n"
    f"Tokenizer : {TOKENIZER_VOCAB_SIZE}\n"
    f"Model     : {MODEL_CONFIG['vocab_size']}"
)


# =========================================================================
# RESOLVE SHARDS
# =========================================================================

TRAIN_SHARD_PATHS = {
    "acts":
        TOKENIZED_DIR / "acts_train.bin",

    "high_courts":
        TOKENIZED_DIR / "high_courts_train.bin",

    "supreme_court":
        TOKENIZED_DIR / "supreme_court_train.bin",

    "tribunals":
        TOKENIZED_DIR / "tribunals_train.bin",
}


VAL_SHARD_PATHS = {
    "acts":
        TOKENIZED_DIR / "acts_val.bin",

    "high_courts":
        TOKENIZED_DIR / "high_courts_val.bin",

    "supreme_court":
        TOKENIZED_DIR / "supreme_court_val.bin",

    "tribunals":
        TOKENIZED_DIR / "tribunals_val.bin",
}


# =========================================================================
# VERIFY ALL SHARDS
# =========================================================================

required_shards = [
    *TRAIN_SHARD_PATHS.values(),
    *VAL_SHARD_PATHS.values(),
]

missing_shards = [
    str(path)
    for path in required_shards
    if not path.is_file()
]

assert not missing_shards, (
    "Missing required tokenized shards:\n"
    + "\n".join(
        f"  - {path}"
        for path in missing_shards
    )
)


# =========================================================================
# LOAD OPTIONAL METADATA
# =========================================================================

tokenizer_metadata = {}

if TOKENIZER_METADATA_PATH.is_file():

    with open(
        TOKENIZER_METADATA_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        tokenizer_metadata = json.load(f)


dataset_metadata = {}

if DATASET_METADATA_PATH.is_file():

    with open(
        DATASET_METADATA_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        dataset_metadata = json.load(f)


tokenization_manifest = {}

if TOKENIZATION_MANIFEST_PATH.is_file():

    with open(
        TOKENIZATION_MANIFEST_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        tokenization_manifest = json.load(f)


# =========================================================================
# OPEN MEMMAPS LAZILY
# =========================================================================

TRAIN_MEMMAPS = {}

VAL_MEMMAPS = {}


for source_name, path in TRAIN_SHARD_PATHS.items():

    TRAIN_MEMMAPS[source_name] = np.memmap(
        path,
        dtype=np.uint16,
        mode="r",
    )


for source_name, path in VAL_SHARD_PATHS.items():

    VAL_MEMMAPS[source_name] = np.memmap(
        path,
        dtype=np.uint16,
        mode="r",
    )


# =========================================================================
# TINY DATA VALIDATION PROBE
# =========================================================================
# Never scan the complete corpus.

for source_name in SOURCE_NAMES:

    train_map = TRAIN_MEMMAPS[
        source_name
    ]

    val_map = VAL_MEMMAPS[
        source_name
    ]

    assert train_map.dtype == np.uint16
    assert val_map.dtype == np.uint16

    assert len(train_map) > 0, (
        f"Empty training shard: {source_name}"
    )

    assert len(val_map) > 0, (
        f"Empty validation shard: {source_name}"
    )

    train_probe = np.asarray(
        train_map[
            :min(32, len(train_map))
        ]
    )

    val_probe = np.asarray(
        val_map[
            :min(32, len(val_map))
        ]
    )

    assert np.all(
        train_probe < TOKENIZER_VOCAB_SIZE
    ), (
        f"Invalid token ID in training shard: "
        f"{source_name}"
    )

    assert np.all(
        val_probe < TOKENIZER_VOCAB_SIZE
    ), (
        f"Invalid token ID in validation shard: "
        f"{source_name}"
    )


# =========================================================================
# SOURCE MIX VALIDATION
# =========================================================================

assert set(
    SOURCE_PROBABILITIES.keys()
) == set(
    SOURCE_NAMES
)

probability_sum = sum(
    SOURCE_PROBABILITIES.values()
)

assert abs(
    probability_sum - 1.0
) < 1e-8, (
    f"Source probabilities must sum to 1.0. "
    f"Got {probability_sum}."
)


# =========================================================================
# SHARD TOKEN COUNTS
# =========================================================================

TRAIN_SHARD_TOKEN_COUNTS = {

    source_name:
        int(
            len(
                TRAIN_MEMMAPS[source_name]
            )
        )

    for source_name in SOURCE_NAMES
}


VAL_SHARD_TOKEN_COUNTS = {

    source_name:
        int(
            len(
                VAL_MEMMAPS[source_name]
            )
        )

    for source_name in SOURCE_NAMES
}


# =========================================================================
# SAVE DATA MANIFEST
# =========================================================================

DATA_MANIFEST = {

    "experiment_id":
        EXPERIMENT_ID,

    "architecture_name":
        ARCHITECTURE_NAME,

    "architecture_version":
        ARCHITECTURE_VERSION,

    "run_mode":
        RUN_MODE,

    "dataset_root":
        str(TOKENIZED_DATASET_ROOT),

    "tokenizer_path":
        str(TOKENIZER_PATH),

    "tokenizer_vocab_size":
        TOKENIZER_VOCAB_SIZE,

    "token_dtype":
        "uint16",

    "train_shards": {
        source_name: str(path)
        for source_name, path
        in TRAIN_SHARD_PATHS.items()
    },

    "validation_shards": {
        source_name: str(path)
        for source_name, path
        in VAL_SHARD_PATHS.items()
    },

    "train_shard_token_counts":
        TRAIN_SHARD_TOKEN_COUNTS,

    "validation_shard_token_counts":
        VAL_SHARD_TOKEN_COUNTS,

    "source_probabilities":
        SOURCE_PROBABILITIES,

    "tokenizer_metadata_available":
        bool(tokenizer_metadata),

    "dataset_metadata_available":
        bool(dataset_metadata),

    "tokenization_manifest_available":
        bool(tokenization_manifest),
}


DATA_MANIFEST_PATH = (
    CONFIGS_DIR
    / "architecture_B_data_manifest.json"
)


with open(
    DATA_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        DATA_MANIFEST,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# SUMMARY
# =========================================================================

print()
print("✅ Architecture-B corpus resolved successfully.")
print()

print(
    f"Dataset root    : "
    f"{TOKENIZED_DATASET_ROOT}"
)

print(
    f"Tokenizer       : "
    f"{TOKENIZER_PATH}"
)

print(
    f"Tokenized dir   : "
    f"{TOKENIZED_DIR}"
)

print(
    f"Vocabulary size : "
    f"{TOKENIZER_VOCAB_SIZE:,}"
)

print(
    "Token dtype     : uint16"
)

print()
print("Training shards:")

for source_name in SOURCE_NAMES:

    print(
        f"  {source_name:18s}: "
        f"{TRAIN_SHARD_TOKEN_COUNTS[source_name]:,} tokens"
    )

print()
print("Validation shards:")

for source_name in SOURCE_NAMES:

    print(
        f"  {source_name:18s}: "
        f"{VAL_SHARD_TOKEN_COUNTS[source_name]:,} tokens"
    )

print()
print(
    f"Data manifest   : "
    f"{DATA_MANIFEST_PATH}"
)

📚 Resolving Architecture-B shared tokenizer and tokenized corpus...
ℹ️ Direct dataset layout not found; searching recursively under /kaggle/input ...

✅ Architecture-B corpus resolved successfully.

Dataset root    : /kaggle/input/datasets/rohitbatra2004/cs781-phase1-tokenized-corpus
Tokenizer       : /kaggle/input/datasets/rohitbatra2004/cs781-phase1-tokenized-corpus/tokenizer/tokenizer.json
Tokenized dir   : /kaggle/input/datasets/rohitbatra2004/cs781-phase1-tokenized-corpus/tokenized
Vocabulary size : 50,000
Token dtype     : uint16

Training shards:
  acts              : 149,598,391 tokens
  high_courts       : 2,327,101,667 tokens
  supreme_court     : 251,833,809 tokens
  tribunals         : 1,607,073,435 tokens

Validation shards:
  acts              : 3,405,295 tokens
  high_courts       : 42,068,902 tokens
  supreme_court     : 5,313,054 tokens
  tribunals         : 32,846,795 tokens

Data manifest   : /kaggle/working/architecture_B/configs/architecture_B_data_manifest.json


In [4]:
# Cell 04 — Architecture-B sequence-to-sequence sampler and batching
# DECISION: ADD

import json
from pathlib import Path

import numpy as np
import torch


print("🧩 Building Architecture-B sequence-to-sequence sampler...")


# =========================================================================
# BASIC CONFIGURATION
# =========================================================================

TRAIN_CONTEXT_LENGTH = (
    MODEL_CONFIG["src_context_length"]
)

TARGET_CONTEXT_LENGTH = (
    MODEL_CONFIG["tgt_context_length"]
)

FAST_BATCH_SIZE = (
    TRAIN_CONFIG["micro_batch_size"]
)

GRAD_ACCUM_STEPS = (
    TRAIN_CONFIG["gradient_accumulation_steps"]
)


# =========================================================================
# SPECIAL TOKEN DISCOVERY
# =========================================================================

def _token_id_from_candidates(
    tokenizer,
    candidates,
):
    """
    Return the first existing token ID from a list of token names.
    """

    for token_name in candidates:

        try:

            token_id = tokenizer.token_to_id(
                token_name
            )

        except Exception:

            token_id = None

        if token_id is not None:

            return int(token_id)

    return None


BOS_TOKEN_ID = _token_id_from_candidates(
    TOKENIZER,
    [
        "<bos>",
        "<BOS>",
        "[BOS]",
        "<s>",
        "[CLS]",
    ],
)


EOS_TOKEN_ID = _token_id_from_candidates(
    TOKENIZER,
    [
        "<eos>",
        "<EOS>",
        "[EOS]",
        "</s>",
        "[SEP]",
    ],
)


PAD_TOKEN_ID = _token_id_from_candidates(
    TOKENIZER,
    [
        "<pad>",
        "<PAD>",
        "[PAD]",
    ],
)


UNK_TOKEN_ID = _token_id_from_candidates(
    TOKENIZER,
    [
        "<unk>",
        "<UNK>",
        "[UNK]",
    ],
)


# Some tokenizers do not define BOS.
# EOS is a valid deterministic decoder-start fallback.
if BOS_TOKEN_ID is not None:

    DECODER_START_TOKEN_ID = BOS_TOKEN_ID

elif EOS_TOKEN_ID is not None:

    DECODER_START_TOKEN_ID = EOS_TOKEN_ID

else:

    # Last-resort fallback only when the tokenizer has no explicit
    # start/end token at all.
    DECODER_START_TOKEN_ID = 0


assert (
    0 <= DECODER_START_TOKEN_ID
    < TOKENIZER_VOCAB_SIZE
), (
    "Invalid decoder start token ID: "
    f"{DECODER_START_TOKEN_ID}"
)


# =========================================================================
# VALIDATE SOURCE SHARDS
# =========================================================================

for source_name in SOURCE_NAMES:

    assert source_name in TRAIN_MEMMAPS
    assert source_name in VAL_MEMMAPS

    assert len(
        TRAIN_MEMMAPS[source_name]
    ) > (
        TRAIN_CONTEXT_LENGTH
        + TARGET_CONTEXT_LENGTH
    ), (
        f"Training shard '{source_name}' is too small "
        "for the configured sequence lengths."
    )

    assert len(
        VAL_MEMMAPS[source_name]
    ) > (
        TRAIN_CONTEXT_LENGTH
        + TARGET_CONTEXT_LENGTH
    ), (
        f"Validation shard '{source_name}' is too small "
        "for the configured sequence lengths."
    )


# =========================================================================
# CHECK SOURCE PROBABILITIES
# =========================================================================

SOURCE_ORDER = list(
    SOURCE_NAMES
)

SOURCE_PROBABILITY_ARRAY = np.asarray(
    [
        SOURCE_PROBABILITIES[name]
        for name in SOURCE_ORDER
    ],
    dtype=np.float64,
)

assert (
    SOURCE_PROBABILITY_ARRAY.sum()
    > 0
)

SOURCE_PROBABILITY_ARRAY /= (
    SOURCE_PROBABILITY_ARRAY.sum()
)


# =========================================================================
# CHECKPOINTABLE SEQUENCE-TO-SEQUENCE SAMPLER
# =========================================================================

class EncoderDecoderTokenSampler:
    """
    Samples contiguous legal-token spans directly from memmaps.

    For every example:

        source:
            tokens[start : start + src_len]

        target:
            tokens[
                start + src_len :
                start + src_len + tgt_len
            ]

    Decoder input:

        [decoder_start] + target[:-1]

    Labels:

        target

    The sampler owns a NumPy RNG so its state can be checkpointed.
    """

    def __init__(
        self,
        train=True,
        seed=42,
    ):

        self.train = bool(train)

        offset = (
            0
            if self.train
            else 10_000_019
        )

        self.rng = np.random.default_rng(
            int(seed) + offset
        )

        self.total_samples = 0


    # ---------------------------------------------------------------------
    # RNG state
    # ---------------------------------------------------------------------

    def get_state(self):

        return {
            "bit_generator": self.rng.bit_generator.state,
            "total_samples": int(
                self.total_samples
            ),
        }


    def set_state(
        self,
        state,
    ):

        self.rng.bit_generator.state = (
            state["bit_generator"]
        )

        self.total_samples = int(
            state.get(
                "total_samples",
                0,
            )
        )


    # ---------------------------------------------------------------------
    # Select source
    # ---------------------------------------------------------------------

    def _select_source(self):

        source_index = int(
            self.rng.choice(
                len(SOURCE_ORDER),
                p=SOURCE_PROBABILITY_ARRAY,
            )
        )

        return SOURCE_ORDER[
            source_index
        ]


    # ---------------------------------------------------------------------
    # Select random start
    # ---------------------------------------------------------------------

    def _sample_start(
        self,
        token_array,
    ):

        required_length = (
            TRAIN_CONTEXT_LENGTH
            + TARGET_CONTEXT_LENGTH
        )

        max_start = (
            len(token_array)
            - required_length
        )

        assert max_start > 0

        return int(
            self.rng.integers(
                0,
                max_start + 1,
            )
        )


    # ---------------------------------------------------------------------
    # One sequence-to-sequence example
    # ---------------------------------------------------------------------

    def sample_example(self):

        source_name = (
            self._select_source()
        )

        if self.train:

            token_array = (
                TRAIN_MEMMAPS[
                    source_name
                ]
            )

        else:

            token_array = (
                VAL_MEMMAPS[
                    source_name
                ]
            )

        start = self._sample_start(
            token_array
        )

        source_end = (
            start
            + TRAIN_CONTEXT_LENGTH
        )

        target_end = (
            source_end
            + TARGET_CONTEXT_LENGTH
        )

        source_np = np.asarray(
            token_array[
                start:source_end
            ],
            dtype=np.int64,
        )

        target_np = np.asarray(
            token_array[
                source_end:target_end
            ],
            dtype=np.int64,
        )

        # -------------------------------------------------------------
        # Decoder input:
        #   [START] + target[:-1]
        #
        # This aligns one decoder input token with each target label.
        # -------------------------------------------------------------
        decoder_input_np = np.empty(
            TARGET_CONTEXT_LENGTH,
            dtype=np.int64,
        )

        decoder_input_np[0] = (
            DECODER_START_TOKEN_ID
        )

        decoder_input_np[1:] = (
            target_np[:-1]
        )

        self.total_samples += 1

        return (
            source_np,
            decoder_input_np,
            target_np,
            source_name,
        )


    # ---------------------------------------------------------------------
    # Batch
    # ---------------------------------------------------------------------

    def sample_batch(
        self,
        batch_size,
        device,
    ):

        sources = []
        decoder_inputs = []
        targets = []
        source_names = []

        for _ in range(
            int(batch_size)
        ):

            (
                source_np,
                decoder_input_np,
                target_np,
                source_name,
            ) = self.sample_example()

            sources.append(
                source_np
            )

            decoder_inputs.append(
                decoder_input_np
            )

            targets.append(
                target_np
            )

            source_names.append(
                source_name
            )

        # -------------------------------------------------------------
        # Stack as contiguous CPU arrays before creating tensors.
        # This avoids unnecessary tensor copies from memmap views.
        # -------------------------------------------------------------
        source_array = np.ascontiguousarray(
            np.stack(
                sources,
                axis=0,
            )
        )

        decoder_input_array = (
            np.ascontiguousarray(
                np.stack(
                    decoder_inputs,
                    axis=0,
                )
            )
        )

        target_array = np.ascontiguousarray(
            np.stack(
                targets,
                axis=0,
            )
        )

        source_tensor = torch.from_numpy(
            source_array
        ).to(
            device=device,
            dtype=torch.long,
            non_blocking=False,
        )

        decoder_input_tensor = (
            torch.from_numpy(
                decoder_input_array
            ).to(
                device=device,
                dtype=torch.long,
                non_blocking=False,
            )
        )

        target_tensor = torch.from_numpy(
            target_array
        ).to(
            device=device,
            dtype=torch.long,
            non_blocking=False,
        )

        return {
            "encoder_input_ids":
                source_tensor,

            "decoder_input_ids":
                decoder_input_tensor,

            "labels":
                target_tensor,

            "source_names":
                source_names,
        }


# =========================================================================
# CREATE TRAINING + VALIDATION SAMPLERS
# =========================================================================

TRAIN_SAMPLER = (
    EncoderDecoderTokenSampler(
        train=True,
        seed=SEED,
    )
)

VAL_SAMPLER = (
    EncoderDecoderTokenSampler(
        train=False,
        seed=SEED,
    )
)


# =========================================================================
# VALIDATION BATCH CONFIGURATION
# =========================================================================
# Keep validation deterministic and bounded.
#
# FAST:
#   4 batches
#
# PRODUCTION:
#   100 batches
#
# This is much cheaper than scanning the complete validation corpus.

if RUN_MODE == "fast":

    VALIDATION_BATCH_SIZE = (
        max(
            1,
            FAST_BATCH_SIZE,
        )
    )

    VALIDATION_NUM_BATCHES = 4

else:

    VALIDATION_BATCH_SIZE = (
        TRAIN_CONFIG["micro_batch_size"]
    )

    VALIDATION_NUM_BATCHES = 100


# =========================================================================
# TOKENS PER OPTIMIZER STEP
# =========================================================================

TOKENS_PER_MICRO_BATCH = (
    TRAIN_CONTEXT_LENGTH
    + TARGET_CONTEXT_LENGTH
)

TOKENS_PER_OPTIMIZER_STEP = (
    TOKENS_PER_MICRO_BATCH
    * TRAIN_CONFIG[
        "micro_batch_size"
    ]
    * TRAIN_CONFIG[
        "gradient_accumulation_steps"
    ]
)


# Recompute production max steps using the exact number of
# source + target tokens consumed by one optimizer step.
if RUN_MODE == "production":

    TRAIN_CONFIG["max_steps"] = int(
        np.ceil(
            TRAIN_CONFIG[
                "train_token_budget"
            ]
            / TOKENS_PER_OPTIMIZER_STEP
        )
    )


# =========================================================================
# SAVE SAMPLER MANIFEST
# =========================================================================

SAMPLER_MANIFEST = {

    "experiment_id":
        EXPERIMENT_ID,

    "architecture_name":
        ARCHITECTURE_NAME,

    "architecture_version":
        ARCHITECTURE_VERSION,

    "run_mode":
        RUN_MODE,

    "source_context_length":
        TRAIN_CONTEXT_LENGTH,

    "target_context_length":
        TARGET_CONTEXT_LENGTH,

    "micro_batch_size":
        TRAIN_CONFIG[
            "micro_batch_size"
        ],

    "gradient_accumulation_steps":
        TRAIN_CONFIG[
            "gradient_accumulation_steps"
        ],

    "tokens_per_micro_batch":
        TOKENS_PER_MICRO_BATCH,

    "tokens_per_optimizer_step":
        TOKENS_PER_OPTIMIZER_STEP,

    "max_steps":
        TRAIN_CONFIG["max_steps"],

    "validation_batch_size":
        VALIDATION_BATCH_SIZE,

    "validation_num_batches":
        VALIDATION_NUM_BATCHES,

    "source_probabilities":
        SOURCE_PROBABILITIES,

    "decoder_start_token_id":
        DECODER_START_TOKEN_ID,

    "bos_token_id":
        BOS_TOKEN_ID,

    "eos_token_id":
        EOS_TOKEN_ID,

    "pad_token_id":
        PAD_TOKEN_ID,

    "unk_token_id":
        UNK_TOKEN_ID,
}


SAMPLER_MANIFEST_PATH = (
    CONFIGS_DIR
    / "architecture_B_sampler.json"
)

with open(
    SAMPLER_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        SAMPLER_MANIFEST,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# SANITY-TEST ONE BATCH
# =========================================================================

probe_batch = (
    TRAIN_SAMPLER.sample_batch(
        batch_size=(
            TRAIN_CONFIG[
                "micro_batch_size"
            ]
        ),
        device=DEVICE,
    )
)


assert probe_batch[
    "encoder_input_ids"
].shape == (
    FAST_BATCH_SIZE,
    TRAIN_CONTEXT_LENGTH,
)

assert probe_batch[
    "decoder_input_ids"
].shape == (
    FAST_BATCH_SIZE,
    TARGET_CONTEXT_LENGTH,
)

assert probe_batch[
    "labels"
].shape == (
    FAST_BATCH_SIZE,
    TARGET_CONTEXT_LENGTH,
)


assert (
    probe_batch[
        "encoder_input_ids"
    ].dtype
    == torch.long
)

assert (
    probe_batch[
        "decoder_input_ids"
    ].dtype
    == torch.long
)

assert (
    probe_batch[
        "labels"
    ].dtype
    == torch.long
)


# Verify token ID ranges.
assert torch.all(
    probe_batch[
        "encoder_input_ids"
    ] < TOKENIZER_VOCAB_SIZE
)

assert torch.all(
    probe_batch[
        "decoder_input_ids"
    ] < TOKENIZER_VOCAB_SIZE
)

assert torch.all(
    probe_batch[
        "labels"
    ] < TOKENIZER_VOCAB_SIZE
)


# =========================================================================
# SUMMARY
# =========================================================================

print()
print("✅ Architecture-B sampler ready.")
print()

print(
    f"Source context          : "
    f"{TRAIN_CONTEXT_LENGTH}"
)

print(
    f"Target context          : "
    f"{TARGET_CONTEXT_LENGTH}"
)

print(
    f"Micro batch size        : "
    f"{TRAIN_CONFIG['micro_batch_size']}"
)

print(
    f"Gradient accumulation   : "
    f"{TRAIN_CONFIG['gradient_accumulation_steps']}"
)

print(
    f"Tokens / micro-batch    : "
    f"{TOKENS_PER_MICRO_BATCH:,}"
)

print(
    f"Tokens / optimizer step : "
    f"{TOKENS_PER_OPTIMIZER_STEP:,}"
)

print(
    f"Maximum steps           : "
    f"{TRAIN_CONFIG['max_steps']:,}"
)

print(
    f"Validation batch size   : "
    f"{VALIDATION_BATCH_SIZE}"
)

print(
    f"Validation batches      : "
    f"{VALIDATION_NUM_BATCHES}"
)

print(
    f"Decoder start token     : "
    f"{DECODER_START_TOKEN_ID}"
)

print(
    f"BOS token               : "
    f"{BOS_TOKEN_ID}"
)

print(
    f"EOS token               : "
    f"{EOS_TOKEN_ID}"
)

print(
    f"Train source sampled    : "
    f"{probe_batch['source_names']}"
)

print(
    f"Sampler manifest        : "
    f"{SAMPLER_MANIFEST_PATH}"
)

🧩 Building Architecture-B sequence-to-sequence sampler...

✅ Architecture-B sampler ready.

Source context          : 1024
Target context          : 1024
Micro batch size        : 1
Gradient accumulation   : 8
Tokens / micro-batch    : 2,048
Tokens / optimizer step : 16,384
Maximum steps           : 6,104
Validation batch size   : 1
Validation batches      : 100
Decoder start token     : 2
BOS token               : 2
EOS token               : 3
Train source sampled    : ['tribunals']
Sampler manifest        : /kaggle/working/architecture_B/configs/architecture_B_sampler.json


In [5]:
# Cell 05 — Architecture-B encoder-decoder Transformer
# DECISION: UPDATE

import math
import json
from contextlib import nullcontext

import torch
import torch.nn as nn
import torch.nn.functional as F


print("🧠 Building Architecture-B encoder-decoder Transformer...")


# =========================================================================
# LOCAL AUTocast HELPER
# =========================================================================
# Architecture-B must be self-contained and must not depend on a helper
# created in Architecture-A.

def autocast_context():
    """
    Return the appropriate mixed-precision context for the current
    Architecture-B runtime.

    CUDA:
        BF16 or FP16 autocast as configured in Cell 02.

    MPS / CPU:
        no-op context.
    """

    if (
        USE_AMP
        and DEVICE.type == "cuda"
    ):

        return torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        )

    return nullcontext()


# =========================================================================
# HELPER: ACTIVATION
# =========================================================================

def gelu(x):

    return F.gelu(
        x,
        approximate="tanh",
    )


# =========================================================================
# MULTI-HEAD SELF-ATTENTION
# =========================================================================

class MultiHeadSelfAttention(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        dropout=0.0,
        bias=True,
        causal=False,
    ):

        super().__init__()

        assert (
            d_model % n_heads == 0
        ), (
            f"d_model={d_model} must be divisible "
            f"by n_heads={n_heads}"
        )

        self.d_model = int(
            d_model
        )

        self.n_heads = int(
            n_heads
        )

        self.head_dim = (
            self.d_model
            // self.n_heads
        )

        self.dropout = float(
            dropout
        )

        self.causal = bool(
            causal
        )

        self.qkv = nn.Linear(
            self.d_model,
            3 * self.d_model,
            bias=bias,
        )

        self.out_proj = nn.Linear(
            self.d_model,
            self.d_model,
            bias=bias,
        )


    def forward(self, x):

        batch_size, seq_len, _ = (
            x.shape
        )

        qkv = self.qkv(x)

        q, k, v = torch.chunk(
            qkv,
            3,
            dim=-1,
        )

        q = q.view(
            batch_size,
            seq_len,
            self.n_heads,
            self.head_dim,
        ).transpose(
            1,
            2,
        )

        k = k.view(
            batch_size,
            seq_len,
            self.n_heads,
            self.head_dim,
        ).transpose(
            1,
            2,
        )

        v = v.view(
            batch_size,
            seq_len,
            self.n_heads,
            self.head_dim,
        ).transpose(
            1,
            2,
        )

        dropout_p = (
            self.dropout
            if self.training
            else 0.0
        )

        attended = (
            F.scaled_dot_product_attention(
                q,
                k,
                v,
                attn_mask=None,
                dropout_p=dropout_p,
                is_causal=self.causal,
            )
        )

        attended = (
            attended
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                seq_len,
                self.d_model,
            )
        )

        return self.out_proj(
            attended
        )


# =========================================================================
# MULTI-HEAD CROSS-ATTENTION
# =========================================================================

class MultiHeadCrossAttention(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        dropout=0.0,
        bias=True,
    ):

        super().__init__()

        assert (
            d_model % n_heads == 0
        ), (
            f"d_model={d_model} must be divisible "
            f"by n_heads={n_heads}"
        )

        self.d_model = int(
            d_model
        )

        self.n_heads = int(
            n_heads
        )

        self.head_dim = (
            self.d_model
            // self.n_heads
        )

        self.dropout = float(
            dropout
        )

        self.q_proj = nn.Linear(
            self.d_model,
            self.d_model,
            bias=bias,
        )

        self.k_proj = nn.Linear(
            self.d_model,
            self.d_model,
            bias=bias,
        )

        self.v_proj = nn.Linear(
            self.d_model,
            self.d_model,
            bias=bias,
        )

        self.out_proj = nn.Linear(
            self.d_model,
            self.d_model,
            bias=bias,
        )


    def forward(
        self,
        x,
        memory,
    ):

        batch_size, target_len, _ = (
            x.shape
        )

        source_len = memory.shape[1]

        q = self.q_proj(x)
        k = self.k_proj(memory)
        v = self.v_proj(memory)

        q = q.view(
            batch_size,
            target_len,
            self.n_heads,
            self.head_dim,
        ).transpose(
            1,
            2,
        )

        k = k.view(
            batch_size,
            source_len,
            self.n_heads,
            self.head_dim,
        ).transpose(
            1,
            2,
        )

        v = v.view(
            batch_size,
            source_len,
            self.n_heads,
            self.head_dim,
        ).transpose(
            1,
            2,
        )

        dropout_p = (
            self.dropout
            if self.training
            else 0.0
        )

        attended = (
            F.scaled_dot_product_attention(
                q,
                k,
                v,
                attn_mask=None,
                dropout_p=dropout_p,
                is_causal=False,
            )
        )

        attended = (
            attended
            .transpose(1, 2)
            .contiguous()
            .view(
                batch_size,
                target_len,
                self.d_model,
            )
        )

        return self.out_proj(
            attended
        )


# =========================================================================
# FEED-FORWARD NETWORK
# =========================================================================

class FeedForward(nn.Module):

    def __init__(
        self,
        d_model,
        d_ff,
        dropout=0.0,
        bias=True,
    ):

        super().__init__()

        self.fc1 = nn.Linear(
            d_model,
            d_ff,
            bias=bias,
        )

        self.fc2 = nn.Linear(
            d_ff,
            d_model,
            bias=bias,
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(self, x):

        x = self.fc1(x)
        x = gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x


# =========================================================================
# ENCODER BLOCK
# =========================================================================

class EncoderBlock(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout=0.0,
        pre_norm=True,
        bias=True,
    ):

        super().__init__()

        self.pre_norm = bool(
            pre_norm
        )

        self.norm1 = nn.LayerNorm(
            d_model,
            bias=bias,
        )

        self.norm2 = nn.LayerNorm(
            d_model,
            bias=bias,
        )

        self.self_attn = (
            MultiHeadSelfAttention(
                d_model=d_model,
                n_heads=n_heads,
                dropout=dropout,
                bias=bias,
                causal=False,
            )
        )

        self.ffn = FeedForward(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout,
            bias=bias,
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(self, x):

        if self.pre_norm:

            x = x + self.dropout(
                self.self_attn(
                    self.norm1(x)
                )
            )

            x = x + self.dropout(
                self.ffn(
                    self.norm2(x)
                )
            )

        else:

            x = self.norm1(
                x
                + self.dropout(
                    self.self_attn(x)
                )
            )

            x = self.norm2(
                x
                + self.dropout(
                    self.ffn(x)
                )
            )

        return x


# =========================================================================
# DECODER BLOCK
# =========================================================================

class DecoderBlock(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout=0.0,
        pre_norm=True,
        bias=True,
    ):

        super().__init__()

        self.pre_norm = bool(
            pre_norm
        )

        self.norm1 = nn.LayerNorm(
            d_model,
            bias=bias,
        )

        self.norm2 = nn.LayerNorm(
            d_model,
            bias=bias,
        )

        self.norm3 = nn.LayerNorm(
            d_model,
            bias=bias,
        )

        self.self_attn = (
            MultiHeadSelfAttention(
                d_model=d_model,
                n_heads=n_heads,
                dropout=dropout,
                bias=bias,
                causal=True,
            )
        )

        self.cross_attn = (
            MultiHeadCrossAttention(
                d_model=d_model,
                n_heads=n_heads,
                dropout=dropout,
                bias=bias,
            )
        )

        self.ffn = FeedForward(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout,
            bias=bias,
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        memory,
    ):

        if self.pre_norm:

            x = x + self.dropout(
                self.self_attn(
                    self.norm1(x)
                )
            )

            x = x + self.dropout(
                self.cross_attn(
                    self.norm2(x),
                    memory,
                )
            )

            x = x + self.dropout(
                self.ffn(
                    self.norm3(x)
                )
            )

        else:

            x = self.norm1(
                x
                + self.dropout(
                    self.self_attn(x)
                )
            )

            x = self.norm2(
                x
                + self.dropout(
                    self.cross_attn(
                        x,
                        memory,
                    )
                )
            )

            x = self.norm3(
                x
                + self.dropout(
                    self.ffn(x)
                )
            )

        return x


# =========================================================================
# ARCHITECTURE-B MODEL
# =========================================================================

class EncoderDecoderTransformer(nn.Module):

    def __init__(
        self,
        config,
    ):

        super().__init__()

        self.config = dict(
            config
        )

        vocab_size = config[
            "vocab_size"
        ]

        d_model = config[
            "d_model"
        ]

        n_heads = config[
            "n_heads"
        ]

        d_ff = config[
            "d_ff"
        ]

        dropout = config[
            "dropout"
        ]

        self.d_model = d_model

        self.src_context_length = (
            config[
                "src_context_length"
            ]
        )

        self.tgt_context_length = (
            config[
                "tgt_context_length"
            ]
        )

        self.tie_embeddings = bool(
            config[
                "tie_embeddings"
            ]
        )

        # -------------------------------------------------------------
        # Shared token embedding.
        # -------------------------------------------------------------
        self.token_embedding = (
            nn.Embedding(
                vocab_size,
                d_model,
            )
        )

        # -------------------------------------------------------------
        # Separate learned positional embeddings.
        # -------------------------------------------------------------
        self.encoder_position_embedding = (
            nn.Embedding(
                self.src_context_length,
                d_model,
            )
        )

        self.decoder_position_embedding = (
            nn.Embedding(
                self.tgt_context_length,
                d_model,
            )
        )

        self.embedding_dropout = (
            nn.Dropout(dropout)
        )

        # -------------------------------------------------------------
        # Encoder.
        # -------------------------------------------------------------
        self.encoder_layers = (
            nn.ModuleList(
                [
                    EncoderBlock(
                        d_model=d_model,
                        n_heads=n_heads,
                        d_ff=d_ff,
                        dropout=dropout,
                        pre_norm=config[
                            "pre_norm"
                        ],
                        bias=config[
                            "bias"
                        ],
                    )
                    for _ in range(
                        config[
                            "n_encoder_layers"
                        ]
                    )
                ]
            )
        )

        self.encoder_final_norm = (
            nn.LayerNorm(
                d_model,
                bias=config[
                    "bias"
                ],
            )
        )

        # -------------------------------------------------------------
        # Decoder.
        # -------------------------------------------------------------
        self.decoder_layers = (
            nn.ModuleList(
                [
                    DecoderBlock(
                        d_model=d_model,
                        n_heads=n_heads,
                        d_ff=d_ff,
                        dropout=dropout,
                        pre_norm=config[
                            "pre_norm"
                        ],
                        bias=config[
                            "bias"
                        ],
                    )
                    for _ in range(
                        config[
                            "n_decoder_layers"
                        ]
                    )
                ]
            )
        )

        self.decoder_final_norm = (
            nn.LayerNorm(
                d_model,
                bias=config[
                    "bias"
                ],
            )
        )

        # -------------------------------------------------------------
        # Output bias.
        # -------------------------------------------------------------
        self.output_bias = (
            nn.Parameter(
                torch.zeros(
                    vocab_size
                )
            )
        )

        # -------------------------------------------------------------
        # From-scratch initialization.
        # -------------------------------------------------------------
        self.apply(
            self._init_weights
        )


    @staticmethod
    def _init_weights(module):

        if isinstance(
            module,
            nn.Linear,
        ):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02,
            )

            if module.bias is not None:

                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(
            module,
            nn.Embedding,
        ):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02,
            )

        elif isinstance(
            module,
            nn.LayerNorm,
        ):

            nn.init.ones_(
                module.weight
            )

            if module.bias is not None:

                nn.init.zeros_(
                    module.bias
                )


    # ---------------------------------------------------------------------
    # Encoder
    # ---------------------------------------------------------------------

    def encode(
        self,
        encoder_input_ids,
    ):

        batch_size, source_len = (
            encoder_input_ids.shape
        )

        assert source_len <= (
            self.src_context_length
        ), (
            f"Source sequence length "
            f"{source_len} exceeds "
            f"maximum "
            f"{self.src_context_length}."
        )

        positions = torch.arange(
            source_len,
            device=encoder_input_ids.device,
        )

        x = (
            self.token_embedding(
                encoder_input_ids
            )
            * math.sqrt(
                self.d_model
            )
        )

        x = x + (
            self.encoder_position_embedding(
                positions
            ).unsqueeze(0)
        )

        x = self.embedding_dropout(
            x
        )

        for layer in (
            self.encoder_layers
        ):

            x = layer(x)

        return self.encoder_final_norm(
            x
        )


    # ---------------------------------------------------------------------
    # Decoder
    # ---------------------------------------------------------------------

    def decode(
        self,
        decoder_input_ids,
        memory,
    ):

        batch_size, target_len = (
            decoder_input_ids.shape
        )

        assert target_len <= (
            self.tgt_context_length
        ), (
            f"Target sequence length "
            f"{target_len} exceeds "
            f"maximum "
            f"{self.tgt_context_length}."
        )

        positions = torch.arange(
            target_len,
            device=decoder_input_ids.device,
        )

        x = (
            self.token_embedding(
                decoder_input_ids
            )
            * math.sqrt(
                self.d_model
            )
        )

        x = x + (
            self.decoder_position_embedding(
                positions
            ).unsqueeze(0)
        )

        x = self.embedding_dropout(
            x
        )

        for layer in (
            self.decoder_layers
        ):

            x = layer(
                x,
                memory,
            )

        return self.decoder_final_norm(
            x
        )


    # ---------------------------------------------------------------------
    # Forward
    # ---------------------------------------------------------------------

    def forward(
        self,
        encoder_input_ids,
        decoder_input_ids,
        labels=None,
    ):

        memory = self.encode(
            encoder_input_ids
        )

        decoder_hidden = self.decode(
            decoder_input_ids,
            memory,
        )

        logits = F.linear(
            decoder_hidden,
            self.token_embedding.weight,
            self.output_bias,
        )

        if labels is None:

            return logits

        loss = F.cross_entropy(
            logits.reshape(
                -1,
                logits.shape[-1],
            ),
            labels.reshape(-1),
        )

        return {
            "logits": logits,
            "loss": loss,
        }


# =========================================================================
# CREATE MODEL
# =========================================================================

MODEL = EncoderDecoderTransformer(
    MODEL_CONFIG
).to(
    DEVICE
)


# =========================================================================
# PARAMETER COUNTS
# =========================================================================

TOTAL_PARAMETERS = sum(
    parameter.numel()
    for parameter in MODEL.parameters()
)

TRAINABLE_PARAMETERS = sum(
    parameter.numel()
    for parameter in MODEL.parameters()
    if parameter.requires_grad
)


# =========================================================================
# PRODUCTION PARAMETER REQUIREMENT
# =========================================================================

if RUN_MODE == "production":

    assert (
        50_000_000
        <= TOTAL_PARAMETERS
        <= 300_000_000
    ), (
        "Architecture-B production model must "
        "contain 50M–300M parameters.\n"
        f"Found: {TOTAL_PARAMETERS:,}"
    )


# =========================================================================
# PARAMETER MEMORY ESTIMATE
# =========================================================================

parameter_memory_mb = (
    TOTAL_PARAMETERS
    * 4
    / (1024 ** 2)
)

fp16_parameter_memory_mb = (
    TOTAL_PARAMETERS
    * 2
    / (1024 ** 2)
)


# =========================================================================
# FORWARD-PASS SANITY TEST
# =========================================================================

MODEL.eval()

probe_batch = TRAIN_SAMPLER.sample_batch(
    batch_size=(
        TRAIN_CONFIG[
            "micro_batch_size"
        ]
    ),
    device=DEVICE,
)


encoder_probe = probe_batch[
    "encoder_input_ids"
]

decoder_probe = probe_batch[
    "decoder_input_ids"
]

label_probe = probe_batch[
    "labels"
]


with torch.no_grad():

    with autocast_context():

        probe_output = MODEL(
            encoder_input_ids=encoder_probe,
            decoder_input_ids=decoder_probe,
            labels=label_probe,
        )


probe_loss = probe_output[
    "loss"
]

probe_logits = probe_output[
    "logits"
]


assert torch.isfinite(
    probe_loss
), (
    "Architecture-B probe loss is not finite."
)


expected_probe_shape = (
    TRAIN_CONFIG[
        "micro_batch_size"
    ],
    TARGET_CONTEXT_LENGTH,
    TOKENIZER_VOCAB_SIZE,
)


assert tuple(
    probe_logits.shape
) == tuple(
    expected_probe_shape
), (
    "Unexpected Architecture-B logits shape.\n"
    f"Expected: {expected_probe_shape}\n"
    f"Found   : {tuple(probe_logits.shape)}"
)


# =========================================================================
# SAVE MODEL MANIFEST
# =========================================================================

MODEL_MANIFEST = {

    "experiment_id":
        EXPERIMENT_ID,

    "architecture_name":
        ARCHITECTURE_NAME,

    "architecture_version":
        ARCHITECTURE_VERSION,

    "run_mode":
        RUN_MODE,

    "model_type":
        "encoder-decoder-transformer",

    "model_config":
        MODEL_CONFIG,

    "total_parameters":
        TOTAL_PARAMETERS,

    "trainable_parameters":
        TRAINABLE_PARAMETERS,

    "parameter_millions":
        TOTAL_PARAMETERS
        / 1_000_000,

    "parameter_memory_mb_fp32":
        parameter_memory_mb,

    "parameter_memory_mb_fp16":
        fp16_parameter_memory_mb,

    "shared_token_embedding":
        True,

    "tied_output_projection":
        bool(
            MODEL_CONFIG[
                "tie_embeddings"
            ]
        ),

    "probe_loss":
        float(
            probe_loss.detach()
            .float()
            .cpu()
        ),

    "probe_logits_shape":
        list(
            probe_logits.shape
        ),
}


MODEL_MANIFEST_PATH = (
    CONFIGS_DIR
    / "architecture_B_model.json"
)


with open(
    MODEL_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        MODEL_MANIFEST,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# SUMMARY
# =========================================================================

print()
print("✅ Architecture-B model built successfully.")
print()

print(
    f"Model type        : "
    f"{MODEL_MANIFEST['model_type']}"
)

print(
    f"Total parameters  : "
    f"{TOTAL_PARAMETERS:,}"
)

print(
    f"Trainable         : "
    f"{TRAINABLE_PARAMETERS:,}"
)

print(
    f"Parameters        : "
    f"{TOTAL_PARAMETERS / 1_000_000:.4f}M"
)

print(
    f"FP32 param memory : "
    f"{parameter_memory_mb:.1f} MB"
)

print(
    f"FP16 param memory : "
    f"{fp16_parameter_memory_mb:.1f} MB"
)

print(
    f"Encoder layers    : "
    f"{MODEL_CONFIG['n_encoder_layers']}"
)

print(
    f"Decoder layers    : "
    f"{MODEL_CONFIG['n_decoder_layers']}"
)

print(
    f"Attention heads   : "
    f"{MODEL_CONFIG['n_heads']}"
)

print(
    f"d_model           : "
    f"{MODEL_CONFIG['d_model']}"
)

print(
    f"d_ff              : "
    f"{MODEL_CONFIG['d_ff']}"
)

print(
    f"Source context    : "
    f"{MODEL_CONFIG['src_context_length']}"
)

print(
    f"Target context    : "
    f"{MODEL_CONFIG['tgt_context_length']}"
)

print(
    f"Shared embedding  : True"
)

print(
    f"Tied LM head      : "
    f"{MODEL_CONFIG['tie_embeddings']}"
)

print(
    f"Probe loss        : "
    f"{float(probe_loss.detach().float().cpu()):.6f}"
)

print(
    f"Probe logits      : "
    f"{tuple(probe_logits.shape)}"
)

print(
    f"Model manifest    : "
    f"{MODEL_MANIFEST_PATH}"
)

🧠 Building Architecture-B encoder-decoder Transformer...

✅ Architecture-B model built successfully.

Model type        : encoder-decoder-transformer
Total parameters  : 139,263,824
Trainable         : 139,263,824
Parameters        : 139.2638M
FP32 param memory : 531.2 MB
FP16 param memory : 265.6 MB
Encoder layers    : 6
Decoder layers    : 6
Attention heads   : 12
d_model           : 768
d_ff              : 3072
Source context    : 1024
Target context    : 1024
Shared embedding  : True
Tied LM head      : True
Probe loss        : 10.856974
Probe logits      : (1, 1024, 50000)
Model manifest    : /kaggle/working/architecture_B/configs/architecture_B_model.json


In [6]:
# Cell 06 — Architecture-B optimizer, scheduler, and mixed-precision scaling
# DECISION: ADD

import math
import torch


print("⚙️ Configuring Architecture-B optimizer and learning-rate schedule...")


# =========================================================================
# LEARNING-RATE SETTINGS
# =========================================================================

INITIAL_LR = float(
    OPTIMIZER_CONFIG["learning_rate"]
)

MIN_LR = float(
    OPTIMIZER_CONFIG["min_learning_rate"]
)

WEIGHT_DECAY = float(
    OPTIMIZER_CONFIG["weight_decay"]
)

GRADIENT_CLIP_NORM = float(
    OPTIMIZER_CONFIG["gradient_clip_norm"]
)

WARMUP_FRACTION = float(
    OPTIMIZER_CONFIG["warmup_fraction"]
)

MAX_STEPS = int(
    TRAIN_CONFIG["max_steps"]
)


assert INITIAL_LR > 0.0
assert MIN_LR >= 0.0
assert MIN_LR <= INITIAL_LR
assert WEIGHT_DECAY >= 0.0
assert GRADIENT_CLIP_NORM > 0.0
assert 0.0 <= WARMUP_FRACTION < 1.0
assert MAX_STEPS > 0


# =========================================================================
# PARAMETER GROUPS
# =========================================================================
# Decay trainable matrix parameters.
# Do not decay biases or normalization parameters.

decay_parameters = []
no_decay_parameters = []

decay_parameter_names = []
no_decay_parameter_names = []


for name, parameter in MODEL.named_parameters():

    if not parameter.requires_grad:
        continue

    parameter_name = name.lower()

    # Biases and LayerNorm parameters are excluded from weight decay.
    if (
        parameter.ndim < 2
        or parameter_name.endswith("bias")
        or "norm" in parameter_name
    ):

        no_decay_parameters.append(
            parameter
        )

        no_decay_parameter_names.append(
            name
        )

    else:

        decay_parameters.append(
            parameter
        )

        decay_parameter_names.append(
            name
        )


assert (
    len(decay_parameters)
    + len(no_decay_parameters)
    > 0
)

# Ensure every trainable parameter appears exactly once.
trainable_ids = {
    id(parameter)
    for parameter in MODEL.parameters()
    if parameter.requires_grad
}

grouped_ids = {
    id(parameter)
    for parameter in (
        decay_parameters
        + no_decay_parameters
    )
}

assert trainable_ids == grouped_ids, (
    "Some trainable parameters were not assigned "
    "to an optimizer parameter group."
)


# =========================================================================
# ADAMW
# =========================================================================

OPTIMIZER = torch.optim.AdamW(
    [
        {
            "params": decay_parameters,
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": no_decay_parameters,
            "weight_decay": 0.0,
        },
    ],
    lr=INITIAL_LR,
    betas=(0.9, 0.95),
    eps=1e-8,
)


# =========================================================================
# WARMUP
# =========================================================================

if MAX_STEPS <= 1:

    WARMUP_STEPS = 1

else:

    WARMUP_STEPS = max(
        1,
        int(
            round(
                MAX_STEPS
                * WARMUP_FRACTION
            )
        ),
    )

WARMUP_STEPS = min(
    WARMUP_STEPS,
    MAX_STEPS,
)


# =========================================================================
# COSINE LR SCHEDULE
# =========================================================================

def learning_rate_lambda(step):

    # -------------------------------------------------------------
    # Linear warmup.
    # -------------------------------------------------------------
    if step < WARMUP_STEPS:

        return float(
            step + 1
        ) / float(
            WARMUP_STEPS
        )

    # -------------------------------------------------------------
    # Cosine decay.
    # -------------------------------------------------------------
    if MAX_STEPS <= WARMUP_STEPS:

        return (
            MIN_LR / INITIAL_LR
        )

    progress = (
        float(step - WARMUP_STEPS)
        / float(
            MAX_STEPS
            - WARMUP_STEPS
        )
    )

    progress = min(
        1.0,
        max(
            0.0,
            progress,
        ),
    )

    cosine_value = (
        0.5
        * (
            1.0
            + math.cos(
                math.pi
                * progress
            )
        )
    )

    min_ratio = (
        MIN_LR / INITIAL_LR
    )

    return (
        min_ratio
        + (
            1.0 - min_ratio
        )
        * cosine_value
    )


LR_SCHEDULER = (
    torch.optim.lr_scheduler.LambdaLR(
        OPTIMIZER,
        lr_lambda=learning_rate_lambda,
    )
)


# =========================================================================
# GRADIENT SCALER
# =========================================================================
# FP16 CUDA needs loss scaling.
# BF16 does not normally require a scaler.

if (
    USE_GRAD_SCALER
    and DEVICE.type == "cuda"
):

    try:

        GRAD_SCALER = torch.amp.GradScaler(
            "cuda",
            enabled=True,
        )

    except Exception:

        # Compatibility fallback for older PyTorch APIs.
        GRAD_SCALER = torch.cuda.amp.GradScaler(
            enabled=True
        )

else:

    # Disabled scaler object for BF16 / FP32.
    try:

        GRAD_SCALER = torch.amp.GradScaler(
            "cuda",
            enabled=False,
        )

    except Exception:

        GRAD_SCALER = torch.cuda.amp.GradScaler(
            enabled=False
        )


# =========================================================================
# HELPER — CURRENT LR
# =========================================================================

def get_current_learning_rate():

    return float(
        OPTIMIZER.param_groups[0]["lr"]
    )


# =========================================================================
# SANITY CHECKS
# =========================================================================

assert len(
    OPTIMIZER.param_groups
) == 2

assert (
    OPTIMIZER.param_groups[0]["weight_decay"]
    == WEIGHT_DECAY
)

assert (
    OPTIMIZER.param_groups[1]["weight_decay"]
    == 0.0
)

assert (
    len(
        decay_parameters
    )
    > 0
)

assert (
    len(
        no_decay_parameters
    )
    > 0
)


# =========================================================================
# SAVE OPTIMIZER MANIFEST
# =========================================================================

OPTIMIZER_MANIFEST = {

    "experiment_id":
        EXPERIMENT_ID,

    "architecture_name":
        ARCHITECTURE_NAME,

    "architecture_version":
        ARCHITECTURE_VERSION,

    "run_mode":
        RUN_MODE,

    "optimizer":
        "AdamW",

    "learning_rate":
        INITIAL_LR,

    "minimum_learning_rate":
        MIN_LR,

    "betas":
        [0.9, 0.95],

    "epsilon":
        1e-8,

    "weight_decay":
        WEIGHT_DECAY,

    "gradient_clip_norm":
        GRADIENT_CLIP_NORM,

    "warmup_fraction":
        WARMUP_FRACTION,

    "warmup_steps":
        WARMUP_STEPS,

    "max_steps":
        MAX_STEPS,

    "scheduler":
        "linear-warmup-cosine-decay",

    "amp_enabled":
        USE_AMP,

    "amp_mode":
        AMP_MODE,

    "grad_scaler_enabled":
        USE_GRAD_SCALER,

    "decay_tensor_count":
        len(decay_parameters),

    "no_decay_tensor_count":
        len(no_decay_parameters),

    "decay_parameter_count":
        sum(
            p.numel()
            for p in decay_parameters
        ),

    "no_decay_parameter_count":
        sum(
            p.numel()
            for p in no_decay_parameters
        ),
}


OPTIMIZER_MANIFEST_PATH = (
    CONFIGS_DIR
    / "architecture_B_optimizer.json"
)

with open(
    OPTIMIZER_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        OPTIMIZER_MANIFEST,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# SUMMARY
# =========================================================================

print()
print("✅ Architecture-B optimizer configured.")
print()

print(
    f"Optimizer       : AdamW"
)

print(
    f"Base LR         : {INITIAL_LR:.2e}"
)

print(
    f"Current LR      : "
    f"{get_current_learning_rate():.2e}"
)

print(
    f"Minimum LR      : {MIN_LR:.2e}"
)

print(
    f"Weight decay    : {WEIGHT_DECAY}"
)

print(
    f"Betas           : (0.9, 0.95)"
)

print(
    f"Grad clip norm  : "
    f"{GRADIENT_CLIP_NORM}"
)

print(
    f"Warmup steps    : "
    f"{WARMUP_STEPS}"
)

print(
    f"Maximum steps   : "
    f"{MAX_STEPS}"
)

print(
    f"AMP mode        : "
    f"{AMP_MODE}"
)

print(
    f"GradScaler      : "
    f"{USE_GRAD_SCALER}"
)

print(
    f"Decay tensors   : "
    f"{len(decay_parameters)}"
)

print(
    f"No-decay tensors: "
    f"{len(no_decay_parameters)}"
)

print(
    f"Manifest        : "
    f"{OPTIMIZER_MANIFEST_PATH}"
)

⚙️ Configuring Architecture-B optimizer and learning-rate schedule...

✅ Architecture-B optimizer configured.

Optimizer       : AdamW
Base LR         : 3.00e-04
Current LR      : 9.84e-07
Minimum LR      : 3.00e-05
Weight decay    : 0.1
Betas           : (0.9, 0.95)
Grad clip norm  : 1.0
Warmup steps    : 305
Maximum steps   : 6104
AMP mode        : fp16
GradScaler      : True
Decay tensors   : 75
No-decay tensors: 137
Manifest        : /kaggle/working/architecture_B/configs/architecture_B_optimizer.json


In [7]:
# Cell 07 — Architecture-B checkpointing and resume system
# DECISION: ADD

import json
import random
import time
from pathlib import Path

import numpy as np
import torch


print("💾 Configuring Architecture-B checkpoint and resume system...")


# =========================================================================
# CHECKPOINT PATHS
# =========================================================================

LATEST_CHECKPOINT_PATH = (
    CHECKPOINTS_DIR / "latest.pt"
)

BEST_CHECKPOINT_PATH = (
    CHECKPOINTS_DIR / "best.pt"
)

CHECKPOINT_INDEX_PATH = (
    METRICS_DIR
    / "checkpoint_index_B.json"
)

CHECKPOINT_SCHEMA_VERSION = 1


# =========================================================================
# CHECKPOINT COMPATIBILITY
# =========================================================================

def checkpoint_is_compatible(
    checkpoint,
):
    """
    A checkpoint can be resumed only when it belongs to the exact
    same experiment configuration.

    This prevents FAST <-> PRODUCTION contamination and prevents
    accidental reuse of an older Architecture-B configuration.
    """

    required_fields = {
        "experiment_id":
            EXPERIMENT_ID,

        "architecture_name":
            ARCHITECTURE_NAME,

        "architecture_version":
            ARCHITECTURE_VERSION,

        "run_mode":
            RUN_MODE,

        "config_fingerprint":
            CONFIG_FINGERPRINT,
    }

    for key, expected_value in (
        required_fields.items()
    ):

        if checkpoint.get(key) != expected_value:

            return False

    return True


# =========================================================================
# RNG STATE
# =========================================================================

def capture_rng_state():

    state = {

        "python":
            random.getstate(),

        "numpy":
            np.random.get_state(),

        "torch":
            torch.get_rng_state(),
    }

    if torch.cuda.is_available():

        state["torch_cuda"] = (
            torch.cuda.get_rng_state_all()
        )

    else:

        state["torch_cuda"] = None

    return state


def restore_rng_state(
    state,
):

    if not state:
        return

    if state.get("python") is not None:

        random.setstate(
            state["python"]
        )

    if state.get("numpy") is not None:

        np.random.set_state(
            state["numpy"]
        )

    if state.get("torch") is not None:

        torch.set_rng_state(
            state["torch"]
        )

    if (
        torch.cuda.is_available()
        and state.get(
            "torch_cuda"
        ) is not None
    ):

        torch.cuda.set_rng_state_all(
            state["torch_cuda"]
        )


# =========================================================================
# CHECKPOINT INDEX
# =========================================================================

def _default_checkpoint_index():

    return {

        "schema_version":
            CHECKPOINT_SCHEMA_VERSION,

        "experiment_id":
            EXPERIMENT_ID,

        "architecture_name":
            ARCHITECTURE_NAME,

        "architecture_version":
            ARCHITECTURE_VERSION,

        "run_mode":
            RUN_MODE,

        "config_fingerprint":
            CONFIG_FINGERPRINT,

        "best_checkpoints": [],
    }


def load_checkpoint_index():

    if not CHECKPOINT_INDEX_PATH.exists():

        return (
            _default_checkpoint_index()
        )

    try:

        with open(
            CHECKPOINT_INDEX_PATH,
            "r",
            encoding="utf-8",
        ) as f:

            index = json.load(f)

    except Exception:

        return (
            _default_checkpoint_index()
        )

    # -------------------------------------------------------------
    # Never trust an incompatible index.
    # -------------------------------------------------------------
    index_identity = {

        "experiment_id":
            index.get(
                "experiment_id"
            ),

        "architecture_name":
            index.get(
                "architecture_name"
            ),

        "architecture_version":
            index.get(
                "architecture_version"
            ),

        "run_mode":
            index.get(
                "run_mode"
            ),

        "config_fingerprint":
            index.get(
                "config_fingerprint"
            ),
    }

    expected_identity = {

        "experiment_id":
            EXPERIMENT_ID,

        "architecture_name":
            ARCHITECTURE_NAME,

        "architecture_version":
            ARCHITECTURE_VERSION,

        "run_mode":
            RUN_MODE,

        "config_fingerprint":
            CONFIG_FINGERPRINT,
    }

    if index_identity != expected_identity:

        return (
            _default_checkpoint_index()
        )

    if not isinstance(
        index.get(
            "best_checkpoints"
        ),
        list,
    ):

        index["best_checkpoints"] = []

    return index


def save_checkpoint_index(
    index,
):

    CHECKPOINT_INDEX_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        CHECKPOINT_INDEX_PATH.with_suffix(
            ".tmp"
        )
    )

    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            index,
            f,
            ensure_ascii=False,
            indent=2,
        )

    temporary_path.replace(
        CHECKPOINT_INDEX_PATH
    )


# =========================================================================
# BUILD CHECKPOINT
# =========================================================================

def build_checkpoint(
    training_state,
):

    checkpoint = {

        # -------------------------------------------------------------
        # Identity
        # -------------------------------------------------------------
        "schema_version":
            CHECKPOINT_SCHEMA_VERSION,

        "experiment_id":
            EXPERIMENT_ID,

        "architecture_name":
            ARCHITECTURE_NAME,

        "architecture_version":
            ARCHITECTURE_VERSION,

        "run_mode":
            RUN_MODE,

        "config_fingerprint":
            CONFIG_FINGERPRINT,

        # -------------------------------------------------------------
        # Configuration snapshots
        # -------------------------------------------------------------
        "model_config":
            dict(MODEL_CONFIG),

        "train_config":
            dict(TRAIN_CONFIG),

        "optimizer_config":
            dict(OPTIMIZER_CONFIG),

        # -------------------------------------------------------------
        # Model
        # -------------------------------------------------------------
        "model_state_dict":
            MODEL.state_dict(),

        # -------------------------------------------------------------
        # Optimization
        # -------------------------------------------------------------
        "optimizer_state_dict":
            OPTIMIZER.state_dict(),

        "scheduler_state_dict":
            LR_SCHEDULER.state_dict(),

        "scaler_state_dict":
            GRAD_SCALER.state_dict()
            if GRAD_SCALER is not None
            else None,

        # -------------------------------------------------------------
        # Sampler states
        # -------------------------------------------------------------
        "train_sampler_state":
            TRAIN_SAMPLER.get_state(),

        "val_sampler_state":
            VAL_SAMPLER.get_state(),

        # -------------------------------------------------------------
        # Training state
        # -------------------------------------------------------------
        "training_state":
            dict(training_state),

        # -------------------------------------------------------------
        # RNG
        # -------------------------------------------------------------
        "rng_state":
            capture_rng_state(),

        # -------------------------------------------------------------
        # Metadata
        # -------------------------------------------------------------
        "saved_timestamp":
            time.time(),

        "current_learning_rate":
            float(
                OPTIMIZER.param_groups[0]["lr"]
            ),
    }

    return checkpoint


# =========================================================================
# ATOMIC TORCH SAVE
# =========================================================================

def _atomic_torch_save(
    checkpoint,
    path,
):

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        path.with_suffix(
            ".tmp"
        )
    )

    torch.save(
        checkpoint,
        temporary_path,
    )

    temporary_path.replace(
        path
    )


# =========================================================================
# SAVE LATEST CHECKPOINT
# =========================================================================

def save_latest_checkpoint(
    training_state,
):

    checkpoint = build_checkpoint(
        training_state
    )

    _atomic_torch_save(
        checkpoint,
        LATEST_CHECKPOINT_PATH,
    )

    return LATEST_CHECKPOINT_PATH


# =========================================================================
# HISTORICAL BEST CHECKPOINT REGISTRATION
# =========================================================================

def _register_best_checkpoint(
    step,
    validation_loss,
    checkpoint_path,
):

    index = load_checkpoint_index()

    entry = {

        "step":
            int(step),

        "validation_loss":
            float(validation_loss),

        "path":
            str(checkpoint_path),
    }

    entries = index[
        "best_checkpoints"
    ]

    entries.append(
        entry
    )

    # Sort from best validation loss to worst.
    # This makes inspection easier, while filenames remain
    # chronological by step.
    entries.sort(
        key=lambda item: (
            item["validation_loss"],
            item["step"],
        )
    )

    # Keep at most max_best_checkpoints historical files.
    max_best = int(
        TRAIN_CONFIG[
            "max_best_checkpoints"
        ]
    )

    while len(entries) > max_best:

        removed = entries.pop()

        removed_path = Path(
            removed["path"]
        )

        # Never remove best.pt here.
        if (
            removed_path.exists()
            and removed_path != BEST_CHECKPOINT_PATH
        ):

            try:

                removed_path.unlink()

            except OSError:

                pass

    # Re-sort chronologically for the saved index.
    entries.sort(
        key=lambda item: item["step"]
    )

    save_checkpoint_index(
        index
    )


# =========================================================================
# SAVE BEST CHECKPOINT
# =========================================================================

def save_best_checkpoint(
    training_state,
    validation_loss,
):

    step = int(
        training_state["step"]
    )

    loss_for_filename = (
        f"{float(validation_loss):.6f}"
    )

    historical_path = (
        CHECKPOINTS_DIR
        / (
            "best_step_"
            f"{step:06d}"
            "_loss_"
            f"{loss_for_filename}"
            ".pt"
        )
    )

    checkpoint = build_checkpoint(
        training_state
    )

    # -------------------------------------------------------------
    # Current-best checkpoint.
    # -------------------------------------------------------------
    _atomic_torch_save(
        checkpoint,
        BEST_CHECKPOINT_PATH,
    )

    # -------------------------------------------------------------
    # Historical best checkpoint.
    # -------------------------------------------------------------
    _atomic_torch_save(
        checkpoint,
        historical_path,
    )

    _register_best_checkpoint(
        step=step,
        validation_loss=validation_loss,
        checkpoint_path=historical_path,
    )

    return (
        BEST_CHECKPOINT_PATH,
        historical_path,
    )


# =========================================================================
# LOAD LATEST CHECKPOINT
# =========================================================================

def load_latest_checkpoint():

    if not LATEST_CHECKPOINT_PATH.exists():

        return None

    try:

        checkpoint = torch.load(
            LATEST_CHECKPOINT_PATH,
            map_location="cpu",
            weights_only=False,
        )

    except Exception as exc:

        print(
            "⚠️ Could not load latest checkpoint: "
            f"{exc}"
        )

        return None

    if not isinstance(
        checkpoint,
        dict,
    ):

        print(
            "⚠️ Latest checkpoint has invalid format."
        )

        return None

    if not checkpoint_is_compatible(
        checkpoint
    ):

        print(
            "ℹ️ Existing latest checkpoint is "
            "incompatible with the current "
            "Architecture-B configuration."
        )

        return None

    try:

        MODEL.load_state_dict(
            checkpoint[
                "model_state_dict"
            ]
        )

        OPTIMIZER.load_state_dict(
            checkpoint[
                "optimizer_state_dict"
            ]
        )

        LR_SCHEDULER.load_state_dict(
            checkpoint[
                "scheduler_state_dict"
            ]
        )

        scaler_state = checkpoint.get(
            "scaler_state_dict"
        )

        if (
            GRAD_SCALER is not None
            and scaler_state is not None
        ):

            GRAD_SCALER.load_state_dict(
                scaler_state
            )

        TRAIN_SAMPLER.set_state(
            checkpoint[
                "train_sampler_state"
            ]
        )

        VAL_SAMPLER.set_state(
            checkpoint[
                "val_sampler_state"
            ]
        )

        restore_rng_state(
            checkpoint[
                "rng_state"
            ]
        )

        training_state = dict(
            checkpoint.get(
                "training_state",
                {},
            )
        )

    except Exception as exc:

        print(
            "⚠️ Checkpoint restore failed: "
            f"{exc}"
        )

        return None

    print()
    print(
        "✅ Resumed compatible Architecture-B checkpoint."
    )

    print(
        f"Step              : "
        f"{training_state.get('step', 0)}"
    )

    print(
        f"Tokens seen       : "
        f"{training_state.get('tokens_seen', 0):,}"
    )

    print(
        f"Best validation   : "
        f"{training_state.get('best_validation_loss')}"
    )

    print(
        f"Best checkpoints  : "
        f"{training_state.get('best_checkpoint_count', 0)}"
    )

    return training_state


# =========================================================================
# CHECKPOINT INSPECTION
# =========================================================================

def inspect_existing_checkpoints():

    print()
    print(
        "📋 Architecture-B checkpoint inventory"
    )

    if LATEST_CHECKPOINT_PATH.exists():

        print(
            f"  ✅ latest.pt"
            f" ({LATEST_CHECKPOINT_PATH.stat().st_size / (1024**2):.2f} MB)"
        )

    else:

        print(
            "  — latest.pt not present"
        )

    if BEST_CHECKPOINT_PATH.exists():

        print(
            f"  ✅ best.pt"
            f" ({BEST_CHECKPOINT_PATH.stat().st_size / (1024**2):.2f} MB)"
        )

    else:

        print(
            "  — best.pt not present"
        )

    index = load_checkpoint_index()

    entries = index.get(
        "best_checkpoints",
        [],
    )

    print(
        f"  Historical best checkpoints: "
        f"{len(entries)}"
    )

    for entry in entries:

        print(
            f"    step={entry['step']:>6} | "
            f"loss={entry['validation_loss']:.6f}"
        )


# =========================================================================
# SAVE CHECKPOINT MANIFEST
# =========================================================================

CHECKPOINT_MANIFEST = {

    "experiment_id":
        EXPERIMENT_ID,

    "architecture_name":
        ARCHITECTURE_NAME,

    "architecture_version":
        ARCHITECTURE_VERSION,

    "run_mode":
        RUN_MODE,

    "config_fingerprint":
        CONFIG_FINGERPRINT,

    "schema_version":
        CHECKPOINT_SCHEMA_VERSION,

    "latest_checkpoint":
        str(LATEST_CHECKPOINT_PATH),

    "best_checkpoint":
        str(BEST_CHECKPOINT_PATH),

    "checkpoint_index":
        str(CHECKPOINT_INDEX_PATH),

    "max_best_checkpoints":
        TRAIN_CONFIG[
            "max_best_checkpoints"
        ],

    "min_best_checkpoints":
        TRAIN_CONFIG[
            "min_best_checkpoints"
        ],

    "patience":
        TRAIN_CONFIG[
            "patience"
        ],

    "min_delta":
        TRAIN_CONFIG[
            "min_delta"
        ],

    "min_steps_before_plateau_stop":
        TRAIN_CONFIG[
            "min_steps_before_plateau_stop"
        ],

    "rng_checkpointed":
        True,

    "sampler_state_checkpointed":
        True,
}


CHECKPOINT_MANIFEST_PATH = (
    CONFIGS_DIR
    / "architecture_B_checkpointing.json"
)

with open(
    CHECKPOINT_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        CHECKPOINT_MANIFEST,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# INITIAL INSPECTION
# =========================================================================

inspect_existing_checkpoints()


# =========================================================================
# SUMMARY
# =========================================================================

print()
print(
    "✅ Architecture-B checkpoint system ready."
)

print(
    f"Schema version         : "
    f"{CHECKPOINT_SCHEMA_VERSION}"
)

print(
    f"Latest checkpoint     : "
    f"{LATEST_CHECKPOINT_PATH}"
)

print(
    f"Best checkpoint       : "
    f"{BEST_CHECKPOINT_PATH}"
)

print(
    f"Historical best limit : "
    f"{TRAIN_CONFIG['max_best_checkpoints']}"
)

print(
    f"Minimum best count    : "
    f"{TRAIN_CONFIG['min_best_checkpoints']}"
)

print(
    f"Patience              : "
    f"{TRAIN_CONFIG['patience']}"
)

print(
    f"Minimum improvement   : "
    f"{TRAIN_CONFIG['min_delta']}"
)

print(
    f"RNG state saved       : True"
)

print(
    f"Sampler state saved   : True"
)

print(
    f"Manifest              : "
    f"{CHECKPOINT_MANIFEST_PATH}"
)

💾 Configuring Architecture-B checkpoint and resume system...

📋 Architecture-B checkpoint inventory
  — latest.pt not present
  — best.pt not present
  Historical best checkpoints: 0

✅ Architecture-B checkpoint system ready.
Schema version         : 1
Latest checkpoint     : /kaggle/working/architecture_B/checkpoints/latest.pt
Best checkpoint       : /kaggle/working/architecture_B/checkpoints/best.pt
Historical best limit : 12
Minimum best count    : 8
Patience              : 4
Minimum improvement   : 0.005
RNG state saved       : True
Sampler state saved   : True
Manifest              : /kaggle/working/architecture_B/configs/architecture_B_checkpointing.json


In [8]:
# Cell 07.5 — Restore Architecture-B checkpoints from attached Kaggle Dataset
# DECISION: UPDATE

from pathlib import Path
import shutil

TARGET_ROOT = Path("/kaggle/working/architecture_B")
TARGET_CHECKPOINT_DIR = TARGET_ROOT / "checkpoints"
TARGET_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_ROOT = Path("/kaggle/input")

print("🔍 Searching all attached Kaggle datasets for Architecture-B checkpoint...")

# Find every latest.pt under /kaggle/input
latest_candidates = sorted(INPUT_ROOT.rglob("latest.pt"))

if not latest_candidates:
    raise FileNotFoundError(
        "❌ No latest.pt found anywhere under /kaggle/input.\n"
        "Check that Arch_Data_1 is attached to this notebook."
    )

print(f"✅ Found {len(latest_candidates)} latest.pt file(s):")
for p in latest_candidates:
    print(f"   {p}")

# Prefer a checkpoint inside architecture_B/checkpoints/
preferred = [
    p for p in latest_candidates
    if "architecture_B" in p.parts and "checkpoints" in p.parts
]

source_latest = preferred[0] if preferred else latest_candidates[0]
source_checkpoint_dir = source_latest.parent

print("\n📦 Using checkpoint directory:")
print(f"   {source_checkpoint_dir}")

# Restore latest.pt and best.pt
for filename in ["latest.pt", "best.pt"]:
    src = source_checkpoint_dir / filename
    dst = TARGET_CHECKPOINT_DIR / filename

    if src.exists():
        print(f"⏳ Copying {filename} ...")
        shutil.copy2(src, dst)
        print(f"✅ Restored {filename}")
    else:
        print(f"⚠️ {filename} not found — skipping")

# Restore configs and metrics if present
source_arch_root = source_checkpoint_dir.parent

for dirname in ["configs", "metrics"]:
    src_dir = source_arch_root / dirname
    dst_dir = TARGET_ROOT / dirname

    if src_dir.exists():
        print(f"📁 Restoring {dirname}/ ...")
        shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
        print(f"✅ Restored {dirname}/")

# Final verification
latest_path = TARGET_CHECKPOINT_DIR / "latest.pt"
best_path = TARGET_CHECKPOINT_DIR / "best.pt"

print("\n" + "=" * 65)
print("✅ RESTORE COMPLETE")
print("=" * 65)

print(f"latest.pt exists : {latest_path.exists()}")
if latest_path.exists():
    print(f"latest.pt size   : {latest_path.stat().st_size / (1024**3):.2f} GB")
    print(f"latest.pt path   : {latest_path}")

print(f"best.pt exists   : {best_path.exists()}")
if best_path.exists():
    print(f"best.pt size     : {best_path.stat().st_size / (1024**3):.2f} GB")

print("=" * 65)

🔍 Searching all attached Kaggle datasets for Architecture-B checkpoint...
✅ Found 1 latest.pt file(s):
   /kaggle/input/datasets/rohitbatra2004/arch-data-1/architecture_B/checkpoints/latest.pt

📦 Using checkpoint directory:
   /kaggle/input/datasets/rohitbatra2004/arch-data-1/architecture_B/checkpoints
⏳ Copying latest.pt ...
✅ Restored latest.pt
⏳ Copying best.pt ...
✅ Restored best.pt
📁 Restoring configs/ ...
✅ Restored configs/
📁 Restoring metrics/ ...
✅ Restored metrics/

✅ RESTORE COMPLETE
latest.pt exists : True
latest.pt size   : 1.56 GB
latest.pt path   : /kaggle/working/architecture_B/checkpoints/latest.pt
best.pt exists   : True
best.pt size     : 0.52 GB


In [9]:
# Cell 08 — Architecture-B training engine with numerical-overflow recovery
# DECISION: UPDATE

import os
import json
import math
import time
import random
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F


# =========================================================================
# RUNTIME / DEVICE
# =========================================================================

DEVICE = globals().get(
    "DEVICE",
    torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    ),
)

if isinstance(
    DEVICE,
    str,
):
    DEVICE = torch.device(DEVICE)

DEVICE_TYPE = DEVICE.type

MODEL_TYPE = (
    "encoder-decoder-transformer"
)


assert "MODEL" in globals(), (
    "MODEL is unavailable. "
    "Run Cell 05 first."
)

assert "TRAIN_CONFIG" in globals(), (
    "TRAIN_CONFIG is unavailable. "
    "Run Cell 01 first."
)

assert "OPTIMIZER_CONFIG" in globals(), (
    "OPTIMIZER_CONFIG is unavailable. "
    "Run Cell 01 first."
)

assert "CONFIG_FINGERPRINT" in globals(), (
    "CONFIG_FINGERPRINT is unavailable. "
    "Run Cell 01 first."
)


MODEL = MODEL.to(
    DEVICE
)


# =========================================================================
# PATHS
# =========================================================================

ARCHITECTURE_B_ROOT = Path(
    globals().get(
        "ARCHITECTURE_B_ROOT",
        "/kaggle/working/architecture_B",
    )
)

CHECKPOINTS_DIR = (
    ARCHITECTURE_B_ROOT
    / "checkpoints"
)

METRICS_DIR = (
    ARCHITECTURE_B_ROOT
    / "metrics"
)

CHECKPOINTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

BEST_CHECKPOINT_PATH = (
    CHECKPOINTS_DIR
    / "best.pt"
)

LATEST_CHECKPOINT_PATH = (
    CHECKPOINTS_DIR
    / "latest.pt"
)

TRAINING_METRICS_PATH = (
    METRICS_DIR
    / "training_metrics_B.json"
)


# =========================================================================
# TRAINING CONFIG
# =========================================================================

MAX_STEPS = int(
    TRAIN_CONFIG[
        "max_steps"
    ]
)

MICRO_BATCH_SIZE = int(
    TRAIN_CONFIG[
        "micro_batch_size"
    ]
)

GRADIENT_ACCUMULATION_STEPS = int(
    TRAIN_CONFIG[
        "gradient_accumulation_steps"
    ]
)

EVAL_INTERVAL_STEPS = int(
    TRAIN_CONFIG[
        "eval_interval_steps"
    ]
)

CHECKPOINT_INTERVAL_STEPS = int(
    TRAIN_CONFIG[
        "checkpoint_interval_steps"
    ]
)

TRAIN_TOKEN_BUDGET = int(
    TRAIN_CONFIG[
        "train_token_budget"
    ]
)

VALIDATION_TOKEN_BUDGET = int(
    TRAIN_CONFIG[
        "validation_token_budget"
    ]
)

MIN_BEST_CHECKPOINTS = int(
    TRAIN_CONFIG[
        "min_best_checkpoints"
    ]
)

MAX_BEST_CHECKPOINTS = int(
    TRAIN_CONFIG[
        "max_best_checkpoints"
    ]
)

PATIENCE = int(
    TRAIN_CONFIG[
        "patience"
    ]
)

MIN_DELTA = float(
    TRAIN_CONFIG[
        "min_delta"
    ]
)

MIN_STEPS_BEFORE_STOP = int(
    TRAIN_CONFIG[
        "min_steps_before_plateau_stop"
    ]
)

EARLY_STOPPING_ENABLED = bool(
    TRAIN_CONFIG[
        "early_stopping"
    ]
)


assert MAX_STEPS > 0, (
    "TRAIN_CONFIG['max_steps'] must be > 0."
)


# =========================================================================
# OPTIMIZER CONFIG
# =========================================================================

LEARNING_RATE = float(
    OPTIMIZER_CONFIG[
        "learning_rate"
    ]
)

MIN_LEARNING_RATE = float(
    OPTIMIZER_CONFIG[
        "min_learning_rate"
    ]
)

WEIGHT_DECAY = float(
    OPTIMIZER_CONFIG[
        "weight_decay"
    ]
)

WARMUP_FRACTION = float(
    OPTIMIZER_CONFIG[
        "warmup_fraction"
    ]
)

GRADIENT_CLIP_NORM = float(
    OPTIMIZER_CONFIG[
        "gradient_clip_norm"
    ]
)


# =========================================================================
# AMP
# =========================================================================

AMP_ENABLED = (
    DEVICE_TYPE == "cuda"
)

AMP_DTYPE = (
    torch.float16
    if AMP_ENABLED
    else torch.float32
)


def autocast_context_B():
    """
    Architecture-B AMP context.
    """

    if not AMP_ENABLED:

        return nullcontext()


    return torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
        enabled=True,
    )


try:

    _GradScalerClass = (
        torch.amp.GradScaler
    )


    def create_grad_scaler_B():

        return _GradScalerClass(
            "cuda",
            enabled=AMP_ENABLED,
            init_scale=2**14,
            growth_factor=2.0,
            backoff_factor=0.5,
            growth_interval=2000,
        )

except AttributeError:


    def create_grad_scaler_B():

        return torch.cuda.amp.GradScaler(
            enabled=AMP_ENABLED,
            init_scale=2**14,
            growth_factor=2.0,
            backoff_factor=0.5,
            growth_interval=2000,
        )


# =========================================================================
# SAMPLER RESOLUTION
# =========================================================================

def resolve_sampler_B(
    role,
):

    if role == "train":

        candidates = (
            "TRAIN_SAMPLER",
            "TRAIN_SAMPLER_B",
            "train_sampler",
            "train_sampler_B",
            "SAMPLER_TRAIN",
        )

    else:

        candidates = (
            "VAL_SAMPLER",
            "VAL_SAMPLER_B",
            "val_sampler",
            "validation_sampler",
            "SAMPLER_VAL",
        )


    for name in candidates:

        value = globals().get(
            name
        )

        if value is not None:

            return (
                value,
                name,
            )


    # Fallback: find an EncoderDecoderTokenSampler instance.
    sampler_class = globals().get(
        "EncoderDecoderTokenSampler"
    )


    if sampler_class is not None:

        for name, value in globals().items():

            if not isinstance(
                value,
                sampler_class,
            ):
                continue


            name_lower = name.lower()


            if role == "train":

                if (
                    "train"
                    in name_lower
                ):

                    return (
                        value,
                        name,
                    )

            else:

                if (
                    "val"
                    in name_lower
                    or "valid"
                    in name_lower
                ):

                    return (
                        value,
                        name,
                    )


    return (
        None,
        None,
    )


TRAIN_SAMPLER, TRAIN_SAMPLER_NAME = (
    resolve_sampler_B("train")
)

VAL_SAMPLER, VAL_SAMPLER_NAME = (
    resolve_sampler_B("val")
)


assert TRAIN_SAMPLER is not None, (
    "Training sampler could not be resolved. "
    "Run Cell 04 first."
)

assert VAL_SAMPLER is not None, (
    "Validation sampler could not be resolved. "
    "Run Cell 04 first."
)


# =========================================================================
# DRAW ONE BATCH FROM SAMPLER
# =========================================================================

def sampler_next_batch_B(
    sampler,
    batch_size,
):

    methods = (
        "sample_batch",
        "sample",
        "next_batch",
        "get_batch",
    )

    last_error = None


    for method_name in methods:

        method = getattr(
            sampler,
            method_name,
            None,
        )

        if not callable(
            method
        ):
            continue


        try:

            return method(
                batch_size=batch_size
            )

        except TypeError as exc:

            last_error = exc


        try:

            return method(
                batch_size
            )

        except TypeError as exc:

            last_error = exc


        try:

            return method()

        except TypeError as exc:

            last_error = exc


    raise RuntimeError(
        f"Could not obtain a batch from "
        f"{type(sampler).__name__}. "
        f"Tried: {methods}. "
        f"Last error: {last_error}"
    )


# =========================================================================
# BATCH EXTRACTION
# =========================================================================

def _find_batch_value(
    batch,
    keys,
):

    for key in keys:

        if key in batch:

            return batch[key]


    return None


def get_decoder_start_token_id_B():

    # Direct globals.
    for name in (
        "DECODER_START_ID",
        "DECODER_START_TOKEN_ID",
        "START_ID",
        "START_TOKEN_ID",
    ):

        value = globals().get(
            name
        )

        if value is not None:

            return int(
                value
            )


    # Dictionaries.
    for container_name in (
        "SPECIAL_TOKEN_IDS",
        "SPECIAL_IDS",
        "TOKEN_IDS",
    ):

        container = globals().get(
            container_name
        )

        if not isinstance(
            container,
            dict,
        ):
            continue


        for key in (
            "decoder_start",
            "decoder_start_id",
            "DECODER_START",
            "[DECODER_START]",
            "start",
            "bos",
        ):

            if key in container:

                return int(
                    container[key]
                )


    # Tokenizer lookup.
    tokenizer_obj = globals().get(
        "tokenizer"
    )

    if tokenizer_obj is not None:

        token_to_id = getattr(
            tokenizer_obj,
            "token_to_id",
            None,
        )

        if callable(
            token_to_id
        ):

            for token in (
                "[DECODER_START]",
                "<decoder_start>",
                "<s>",
                "<bos>",
            ):

                token_id = token_to_id(
                    token
                )

                if token_id is not None:

                    return int(
                        token_id
                    )


    # Model attribute fallback.
    for attr in (
        "decoder_start_token_id",
        "decoder_start_id",
        "start_token_id",
    ):

        value = getattr(
            MODEL,
            attr,
            None,
        )

        if value is not None:

            return int(
                value
            )


    return None


def unpack_batch_B(
    batch,
):

    # ---------------------------------------------------------------------
    # Dictionary output.
    # ---------------------------------------------------------------------

    if isinstance(
        batch,
        dict,
    ):

        encoder_ids = _find_batch_value(
            batch,
            (
                "encoder_input_ids",
                "encoder_ids",
                "src_tokens",
                "source_tokens",
                "src",
                "source",
                "encoder_input",
            ),
        )


        decoder_ids = _find_batch_value(
            batch,
            (
                "decoder_input_ids",
                "decoder_ids",
                "tgt_input_ids",
                "decoder_input",
                "decoder_inputs",
            ),
        )


        labels = _find_batch_value(
            batch,
            (
                "labels",
                "target_ids",
                "target_tokens",
                "targets",
                "target",
                "tgt_tokens",
                "tgt",
            ),
        )


        assert (
            encoder_ids is not None
        ), (
            "Sampler dictionary is missing "
            "encoder input IDs. "
            f"Keys: {list(batch.keys())}"
        )


        assert (
            labels is not None
        ), (
            "Sampler dictionary is missing "
            "target/label IDs. "
            f"Keys: {list(batch.keys())}"
        )


    # ---------------------------------------------------------------------
    # Tuple/list output.
    # ---------------------------------------------------------------------

    elif isinstance(
        batch,
        (tuple, list),
    ):

        if len(batch) >= 3:

            encoder_ids = batch[0]
            decoder_ids = batch[1]
            labels = batch[2]

        elif len(batch) == 2:

            encoder_ids = batch[0]
            decoder_ids = None
            labels = batch[1]

        else:

            raise ValueError(
                "Sampler returned fewer than "
                "two elements."
            )

    else:

        raise TypeError(
            "Unsupported sampler batch type: "
            f"{type(batch)}"
        )


    encoder_ids = torch.as_tensor(
        encoder_ids,
        dtype=torch.long,
    )

    labels = torch.as_tensor(
        labels,
        dtype=torch.long,
    )


    # ---------------------------------------------------------------------
    # Build decoder input if sampler only returns source + target.
    # ---------------------------------------------------------------------

    if decoder_ids is None:

        decoder_start_id = (
            get_decoder_start_token_id_B()
        )


        assert (
            decoder_start_id is not None
        ), (
            "Decoder input IDs were not returned "
            "by the sampler and the "
            "decoder-start token could not be resolved."
        )


        assert labels.ndim == 2


        decoder_ids = torch.empty_like(
            labels
        )


        decoder_ids[:, 0] = (
            decoder_start_id
        )


        if labels.size(1) > 1:

            decoder_ids[:, 1:] = (
                labels[:, :-1]
            )

    else:

        decoder_ids = torch.as_tensor(
            decoder_ids,
            dtype=torch.long,
        )


    assert encoder_ids.ndim == 2

    assert decoder_ids.ndim == 2

    assert labels.ndim == 2


    assert (
        decoder_ids.shape
        == labels.shape
    ), (
        "Decoder inputs and labels must "
        "have identical shapes.\n"
        f"Decoder: {decoder_ids.shape}\n"
        f"Labels : {labels.shape}"
    )


    return (
        encoder_ids,
        decoder_ids,
        labels,
    )


def move_batch_to_device_B(
    batch,
):

    (
        encoder_ids,
        decoder_ids,
        labels,
    ) = unpack_batch_B(
        batch
    )


    return (
        encoder_ids.to(
            DEVICE,
            non_blocking=True,
        ),
        decoder_ids.to(
            DEVICE,
            non_blocking=True,
        ),
        labels.to(
            DEVICE,
            non_blocking=True,
        ),
    )


# =========================================================================
# MODEL FORWARD
# =========================================================================

def model_forward_B(
    encoder_ids,
    decoder_ids,
):

    attempts = (

        lambda: MODEL(
            encoder_ids,
            decoder_ids,
        ),

        lambda: MODEL(
            encoder_input_ids=encoder_ids,
            decoder_input_ids=decoder_ids,
        ),

        lambda: MODEL(
            src_tokens=encoder_ids,
            decoder_input_ids=decoder_ids,
        ),

        lambda: MODEL(
            src=encoder_ids,
            decoder_input=decoder_ids,
        ),
    )


    last_error = None


    for attempt in attempts:

        try:

            output = attempt()


            if isinstance(
                output,
                dict,
            ):

                if "logits" in output:

                    return output[
                        "logits"
                    ]


                for value in output.values():

                    if torch.is_tensor(
                        value
                    ):

                        return value


            elif isinstance(
                output,
                (tuple, list),
            ):

                for value in output:

                    if torch.is_tensor(
                        value
                    ):

                        return value


            elif torch.is_tensor(
                output
            ):

                return output


            else:

                raise TypeError(
                    "Model output contains "
                    "no tensor logits."
                )


        except (
            TypeError,
            ValueError,
        ) as exc:

            last_error = exc


    raise RuntimeError(
        "Architecture-B forward pass could "
        "not be resolved.\n"
        f"Last error: {last_error}"
    )


# =========================================================================
# LOSS
# =========================================================================

def architecture_B_loss(
    logits,
    labels,
):

    assert logits.ndim == 3

    assert labels.ndim == 2

    assert (
        logits.shape[0]
        == labels.shape[0]
    )

    assert (
        logits.shape[1]
        == labels.shape[1]
    )


    # Explicit FP32 logits for numerically stable
    # cross-entropy computation.
    logits = logits.float()


    return F.cross_entropy(
        logits.reshape(
            -1,
            logits.size(-1),
        ),
        labels.reshape(-1),
    )


# =========================================================================
# OPTIMIZER
# =========================================================================

def build_optimizer_B():

    decay_parameters = []
    no_decay_parameters = []


    for name, parameter in (
        MODEL.named_parameters()
    ):

        if not parameter.requires_grad:
            continue


        if (
            parameter.ndim >= 2
            and not name.endswith(
                ".bias"
            )
        ):

            decay_parameters.append(
                parameter
            )

        else:

            no_decay_parameters.append(
                parameter
            )


    optimizer = torch.optim.AdamW(

        [
            {
                "params":
                    decay_parameters,

                "weight_decay":
                    WEIGHT_DECAY,
            },

            {
                "params":
                    no_decay_parameters,

                "weight_decay":
                    0.0,
            },
        ],

        lr=LEARNING_RATE,

        betas=(
            0.9,
            0.95,
        ),

        eps=1e-8,
    )


    return optimizer


# =========================================================================
# LR SCHEDULE
# =========================================================================

WARMUP_STEPS = max(
    1,
    int(
        round(
            MAX_STEPS
            * WARMUP_FRACTION
        )
    ),
)

MIN_LR_RATIO = (
    MIN_LEARNING_RATE
    / LEARNING_RATE
)


def lr_lambda_B(
    current_step,
):

    if current_step < WARMUP_STEPS:

        return max(
            1e-8,
            (
                current_step + 1
            )
            / WARMUP_STEPS,
        )


    progress = (
        current_step
        - WARMUP_STEPS
    ) / max(
        1,
        MAX_STEPS
        - WARMUP_STEPS,
    )


    progress = min(
        max(
            progress,
            0.0,
        ),
        1.0,
    )


    cosine_factor = (
        0.5
        * (
            1.0
            + math.cos(
                math.pi
                * progress
            )
        )
    )


    return (
        MIN_LR_RATIO
        + (
            1.0
            - MIN_LR_RATIO
        )
        * cosine_factor
    )


# =========================================================================
# RNG CHECKPOINTING
# =========================================================================

def capture_rng_state_B():

    state = {

        "python":
            random.getstate(),

        "numpy":
            np.random.get_state(),

        "torch":
            torch.get_rng_state(),
    }


    if torch.cuda.is_available():

        state[
            "torch_cuda"
        ] = (
            torch.cuda.get_rng_state_all()
        )


    return state


def restore_rng_state_B(
    state,
):

    if not isinstance(
        state,
        dict,
    ):

        return


    if "python" in state:

        random.setstate(
            state["python"]
        )


    if "numpy" in state:

        np.random.set_state(
            state["numpy"]
        )


    if "torch" in state:

        torch.set_rng_state(
            state["torch"]
        )


    if (
        torch.cuda.is_available()
        and "torch_cuda" in state
    ):

        try:

            torch.cuda.set_rng_state_all(
                state["torch_cuda"]
            )

        except Exception:

            pass


# =========================================================================
# SAMPLER STATE CHECKPOINTING
# =========================================================================

def capture_sampler_state_B(
    sampler,
):

    if sampler is None:
        return None


    method = getattr(
        sampler,
        "state_dict",
        None,
    )


    if callable(
        method
    ):

        try:

            return method()

        except Exception:

            pass


    method = getattr(
        sampler,
        "get_state",
        None,
    )


    if callable(
        method
    ):

        try:

            return method()

        except Exception:

            pass


    return None


def restore_sampler_state_B(
    sampler,
    state,
):

    if (
        sampler is None
        or state is None
    ):

        return


    method = getattr(
        sampler,
        "load_state_dict",
        None,
    )


    if callable(
        method
    ):

        try:

            method(
                state
            )

            return

        except Exception:

            pass


    method = getattr(
        sampler,
        "set_state",
        None,
    )


    if callable(
        method
    ):

        try:

            method(
                state
            )

        except Exception:

            pass


# =========================================================================
# CHECKPOINT HELPERS
# =========================================================================

def atomic_torch_save_B(
    object_to_save,
    path,
):

    path = Path(
        path
    )


    temporary_path = (
        path.with_name(
            path.name
            + ".tmp"
        )
    )


    torch.save(
        object_to_save,
        temporary_path,
    )


    os.replace(
        temporary_path,
        path,
    )


def checkpoint_is_compatible_B(
    checkpoint,
):

    if not isinstance(
        checkpoint,
        dict,
    ):

        return False


    required_identity = {

        "experiment_id":
            EXPERIMENT_ID,

        "architecture_name":
            ARCHITECTURE_NAME,

        "architecture_version":
            ARCHITECTURE_VERSION,

        "run_mode":
            RUN_MODE,

        "config_fingerprint":
            CONFIG_FINGERPRINT,
    }


    for key, expected in (
        required_identity.items()
    ):

        if checkpoint.get(
            key
        ) != expected:

            return False


    return (
        "model_state_dict"
        in checkpoint
    )


def save_best_checkpoint_B(
    step,
    tokens_seen,
    best_validation_step,
    best_validation_loss,
    best_validation_perplexity,
    best_checkpoint_count,
    bad_validation_count,
):

    checkpoint = {

        "format_version":
            2,

        "project":
            "CS781 Phase 1",

        "experiment_id":
            EXPERIMENT_ID,

        "architecture_name":
            ARCHITECTURE_NAME,

        "architecture_version":
            ARCHITECTURE_VERSION,

        "run_mode":
            RUN_MODE,

        "config_fingerprint":
            CONFIG_FINGERPRINT,

        "model_type":
            MODEL_TYPE,

        "model_state_dict":
            {
                key:
                    value.detach()
                    .cpu()
                for key, value
                in MODEL.state_dict().items()
            },

        "step":
            int(step),

        "tokens_seen":
            int(tokens_seen),

        "best_validation_step":
            int(
                best_validation_step
            ),

        "best_validation_loss":
            float(
                best_validation_loss
            ),

        "best_validation_perplexity":
            float(
                best_validation_perplexity
            ),

        "best_checkpoint_count":
            int(
                best_checkpoint_count
            ),

        "bad_validation_count":
            int(
                bad_validation_count
            ),

        "model_config":
            dict(
                MODEL_CONFIG
            ),
    }


    historical_path = (
        CHECKPOINTS_DIR
        / f"best_step_{step:07d}.pt"
    )


    atomic_torch_save_B(
        checkpoint,
        historical_path,
    )


    atomic_torch_save_B(
        checkpoint,
        BEST_CHECKPOINT_PATH,
    )


    # Keep only the requested number of historical best checkpoints.
    historical = sorted(
        CHECKPOINTS_DIR.glob(
            "best_step_*.pt"
        )
    )


    while len(
        historical
    ) > MAX_BEST_CHECKPOINTS:

        oldest = historical.pop(
            0
        )

        try:

            oldest.unlink()

        except FileNotFoundError:

            pass


def save_latest_checkpoint_B(
    step,
    tokens_seen,
    best_validation_step,
    best_validation_loss,
    best_validation_perplexity,
    best_checkpoint_count,
    bad_validation_count,
    optimizer,
    scheduler,
    scaler,
    training_history,
    numerical_overflow_count,
):

    checkpoint = {

        "format_version":
            2,

        "project":
            "CS781 Phase 1",

        "experiment_id":
            EXPERIMENT_ID,

        "architecture_name":
            ARCHITECTURE_NAME,

        "architecture_version":
            ARCHITECTURE_VERSION,

        "run_mode":
            RUN_MODE,

        "config_fingerprint":
            CONFIG_FINGERPRINT,

        "model_type":
            MODEL_TYPE,

        "model_state_dict":
            MODEL.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "scaler_state_dict":
            scaler.state_dict(),

        "train_sampler_state":
            capture_sampler_state_B(
                TRAIN_SAMPLER
            ),

        "val_sampler_state":
            capture_sampler_state_B(
                VAL_SAMPLER
            ),

        "rng_state":
            capture_rng_state_B(),

        "step":
            int(step),

        "tokens_seen":
            int(tokens_seen),

        "best_validation_step":
            int(
                best_validation_step
            ),

        "best_validation_loss":
            float(
                best_validation_loss
            ),

        "best_validation_perplexity":
            float(
                best_validation_perplexity
            ),

        "best_checkpoint_count":
            int(
                best_checkpoint_count
            ),

        "bad_validation_count":
            int(
                bad_validation_count
            ),

        "numerical_overflow_count":
            int(
                numerical_overflow_count
            ),

        "model_config":
            dict(
                MODEL_CONFIG
            ),

        "train_config":
            dict(
                TRAIN_CONFIG
            ),

        "optimizer_config":
            dict(
                OPTIMIZER_CONFIG
            ),

        "training_history":
            list(
                training_history
            ),
    }


    atomic_torch_save_B(
        checkpoint,
        LATEST_CHECKPOINT_PATH,
    )


# =========================================================================
# METRICS
# =========================================================================

def perplexity_from_loss_B(
    loss_value,
):

    loss_value = float(
        loss_value
    )


    if not math.isfinite(
        loss_value
    ):

        return float(
            "inf"
        )


    return math.exp(
        min(
            loss_value,
            20.0,
        )
    )


def optimizer_learning_rate_B(
    optimizer,
):

    if not optimizer.param_groups:

        return 0.0


    return float(
        optimizer.param_groups[0][
            "lr"
        ]
    )


# =========================================================================
# VALIDATION
# =========================================================================

@torch.no_grad()
def evaluate_architecture_B(
    num_batches,
):

    MODEL.eval()


    total_loss = 0.0
    total_target_tokens = 0


    for _ in range(
        int(num_batches)
    ):

        batch = sampler_next_batch_B(
            VAL_SAMPLER,
            MICRO_BATCH_SIZE,
        )


        (
            encoder_ids,
            decoder_ids,
            labels,
        ) = move_batch_to_device_B(
            batch
        )


        with autocast_context_B():

            logits = model_forward_B(
                encoder_ids,
                decoder_ids,
            )


            loss = architecture_B_loss(
                logits,
                labels,
            )


        loss_value = float(
            loss.detach()
            .float()
            .item()
        )


        if not math.isfinite(
            loss_value
        ):

            MODEL.train()


            return {

                "loss":
                    float("inf"),

                "perplexity":
                    float("inf"),

                "tokens":
                    0,

                "finite":
                    False,
            }


        target_tokens = int(
            labels.numel()
        )


        total_loss += (
            loss_value
            * target_tokens
        )


        total_target_tokens += (
            target_tokens
        )


    MODEL.train()


    if total_target_tokens == 0:

        return {

            "loss":
                float("inf"),

            "perplexity":
                float("inf"),

            "tokens":
                0,

            "finite":
                False,
        }


    mean_loss = (
        total_loss
        / total_target_tokens
    )


    return {

        "loss":
            float(mean_loss),

        "perplexity":
            perplexity_from_loss_B(
                mean_loss
            ),

        "tokens":
            int(
                total_target_tokens
            ),

        "finite":
            math.isfinite(
                mean_loss
            ),
    }


# =========================================================================
# VALIDATION WORKLOAD
# =========================================================================

if RUN_MODE == "fast":

    EVAL_BATCHES = 4

else:

    EVAL_BATCHES = 100


# =========================================================================
# TRAINING FUNCTION
# =========================================================================

def run_architecture_B_training():

    global MODEL


    start_time = (
        time.perf_counter()
    )


    # ---------------------------------------------------------------------
    # Create optimizer / scheduler / scaler.
    # ---------------------------------------------------------------------

    optimizer = (
        build_optimizer_B()
    )


    scheduler = (
        torch.optim.lr_scheduler.LambdaLR(
            optimizer,
            lr_lambda_B,
        )
    )


    scaler = (
        create_grad_scaler_B()
    )


    # ---------------------------------------------------------------------
    # Initial state.
    # ---------------------------------------------------------------------

    step = 0

    tokens_seen = 0

    best_validation_step = 0

    best_validation_loss = (
        float("inf")
    )

    best_validation_perplexity = (
        float("inf")
    )

    best_checkpoint_count = 0

    bad_validation_count = 0

    numerical_overflow_count = 0

    consecutive_numerical_overflows = 0

    training_history = []

    resumed = False


    # ---------------------------------------------------------------------
    # Resume from compatible latest checkpoint.
    # ---------------------------------------------------------------------

    if LATEST_CHECKPOINT_PATH.is_file():

        try:

            latest_checkpoint = (
                torch.load(
                    LATEST_CHECKPOINT_PATH,
                    map_location="cpu",
                    weights_only=False,
                )
            )


            if checkpoint_is_compatible_B(
                latest_checkpoint
            ):

                print(
                    "♻️ Compatible Architecture-B "
                    "checkpoint found."
                )


                MODEL.load_state_dict(
                    latest_checkpoint[
                        "model_state_dict"
                    ]
                )


                if (
                    "optimizer_state_dict"
                    in latest_checkpoint
                ):

                    optimizer.load_state_dict(
                        latest_checkpoint[
                            "optimizer_state_dict"
                        ]
                    )


                if (
                    "scheduler_state_dict"
                    in latest_checkpoint
                ):

                    scheduler.load_state_dict(
                        latest_checkpoint[
                            "scheduler_state_dict"
                        ]
                    )


                if (
                    "scaler_state_dict"
                    in latest_checkpoint
                ):

                    try:

                        scaler.load_state_dict(
                            latest_checkpoint[
                                "scaler_state_dict"
                            ]
                        )

                    except Exception as exc:

                        print(
                            "⚠️ GradScaler state "
                            "could not be restored:"
                        )

                        print(
                            f"   {exc}"
                        )


                restore_sampler_state_B(
                    TRAIN_SAMPLER,
                    latest_checkpoint.get(
                        "train_sampler_state"
                    ),
                )


                restore_sampler_state_B(
                    VAL_SAMPLER,
                    latest_checkpoint.get(
                        "val_sampler_state"
                    ),
                )


                restore_rng_state_B(
                    latest_checkpoint.get(
                        "rng_state"
                    )
                )


                step = int(
                    latest_checkpoint.get(
                        "step",
                        0,
                    )
                )


                tokens_seen = int(
                    latest_checkpoint.get(
                        "tokens_seen",
                        0,
                    )
                )


                best_validation_step = int(
                    latest_checkpoint.get(
                        "best_validation_step",
                        0,
                    )
                )


                best_validation_loss = float(
                    latest_checkpoint.get(
                        "best_validation_loss",
                        float("inf"),
                    )
                )


                best_validation_perplexity = float(
                    latest_checkpoint.get(
                        "best_validation_perplexity",
                        perplexity_from_loss_B(
                            best_validation_loss
                        ),
                    )
                )


                best_checkpoint_count = int(
                    latest_checkpoint.get(
                        "best_checkpoint_count",
                        0,
                    )
                )


                bad_validation_count = int(
                    latest_checkpoint.get(
                        "bad_validation_count",
                        0,
                    )
                )


                numerical_overflow_count = int(
                    latest_checkpoint.get(
                        "numerical_overflow_count",
                        0,
                    )
                )


                saved_history = (
                    latest_checkpoint.get(
                        "training_history",
                        [],
                    )
                )


                if isinstance(
                    saved_history,
                    list,
                ):

                    training_history = (
                        list(
                            saved_history
                        )
                    )


                # If the checkpoint does not contain the history,
                # recover it from the metrics JSON.
                if not training_history:

                    if (
                        TRAINING_METRICS_PATH.is_file()
                    ):

                        try:

                            existing_metrics = (
                                json.load(
                                    open(
                                        TRAINING_METRICS_PATH,
                                        "r",
                                        encoding="utf-8",
                                    )
                                )
                            )


                            if isinstance(
                                existing_metrics,
                                list,
                            ):

                                training_history = (
                                    existing_metrics
                                )

                        except Exception:

                            pass


                resumed = True


                print()
                print(
                    f"✅ Resuming from step "
                    f"{step:,}."
                )

                print(
                    f"   Tokens seen       : "
                    f"{tokens_seen:,}"
                )

                print(
                    f"   Best val loss     : "
                    f"{best_validation_loss:.6f}"
                )

                print(
                    f"   Best checkpoints  : "
                    f"{best_checkpoint_count}"
                )


            else:

                print(
                    "ℹ️ Existing latest checkpoint "
                    "is incompatible with the current "
                    "production configuration."
                )

                print(
                    "   Starting from scratch."
                )


        except Exception as exc:

            print(
                "⚠️ Existing latest checkpoint "
                "could not be restored:"
            )

            print(
                f"   {exc}"
            )

            print(
                "   Starting from scratch."
            )


    if not resumed:

        print(
            "🆕 Starting Architecture-B "
            "training from scratch."
        )


    # ---------------------------------------------------------------------
    # Device / train mode.
    # ---------------------------------------------------------------------

    MODEL = MODEL.to(
        DEVICE
    )

    MODEL.train()


    # ---------------------------------------------------------------------
    # Parameters for gradient inspection.
    # ---------------------------------------------------------------------

    trainable_parameters = [
        parameter
        for parameter in MODEL.parameters()
        if parameter.requires_grad
    ]


    # ---------------------------------------------------------------------
    # Running loss accumulators.
    # ---------------------------------------------------------------------

    running_loss_sum = 0.0

    running_loss_tokens = 0


    optimizer.zero_grad(
        set_to_none=True
    )


    # ---------------------------------------------------------------------
    # Stop state.
    # ---------------------------------------------------------------------

    status = "completed"

    stop_reason = "max_steps"


    # =====================================================================
    # MAIN LOOP
    # =====================================================================

    try:

        while (
            step < MAX_STEPS
            and tokens_seen
            < TRAIN_TOKEN_BUDGET
        ):

            # -------------------------------------------------------------
            # One optimizer step.
            # -------------------------------------------------------------

            optimizer.zero_grad(
                set_to_none=True
            )


            nonfinite_this_step = False


            # -------------------------------------------------------------
            # Gradient accumulation.
            # -------------------------------------------------------------

            for micro_step in range(
                GRADIENT_ACCUMULATION_STEPS
            ):

                batch = (
                    sampler_next_batch_B(
                        TRAIN_SAMPLER,
                        MICRO_BATCH_SIZE,
                    )
                )


                (
                    encoder_ids,
                    decoder_ids,
                    labels,
                ) = move_batch_to_device_B(
                    batch
                )


                # Source + target tokens consumed.
                batch_tokens = int(
                    encoder_ids.numel()
                    + labels.numel()
                )


                tokens_seen += (
                    batch_tokens
                )


                with autocast_context_B():

                    logits = model_forward_B(
                        encoder_ids,
                        decoder_ids,
                    )


                    loss = architecture_B_loss(
                        logits,
                        labels,
                    )


                loss_value = float(
                    loss.detach()
                    .float()
                    .item()
                )


                if math.isfinite(
                    loss_value
                ):

                    running_loss_sum += (
                        loss_value
                        * labels.numel()
                    )

                    running_loss_tokens += (
                        labels.numel()
                    )


                else:

                    nonfinite_this_step = True


                scaled_loss = (
                    loss
                    / GRADIENT_ACCUMULATION_STEPS
                )


                if AMP_ENABLED:

                    scaler.scale(
                        scaled_loss
                    ).backward()

                else:

                    scaled_loss.backward()


            # -------------------------------------------------------------
            # Unscale BEFORE gradient checks/clipping.
            # -------------------------------------------------------------

            if AMP_ENABLED:

                scaler.unscale_(
                    optimizer
                )


            # -------------------------------------------------------------
            # Check individual gradients.
            # -------------------------------------------------------------

            if not nonfinite_this_step:

                for parameter in (
                    trainable_parameters
                ):

                    gradient = (
                        parameter.grad
                    )


                    if gradient is None:

                        continue


                    if not torch.isfinite(
                        gradient
                    ).all():

                        nonfinite_this_step = True

                        break


            # -------------------------------------------------------------
            # RECOVERY PATH
            # -------------------------------------------------------------
            #
            # IMPORTANT:
            # A numerical overflow does NOT terminate the complete
            # production run.
            #
            # The accumulated gradient is discarded.
            # The optimizer step is skipped.
            # GradScaler reduces its scale.
            # The next batch is then retried.
            # -------------------------------------------------------------

            if nonfinite_this_step:

                optimizer.zero_grad(
                    set_to_none=True
                )


                numerical_overflow_count += 1

                consecutive_numerical_overflows += 1


                if AMP_ENABLED:

                    scaler.update()


                print(
                    f"⚠️ Numerical overflow near "
                    f"optimizer step {step + 1:,}; "
                    f"update skipped "
                    f"({consecutive_numerical_overflows} "
                    f"consecutive)."
                )


                if (
                    consecutive_numerical_overflows
                    >= 20
                ):

                    raise FloatingPointError(
                        "20 consecutive numerical "
                        "overflows occurred."
                    )


                # Retry the same optimizer step number.
                continue


            # -------------------------------------------------------------
            # FP32 global gradient norm.
            # -------------------------------------------------------------

            squared_norm = torch.zeros(
                (),
                dtype=torch.float32,
                device=DEVICE,
            )


            for parameter in (
                trainable_parameters
            ):

                gradient = (
                    parameter.grad
                )


                if gradient is None:

                    continue


                gradient_fp32 = (
                    gradient.detach()
                    .float()
                )


                squared_norm += (
                    gradient_fp32
                    .pow(2)
                    .sum()
                )


            gradient_norm = torch.sqrt(
                squared_norm
            )


            # -------------------------------------------------------------
            # Gradient norm recovery.
            # -------------------------------------------------------------

            if not torch.isfinite(
                gradient_norm
            ):

                optimizer.zero_grad(
                    set_to_none=True
                )


                numerical_overflow_count += 1

                consecutive_numerical_overflows += 1


                if AMP_ENABLED:

                    scaler.update()


                print(
                    f"⚠️ Non-finite FP32 gradient "
                    f"norm near step {step + 1:,}; "
                    f"update skipped."
                )


                if (
                    consecutive_numerical_overflows
                    >= 20
                ):

                    raise FloatingPointError(
                        "20 consecutive non-finite "
                        "gradient norms occurred."
                    )


                continue


            # -------------------------------------------------------------
            # Safe gradient clipping.
            # -------------------------------------------------------------

            gradient_norm_value = float(
                gradient_norm.item()
            )


            if (
                GRADIENT_CLIP_NORM > 0.0
                and gradient_norm_value
                > GRADIENT_CLIP_NORM
            ):

                clip_scale = (
                    GRADIENT_CLIP_NORM
                    / (
                        gradient_norm_value
                        + 1e-6
                    )
                )


                for parameter in (
                    trainable_parameters
                ):

                    if parameter.grad is not None:

                        parameter.grad.mul_(
                            clip_scale
                        )


            # -------------------------------------------------------------
            # Successful optimizer step.
            # -------------------------------------------------------------

            consecutive_numerical_overflows = 0


            if AMP_ENABLED:

                scaler.step(
                    optimizer
                )

                scaler.update()

            else:

                optimizer.step()


            # Scheduler moves only after a successful update.
            scheduler.step()


            step += 1


            # =============================================================
            # EVALUATION
            # =============================================================

            should_evaluate = (

                step
                % EVAL_INTERVAL_STEPS
                == 0

                or step >= MAX_STEPS

                or tokens_seen
                >= TRAIN_TOKEN_BUDGET
            )


            if not should_evaluate:

                continue


            # -------------------------------------------------------------
            # Training loss for current interval.
            # -------------------------------------------------------------

            if running_loss_tokens > 0:

                train_loss = (
                    running_loss_sum
                    / running_loss_tokens
                )

            else:

                train_loss = (
                    float("nan")
                )


            # -------------------------------------------------------------
            # Validation.
            # -------------------------------------------------------------

            validation = (
                evaluate_architecture_B(
                    EVAL_BATCHES
                )
            )


            validation_loss = float(
                validation[
                    "loss"
                ]
            )


            validation_perplexity = float(
                validation[
                    "perplexity"
                ]
            )


            elapsed = (
                time.perf_counter()
                - start_time
            )


            # -------------------------------------------------------------
            # Best checkpoint logic.
            # -------------------------------------------------------------

            first_finite_validation = (
                best_checkpoint_count
                == 0
                and math.isfinite(
                    validation_loss
                )
            )


            meaningful_improvement = (

                math.isfinite(
                    validation_loss
                )

                and
                validation_loss
                < (
                    best_validation_loss
                    - MIN_DELTA
                )
            )


            is_best = (
                first_finite_validation
                or meaningful_improvement
            )


            if is_best:

                best_validation_loss = (
                    validation_loss
                )

                best_validation_perplexity = (
                    validation_perplexity
                )

                best_validation_step = (
                    step
                )

                best_checkpoint_count += 1

                bad_validation_count = 0


                save_best_checkpoint_B(

                    step=step,

                    tokens_seen=tokens_seen,

                    best_validation_step=(
                        best_validation_step
                    ),

                    best_validation_loss=(
                        best_validation_loss
                    ),

                    best_validation_perplexity=(
                        best_validation_perplexity
                    ),

                    best_checkpoint_count=(
                        best_checkpoint_count
                    ),

                    bad_validation_count=(
                        bad_validation_count
                    ),
                )


            else:

                if math.isfinite(
                    validation_loss
                ):

                    bad_validation_count += 1


            # -------------------------------------------------------------
            # Record metrics.
            # -------------------------------------------------------------

            history_entry = {

                "step":
                    int(step),

                "train_loss":
                    float(train_loss),

                "val_loss":
                    float(validation_loss),

                "val_perplexity":
                    float(
                        validation_perplexity
                    ),

                "learning_rate":
                    optimizer_learning_rate_B(
                        optimizer
                    ),

                "tokens_seen":
                    int(tokens_seen),

                "best":
                    bool(is_best),

                "best_checkpoint_count":
                    int(
                        best_checkpoint_count
                    ),

                "bad_validation_count":
                    int(
                        bad_validation_count
                    ),

                "numerical_overflow_count":
                    int(
                        numerical_overflow_count
                    ),

                "elapsed_seconds":
                    float(elapsed),
            }


            training_history.append(
                history_entry
            )


            # -------------------------------------------------------------
            # Save complete resumable state.
            # -------------------------------------------------------------

            save_latest_checkpoint_B(

                step=step,

                tokens_seen=tokens_seen,

                best_validation_step=(
                    best_validation_step
                ),

                best_validation_loss=(
                    best_validation_loss
                ),

                best_validation_perplexity=(
                    best_validation_perplexity
                ),

                best_checkpoint_count=(
                    best_checkpoint_count
                ),

                bad_validation_count=(
                    bad_validation_count
                ),

                optimizer=optimizer,

                scheduler=scheduler,

                scaler=scaler,

                training_history=(
                    training_history
                ),

                numerical_overflow_count=(
                    numerical_overflow_count
                ),
            )


            # -------------------------------------------------------------
            # Save metrics.
            # -------------------------------------------------------------

            with open(
                TRAINING_METRICS_PATH,
                "w",
                encoding="utf-8",
            ) as metrics_file:

                json.dump(
                    training_history,
                    metrics_file,
                    ensure_ascii=False,
                    indent=2,
                )


            # -------------------------------------------------------------
            # Console output.
            # -------------------------------------------------------------

            if is_best:

                marker = (
                    f"BEST #{best_checkpoint_count}"
                )

            else:

                marker = (
                    f"bad {bad_validation_count}"
                )


            print(
                f"Step {step:6d} | "
                f"train_loss {train_loss:.6f} | "
                f"val_loss {validation_loss:.6f} | "
                f"val_ppl {validation_perplexity:.3f} | "
                f"{marker} | "
                f"lr {optimizer_learning_rate_B(optimizer):.3e}"
            )


            # Reset interval training statistics.
            running_loss_sum = 0.0
            running_loss_tokens = 0


            # =============================================================
            # MAX-BEST EARLY STOP
            # =============================================================

            if (
                best_checkpoint_count
                >= MAX_BEST_CHECKPOINTS
            ):

                status = (
                    "early_stopped"
                )

                stop_reason = (
                    "max_best_checkpoints"
                )

                break


            # =============================================================
            # PATIENCE EARLY STOP
            # =============================================================

            if (

                EARLY_STOPPING_ENABLED

                and
                best_checkpoint_count
                >= MIN_BEST_CHECKPOINTS

                and
                step
                >= MIN_STEPS_BEFORE_STOP

                and
                bad_validation_count
                >= PATIENCE
            ):

                status = (
                    "early_stopped"
                )

                stop_reason = (
                    "validation_patience"
                )

                break


        # =================================================================
        # NORMAL STOP REASONS
        # =================================================================

        if (
            status == "completed"
            and step >= MAX_STEPS
        ):

            stop_reason = (
                "max_steps"
            )


        if (
            status == "completed"
            and tokens_seen
            >= TRAIN_TOKEN_BUDGET
        ):

            stop_reason = (
                "train_token_budget"
            )


        # =================================================================
        # FINAL LATEST CHECKPOINT
        # =================================================================

        save_latest_checkpoint_B(

            step=step,

            tokens_seen=tokens_seen,

            best_validation_step=(
                best_validation_step
            ),

            best_validation_loss=(
                best_validation_loss
            ),

            best_validation_perplexity=(
                best_validation_perplexity
            ),

            best_checkpoint_count=(
                best_checkpoint_count
            ),

            bad_validation_count=(
                bad_validation_count
            ),

            optimizer=optimizer,

            scheduler=scheduler,

            scaler=scaler,

            training_history=(
                training_history
            ),

            numerical_overflow_count=(
                numerical_overflow_count
            ),
        )


        # =================================================================
        # FINAL METRICS SAVE
        # =================================================================

        with open(
            TRAINING_METRICS_PATH,
            "w",
            encoding="utf-8",
        ) as metrics_file:

            json.dump(
                training_history,
                metrics_file,
                ensure_ascii=False,
                indent=2,
            )


    except Exception:

        # =================================================================
        # EMERGENCY RECOVERABLE CHECKPOINT
        # =================================================================

        try:

            save_latest_checkpoint_B(

                step=step,

                tokens_seen=tokens_seen,

                best_validation_step=(
                    best_validation_step
                ),

                best_validation_loss=(
                    best_validation_loss
                ),

                best_validation_perplexity=(
                    best_validation_perplexity
                ),

                best_checkpoint_count=(
                    best_checkpoint_count
                ),

                bad_validation_count=(
                    bad_validation_count
                ),

                optimizer=optimizer,

                scheduler=scheduler,

                scaler=scaler,

                training_history=(
                    training_history
                ),

                numerical_overflow_count=(
                    numerical_overflow_count
                ),
            )


            print()
            print(
                "✅ Recoverable latest checkpoint "
                "saved after training error."
            )


        except Exception as checkpoint_error:

            print()
            print(
                "⚠️ Emergency checkpoint could "
                "not be saved:"
            )

            print(
                f"   {checkpoint_error}"
            )


        raise


    # =====================================================================
    # RETURN TRAINING STATE
    # =====================================================================

    elapsed_training_seconds = (
        time.perf_counter()
        - start_time
    )


    return {

        "status":
            status,

        "stop_reason":
            stop_reason,

        "step":
            int(step),

        "tokens_seen":
            int(tokens_seen),

        "best_validation_step":
            int(
                best_validation_step
            ),

        "best_validation_loss":
            float(
                best_validation_loss
            ),

        "best_validation_perplexity":
            float(
                best_validation_perplexity
            ),

        "best_checkpoint_count":
            int(
                best_checkpoint_count
            ),

        "bad_validation_count":
            int(
                bad_validation_count
            ),

        "elapsed_training_seconds":
            float(
                elapsed_training_seconds
            ),

        "numerical_overflow_count":
            int(
                numerical_overflow_count
            ),
    }


# =========================================================================
# ENGINE READY
# =========================================================================

parameter_count_B = sum(
    parameter.numel()
    for parameter in MODEL.parameters()
)


print(
    "✅ Architecture-B training engine ready."
)

print()

print(
    f"Device                : "
    f"{DEVICE}"
)

print(
    f"AMP enabled           : "
    f"{AMP_ENABLED}"
)

print(
    f"Model parameters      : "
    f"{parameter_count_B:,}"
    f" ({parameter_count_B / 1_000_000:.3f}M)"
)

print(
    f"Train sampler         : "
    f"{TRAIN_SAMPLER_NAME}"
)

print(
    f"Validation sampler    : "
    f"{VAL_SAMPLER_NAME}"
)

print(
    f"Maximum steps         : "
    f"{MAX_STEPS:,}"
)

print(
    f"Train token budget    : "
    f"{TRAIN_TOKEN_BUDGET:,}"
)

print(
    f"Validation batches    : "
    f"{EVAL_BATCHES}"
)

print(
    f"Warmup steps          : "
    f"{WARMUP_STEPS:,}"
)

print(
    f"Gradient accumulation : "
    f"{GRADIENT_ACCUMULATION_STEPS}"
)

print(
    f"Gradient clip norm    : "
    f"{GRADIENT_CLIP_NORM}"
)

print()
print(
    "Numerical overflow recovery: ENABLED"
)

✅ Architecture-B training engine ready.

Device                : cuda
AMP enabled           : True
Model parameters      : 139,263,824 (139.264M)
Train sampler         : TRAIN_SAMPLER
Validation sampler    : VAL_SAMPLER
Maximum steps         : 6,104
Train token budget    : 100,000,000
Validation batches    : 100
Warmup steps          : 305
Gradient accumulation : 8
Gradient clip norm    : 1.0

Numerical overflow recovery: ENABLED


In [10]:
# Cell 08.5 — Architecture-B resume compatibility patch
# DECISION: ADD

import torch

print("🔧 Applying Architecture-B resume compatibility patch...")

# ============================================================================
# PATCH 1: Make old checkpoint loader see training_state fields at root level
# ============================================================================

_original_torch_load = torch.load

def _patched_torch_load(*args, **kwargs):
    checkpoint = _original_torch_load(*args, **kwargs)

    if isinstance(checkpoint, dict):
        training_state = checkpoint.get("training_state")

        if isinstance(training_state, dict):

            field_mapping = {
                "step": "step",
                "tokens_seen": "tokens_seen",
                "best_validation_loss": "best_validation_loss",
                "best_validation_step": "best_validation_step",
                "best_checkpoint_count": "best_checkpoint_count",
                "bad_validation_count": "bad_validation_count",
            }

            for root_key, state_key in field_mapping.items():
                if root_key not in checkpoint and state_key in training_state:
                    checkpoint[root_key] = training_state[state_key]

    return checkpoint


torch.load = _patched_torch_load


# ============================================================================
# PATCH 2: Fix EncoderDecoderTokenSampler call
# ============================================================================

if "device" in globals():
    SAMPLER_DEVICE = device
elif "DEVICE" in globals():
    SAMPLER_DEVICE = DEVICE
else:
    SAMPLER_DEVICE = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )


def sampler_next_batch_B(sampler, batch_size):
    """
    Correct Architecture-B sampler interface.

    EncoderDecoderTokenSampler.sample_batch()
    requires:
        sample_batch(batch_size, device)
    """

    if not hasattr(sampler, "sample_batch"):
        raise RuntimeError(
            f"{type(sampler).__name__} does not provide sample_batch()."
        )

    return sampler.sample_batch(
        batch_size=batch_size,
        device=SAMPLER_DEVICE,
    )


print("✅ Checkpoint resume patch installed.")
print("✅ Sampler patch installed.")
print(f"   Sampler device: {SAMPLER_DEVICE}")

print()
print("=" * 72)
print("✅ CELL 08.5 READY")
print("=" * 72)

🔧 Applying Architecture-B resume compatibility patch...
✅ Checkpoint resume patch installed.
✅ Sampler patch installed.
   Sampler device: cuda

✅ CELL 08.5 READY


In [11]:
# Cell 09 — Run Architecture-B training
# DECISION: ADD

print("🚀 Starting Architecture-B training...")
print()

print(
    f"Mode                  : {RUN_MODE}"
)

print(
    f"Maximum optimizer steps: "
    f"{TRAIN_CONFIG['max_steps']:,}"
)

print(
    f"Maximum token budget   : "
    f"{TRAIN_CONFIG['train_token_budget']:,}"
)

print(
    f"Early stopping         : "
    f"{TRAIN_CONFIG['early_stopping']}"
)

print(
    f"Best checkpoint range  : "
    f"{TRAIN_CONFIG['min_best_checkpoints']}"
    f" → "
    f"{TRAIN_CONFIG['max_best_checkpoints']}"
)

print(
    f"Patience               : "
    f"{TRAIN_CONFIG['patience']}"
)

print(
    f"Minimum improvement    : "
    f"{TRAIN_CONFIG['min_delta']}"
)

print()

TRAINING_STATE_B = (
    run_architecture_B_training()
)


# =========================================================================
# FINAL TRAINING SUMMARY
# =========================================================================

print()
print("=" * 72)
print("🏁 Architecture-B training finished.")
print("=" * 72)
print()

print(
    f"Status                 : "
    f"{TRAINING_STATE_B['status']}"
)

print(
    f"Stop reason            : "
    f"{TRAINING_STATE_B['stop_reason']}"
)

print(
    f"Optimizer steps        : "
    f"{TRAINING_STATE_B['step']:,}"
)

print(
    f"Tokens seen            : "
    f"{TRAINING_STATE_B['tokens_seen']:,}"
)

print(
    f"Best validation step   : "
    f"{TRAINING_STATE_B['best_validation_step']}"
)

print(
    f"Best validation loss   : "
    f"{TRAINING_STATE_B['best_validation_loss']:.6f}"
)

print(
    f"Best validation PPL    : "
    f"{TRAINING_STATE_B['best_validation_perplexity']:.3f}"
)

print(
    f"Best checkpoints saved : "
    f"{TRAINING_STATE_B['best_checkpoint_count']}"
)

print(
    f"Bad evaluations       : "
    f"{TRAINING_STATE_B['bad_validation_count']}"
)

print(
    f"Elapsed time (sec)     : "
    f"{TRAINING_STATE_B['elapsed_training_seconds']:.2f}"
)

print()
print(
    f"Best checkpoint        : "
    f"{BEST_CHECKPOINT_PATH}"
)

print(
    f"Latest checkpoint      : "
    f"{LATEST_CHECKPOINT_PATH}"
)

print(
    f"Training metrics       : "
    f"{TRAINING_METRICS_PATH}"
)

🚀 Starting Architecture-B training...

Mode                  : production
Maximum optimizer steps: 6,104
Maximum token budget   : 100,000,000
Early stopping         : True
Best checkpoint range  : 8 → 12
Patience               : 4
Minimum improvement    : 0.005

♻️ Compatible Architecture-B checkpoint found.

✅ Resuming from step 6,100.
   Tokens seen       : 100,007,936
   Best val loss     : 3.460149
   Best checkpoints  : 11

🏁 Architecture-B training finished.

Status                 : completed
Stop reason            : train_token_budget
Optimizer steps        : 6,100
Tokens seen            : 100,007,936
Best validation step   : 6000
Best validation loss   : 3.460149
Best validation PPL    : 31.822
Best checkpoints saved : 11
Bad evaluations       : 1
Elapsed time (sec)     : 5.44

Best checkpoint        : /kaggle/working/architecture_B/checkpoints/best.pt
Latest checkpoint      : /kaggle/working/architecture_B/checkpoints/latest.pt
Training metrics       : /kaggle/working/archite

In [12]:
# Cell 10 — Select best Architecture-B checkpoint and export final model
# DECISION: UPDATE

import json
import math
from pathlib import Path

import torch

print("🏆 Selecting best Architecture-B checkpoint...")
print()


# =========================================================================
# OUTPUT PATHS
# =========================================================================

FINAL_MODEL_PATH = (
    ARCHITECTURE_B_ROOT
    / "model_B_final.pt"
)

STATE_DICT_PATH = (
    ARCHITECTURE_B_ROOT
    / "model_B_state_dict.pt"
)

MODEL_METADATA_PATH = (
    ARCHITECTURE_B_ROOT
    / "model_B_metadata.json"
)

SELECTED_CHECKPOINT_PATH = (
    METRICS_DIR
    / "selected_checkpoint_B.json"
)


# =========================================================================
# VERIFY BEST CHECKPOINT
# =========================================================================

assert BEST_CHECKPOINT_PATH.exists(), (
    f"Best checkpoint not found:\n"
    f"{BEST_CHECKPOINT_PATH}\n"
    "Run Cell 09 successfully first."
)


# =========================================================================
# LOAD BEST CHECKPOINT
# =========================================================================

print("📦 Loading best checkpoint...")

best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

assert isinstance(
    best_checkpoint,
    dict,
), "Invalid best checkpoint format."


# =========================================================================
# CHECK CHECKPOINT IDENTITY
# =========================================================================

assert checkpoint_is_compatible(
    best_checkpoint
), (
    "Best checkpoint is incompatible with "
    "the current Architecture-B configuration."
)

print("✅ Best checkpoint is compatible.")


# =========================================================================
# RESTORE BEST MODEL
# =========================================================================

assert (
    "model_state_dict" in best_checkpoint
), "Best checkpoint does not contain model_state_dict."

MODEL.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)

MODEL = MODEL.to(
    DEVICE
)

MODEL.eval()


# =========================================================================
# EXTRACT BEST TRAINING STATE
# =========================================================================

best_training_state = best_checkpoint.get(
    "training_state",
    {}
)

if not isinstance(
    best_training_state,
    dict,
):
    best_training_state = {}


# =========================================================================
# ROBUST BEST-METRIC EXTRACTION
#
# The historical Architecture-B checkpoint format does not necessarily
# store every derived metric inside training_state. Recover what is
# available and derive the rest safely.
# =========================================================================

# -------------------------------------------------------------------------
# Best validation step
# -------------------------------------------------------------------------

best_validation_step = (
    best_training_state.get(
        "best_validation_step"
    )
)

if best_validation_step is None:

    best_validation_step = (
        best_training_state.get(
            "step"
        )
    )

if best_validation_step is None:

    best_validation_step = (
        best_checkpoint.get(
            "best_validation_step"
        )
    )

if best_validation_step is None:

    best_validation_step = (
        best_checkpoint.get(
            "step"
        )
    )


# -------------------------------------------------------------------------
# Best validation loss
# -------------------------------------------------------------------------

best_validation_loss = (
    best_training_state.get(
        "best_validation_loss"
    )
)

if best_validation_loss is None:

    best_validation_loss = (
        best_checkpoint.get(
            "best_validation_loss"
        )
    )

# Some checkpoint versions may store validation loss under another key.
if best_validation_loss is None:

    best_validation_loss = (
        best_training_state.get(
            "validation_loss"
        )
    )


# -------------------------------------------------------------------------
# Best validation perplexity
# -------------------------------------------------------------------------

best_validation_perplexity = (
    best_training_state.get(
        "best_validation_perplexity"
    )
)

if best_validation_perplexity is None:

    best_validation_perplexity = (
        best_checkpoint.get(
            "best_validation_perplexity"
        )
    )

# Derive PPL from validation loss when it was not stored.
if (
    best_validation_perplexity is None
    and best_validation_loss is not None
):

    best_validation_perplexity = math.exp(
        min(
            float(
                best_validation_loss
            ),
            20.0,
        )
    )


# -------------------------------------------------------------------------
# Tokens seen at best
# -------------------------------------------------------------------------

tokens_seen_at_best = (
    best_training_state.get(
        "tokens_seen"
    )
)

if tokens_seen_at_best is None:

    tokens_seen_at_best = (
        best_training_state.get(
            "tokens_seen_at_best"
        )
    )

if tokens_seen_at_best is None:

    tokens_seen_at_best = (
        best_checkpoint.get(
            "tokens_seen"
        )
    )

# -------------------------------------------------------------------------
# Last-resort deterministic derivation from the saved step.
#
# Production Architecture-B:
#   source context + target context = 1024 + 1024
#   micro batch = 1
#   gradient accumulation = 8
#
# We prefer the checkpoint's stored value, so this is only a fallback.
# -------------------------------------------------------------------------
if (
    tokens_seen_at_best is None
    and best_validation_step is not None
):

    try:

        _train_cfg = dict(
            best_checkpoint.get(
                "train_config",
                {}
            )
        )

        _micro_batch = int(
            _train_cfg.get(
                "micro_batch_size",
                MICRO_BATCH_SIZE,
            )
        )

        _grad_accum = int(
            _train_cfg.get(
                "gradient_accumulation_steps",
                GRAD_ACCUM_STEPS,
            )
        )

        _src_context = int(
            _train_cfg.get(
                "src_context_length",
                MODEL_CONFIG[
                    "src_context_length"
                ],
            )
        )

        _tgt_context = int(
            _train_cfg.get(
                "tgt_context_length",
                MODEL_CONFIG[
                    "tgt_context_length"
                ],
            )
        )

        _tokens_per_optimizer_step = (
            _micro_batch
            * _grad_accum
            * (
                _src_context
                + _tgt_context
            )
        )

        tokens_seen_at_best = (
            int(
                best_validation_step
            )
            * _tokens_per_optimizer_step
        )

        print(
            "ℹ️ tokens_seen_at_best was not "
            "stored explicitly; derived from "
            "checkpoint configuration."
        )

    except Exception:

        tokens_seen_at_best = None


# -------------------------------------------------------------------------
# Best checkpoint count
# -------------------------------------------------------------------------

best_checkpoint_count = (
    best_training_state.get(
        "best_checkpoint_count",
        0,
    )
)

if best_checkpoint_count is None:

    best_checkpoint_count = (
        best_checkpoint.get(
            "best_checkpoint_count",
            0,
        )
    )


# -------------------------------------------------------------------------
# Status / stopping reason
# -------------------------------------------------------------------------

training_status = (
    best_training_state.get(
        "status",
        "best",
    )
)

early_stopping_reason = (
    best_training_state.get(
        "stop_reason",
        None,
    )
)


# =========================================================================
# REQUIRED VALUE CHECKS
# =========================================================================

assert best_validation_step is not None, (
    "Could not determine best validation step."
)

assert best_validation_loss is not None, (
    "Could not determine best validation loss."
)

assert best_validation_perplexity is not None, (
    "Could not determine best validation perplexity."
)

assert tokens_seen_at_best is not None, (
    "Could not determine tokens seen at best checkpoint."
)

assert math.isfinite(
    float(best_validation_loss)
)

assert math.isfinite(
    float(best_validation_perplexity)
)


# =========================================================================
# PRINT SELECTED METRICS
# =========================================================================

print()
print("📊 Selected best checkpoint:")
print(
    f"   Step          : "
    f"{int(best_validation_step):,}"
)

print(
    f"   Val loss      : "
    f"{float(best_validation_loss):.6f}"
)

print(
    f"   Val PPL       : "
    f"{float(best_validation_perplexity):.4f}"
)

print(
    f"   Tokens seen   : "
    f"{int(tokens_seen_at_best):,}"
)

print(
    f"   Best count    : "
    f"{int(best_checkpoint_count)}"
)

print(
    f"   Status        : "
    f"{training_status}"
)


# =========================================================================
# PARAMETER COUNT
# =========================================================================

parameter_count = sum(
    parameter.numel()
    for parameter in MODEL.parameters()
)

trainable_parameter_count = sum(
    parameter.numel()
    for parameter in MODEL.parameters()
    if parameter.requires_grad
)


# =========================================================================
# PRODUCTION PARAMETER VALIDATION
# =========================================================================

if RUN_MODE == "production":

    assert (
        50_000_000
        <= parameter_count
        <= 300_000_000
    ), (
        "Architecture-B production model is "
        "outside the required 50M–300M range.\n"
        f"Parameters: {parameter_count:,}"
    )


# =========================================================================
# SAVE FULL FINAL MODEL PACKAGE
# =========================================================================

final_model_package = {

    "experiment_id":
        EXPERIMENT_ID,

    "architecture_name":
        ARCHITECTURE_NAME,

    "architecture_version":
        ARCHITECTURE_VERSION,

    "run_mode":
        RUN_MODE,

    "config_fingerprint":
        CONFIG_FINGERPRINT,

    "model_type":
        "encoder-decoder-transformer",

    "model_config":
        dict(MODEL_CONFIG),

    "model_state_dict":
        MODEL.state_dict(),

    "parameter_count":
        parameter_count,

    "parameter_millions":
        parameter_count / 1_000_000,

    "selected_checkpoint":
        str(BEST_CHECKPOINT_PATH),

    "best_validation_step":
        int(best_validation_step),

    "best_validation_loss":
        float(best_validation_loss),

    "best_validation_perplexity":
        float(best_validation_perplexity),

    "tokens_seen_at_best":
        int(tokens_seen_at_best),

    "best_checkpoint_count":
        int(best_checkpoint_count),

    "early_stopping":
        {
            "enabled":
                bool(
                    TRAIN_CONFIG[
                        "early_stopping"
                    ]
                ),

            "status":
                training_status,

            "reason":
                early_stopping_reason,

            "min_best_checkpoints":
                TRAIN_CONFIG[
                    "min_best_checkpoints"
                ],

            "max_best_checkpoints":
                TRAIN_CONFIG[
                    "max_best_checkpoints"
                ],

            "patience":
                TRAIN_CONFIG[
                    "patience"
                ],

            "min_delta":
                TRAIN_CONFIG[
                    "min_delta"
                ],

            "min_steps_before_plateau_stop":
                TRAIN_CONFIG[
                    "min_steps_before_plateau_stop"
                ],
        },

    "source_probabilities":
        dict(SOURCE_PROBABILITIES),

    "tokenizer_vocab_size":
        TOKENIZER_VOCAB_SIZE,
}


torch.save(
    final_model_package,
    FINAL_MODEL_PATH,
)


# =========================================================================
# SAVE STATE DICT
# =========================================================================

torch.save(
    MODEL.state_dict(),
    STATE_DICT_PATH,
)


# =========================================================================
# SAVE MODEL METADATA
# =========================================================================

model_metadata = {

    "project":
        "CS781 Phase 1",

    "experiment_id":
        EXPERIMENT_ID,

    "architecture_name":
        ARCHITECTURE_NAME,

    "architecture_version":
        ARCHITECTURE_VERSION,

    "run_mode":
        RUN_MODE,

    "model_type":
        "encoder-decoder-transformer",

    "parameter_count":
        parameter_count,

    "parameter_millions":
        parameter_count / 1_000_000,

    "model_config":
        dict(MODEL_CONFIG),

    "best_checkpoint":
        str(BEST_CHECKPOINT_PATH),

    "best_validation_step":
        int(best_validation_step),

    "best_validation_loss":
        float(best_validation_loss),

    "best_validation_perplexity":
        float(best_validation_perplexity),

    "tokens_seen_at_best":
        int(tokens_seen_at_best),

    "best_checkpoint_count":
        int(best_checkpoint_count),

    "early_stopping":
        final_model_package[
            "early_stopping"
        ],

    "source_probabilities":
        dict(SOURCE_PROBABILITIES),

    "tokenizer_vocab_size":
        TOKENIZER_VOCAB_SIZE,

    "config_fingerprint":
        CONFIG_FINGERPRINT,
}


with open(
    MODEL_METADATA_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        model_metadata,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# SAVE SELECTED CHECKPOINT RECORD
# =========================================================================

selected_checkpoint_record = {

    "project":
        "CS781 Phase 1",

    "experiment_id":
        EXPERIMENT_ID,

    "architecture_name":
        ARCHITECTURE_NAME,

    "architecture_version":
        ARCHITECTURE_VERSION,

    "run_mode":
        RUN_MODE,

    "config_fingerprint":
        CONFIG_FINGERPRINT,

    "selected_checkpoint":
        str(BEST_CHECKPOINT_PATH),

    "best_validation_step":
        int(best_validation_step),

    "best_validation_loss":
        float(best_validation_loss),

    "best_validation_perplexity":
        float(best_validation_perplexity),

    "tokens_seen_at_best":
        int(tokens_seen_at_best),

    "best_checkpoint_count":
        int(best_checkpoint_count),

    "training_status":
        training_status,

    "early_stopping_reason":
        early_stopping_reason,
}


with open(
    SELECTED_CHECKPOINT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        selected_checkpoint_record,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# VERIFY SAVED STATE
# =========================================================================

assert FINAL_MODEL_PATH.exists()
assert STATE_DICT_PATH.exists()
assert MODEL_METADATA_PATH.exists()
assert SELECTED_CHECKPOINT_PATH.exists()


# =========================================================================
# RELOAD STATE DICT INTO FRESH MODEL
# =========================================================================

reload_model = EncoderDecoderTransformer(
    MODEL_CONFIG
)

reload_model.load_state_dict(
    torch.load(
        STATE_DICT_PATH,
        map_location="cpu",
        weights_only=False,
    )
)

del reload_model


# =========================================================================
# SUMMARY
# =========================================================================

print()
print(
    "✅ Best Architecture-B model exported."
)

print()

print(
    f"Selected step        : "
    f"{int(best_validation_step):,}"
)

print(
    f"Validation loss      : "
    f"{float(best_validation_loss):.6f}"
)

print(
    f"Validation PPL       : "
    f"{float(best_validation_perplexity):.3f}"
)

print(
    f"Tokens seen at best  : "
    f"{int(tokens_seen_at_best):,}"
)

print(
    f"Best checkpoints     : "
    f"{int(best_checkpoint_count)}"
)

print(
    f"Training status      : "
    f"{training_status}"
)

print(
    f"Stopping reason      : "
    f"{early_stopping_reason}"
)

print(
    f"Parameters           : "
    f"{parameter_count:,} "
    f"({parameter_count / 1_000_000:.4f}M)"
)

print()

print(
    f"Final model          : "
    f"{FINAL_MODEL_PATH}"
)

print(
    f"State dict           : "
    f"{STATE_DICT_PATH}"
)

print(
    f"Metadata             : "
    f"{MODEL_METADATA_PATH}"
)

print(
    f"Selected checkpoint  : "
    f"{SELECTED_CHECKPOINT_PATH}"
)

🏆 Selecting best Architecture-B checkpoint...

📦 Loading best checkpoint...
✅ Best checkpoint is compatible.

📊 Selected best checkpoint:
   Step          : 6,000
   Val loss      : 3.460149
   Val PPL       : 31.8217
   Tokens seen   : 98,369,536
   Best count    : 0
   Status        : best

✅ Best Architecture-B model exported.

Selected step        : 6,000
Validation loss      : 3.460149
Validation PPL       : 31.822
Tokens seen at best  : 98,369,536
Best checkpoints     : 0
Training status      : best
Stopping reason      : None
Parameters           : 139,263,824 (139.2638M)

Final model          : /kaggle/working/architecture_B/model_B_final.pt
State dict           : /kaggle/working/architecture_B/model_B_state_dict.pt
Metadata             : /kaggle/working/architecture_B/model_B_metadata.json
Selected checkpoint  : /kaggle/working/architecture_B/metrics/selected_checkpoint_B.json


In [13]:
# Cell 11 — Architecture-B generation configuration
# DECISION: ADD

import json


print("🎛️ Configuring Architecture-B generation...")


# =========================================================================
# GENERATION CONFIGURATION
# =========================================================================
#
# FAST:
#   Used only to validate the complete generation/submission pipeline.
#
# PRODUCTION:
#   Matches the official Phase-1 generation requirement of 128 words.

if RUN_MODE == "fast":

    GENERATION_CONFIG = {

        "target_words": 8,

        # Larger token allowance because BPE tokens are not
        # equivalent to whitespace-separated words.
        "max_new_tokens": 256,

        "temperature": 1.0,

        "top_p": 1.0,

        "top_k": 100,

        "repetition_penalty": 1.20,

        "test_examples": 2,

        # The FAST encoder context is intentionally shorter
        # than the official 512-word prompt.
        "prompt_overflow_strategy": "truncate_left",
    }

else:

    GENERATION_CONFIG = {

        "target_words": 128,

        # 1536 provides ample BPE-token budget for 128 words.
        "max_new_tokens": 1536,

        "temperature": 0.70,

        "top_p": 0.95,

        "top_k": 50,

        "repetition_penalty": 1.10,

        "test_examples": 1000,

        # Production must preserve the full official prompt whenever
        # its tokenized form fits inside the encoder context.
        "prompt_overflow_strategy": "error",
    }


# =========================================================================
# SANITY CHECKS
# =========================================================================

assert (
    GENERATION_CONFIG["target_words"]
    > 0
)

assert (
    GENERATION_CONFIG["max_new_tokens"]
    >= GENERATION_CONFIG["target_words"]
)

assert (
    0.0
    < GENERATION_CONFIG["temperature"]
    <= 10.0
)

assert (
    0.0
    < GENERATION_CONFIG["top_p"]
    <= 1.0
)

assert (
    GENERATION_CONFIG["top_k"]
    >= 0
)

assert (
    GENERATION_CONFIG["repetition_penalty"]
    >= 1.0
)


# =========================================================================
# PRODUCTION REQUIREMENT CHECKS
# =========================================================================

if RUN_MODE == "production":

    assert (
        GENERATION_CONFIG["target_words"]
        == 128
    )

    assert (
        GENERATION_CONFIG["test_examples"]
        == 1000
    )

    assert (
        GENERATION_CONFIG[
            "prompt_overflow_strategy"
        ]
        == "error"
    )


# =========================================================================
# SAVE GENERATION MANIFEST
# =========================================================================

GENERATION_MANIFEST = {

    "project":
        "CS781 Phase 1",

    "experiment_id":
        EXPERIMENT_ID,

    "architecture_name":
        ARCHITECTURE_NAME,

    "architecture_version":
        ARCHITECTURE_VERSION,

    "run_mode":
        RUN_MODE,

    "generation":
        dict(
            GENERATION_CONFIG
        ),

    "official_requirement": {

        "prompt_words":
            512,

        "output_words":
            128,

        "prompt_excluded_from_output":
            True,

        "official_test_examples":
            1000,
    },
}


GENERATION_CONFIG_PATH = (
    CONFIGS_DIR
    / "architecture_B_generation_config.json"
)


with open(
    GENERATION_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        GENERATION_MANIFEST,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# SUMMARY
# =========================================================================

print()
print("✅ Architecture-B generation configured.")
print()

print(
    f"Run mode           : "
    f"{RUN_MODE}"
)

print(
    f"Target words       : "
    f"{GENERATION_CONFIG['target_words']}"
)

print(
    f"Maximum new tokens : "
    f"{GENERATION_CONFIG['max_new_tokens']}"
)

print(
    f"Temperature        : "
    f"{GENERATION_CONFIG['temperature']}"
)

print(
    f"Top-p              : "
    f"{GENERATION_CONFIG['top_p']}"
)

print(
    f"Top-k              : "
    f"{GENERATION_CONFIG['top_k']}"
)

print(
    f"Repetition penalty : "
    f"{GENERATION_CONFIG['repetition_penalty']}"
)

print(
    f"Test examples      : "
    f"{GENERATION_CONFIG['test_examples']}"
)

print(
    f"Overflow strategy  : "
    f"{GENERATION_CONFIG['prompt_overflow_strategy']}"
)

print(
    f"Config manifest    : "
    f"{GENERATION_CONFIG_PATH}"
)

🎛️ Configuring Architecture-B generation...

✅ Architecture-B generation configured.

Run mode           : production
Target words       : 128
Maximum new tokens : 1536
Temperature        : 0.7
Top-p              : 0.95
Top-k              : 50
Repetition penalty : 1.1
Test examples      : 1000
Overflow strategy  : error
Config manifest    : /kaggle/working/architecture_B/configs/architecture_B_generation_config.json


In [14]:
# Cell 12 — Robust Architecture-B encoder-decoder generation with rolling decoder context
# DECISION: UPDATE

import math
from contextlib import nullcontext

import torch
import torch.nn.functional as F

print("✍️ Building Architecture-B generation function...")


# =========================================================================
# LOCAL AUTOCAST FALLBACK
# =========================================================================

def _architecture_B_autocast():

    helper = globals().get(
        "autocast_context"
    )

    if callable(helper):

        return helper()

    if (
        USE_AMP
        and DEVICE.type == "cuda"
    ):

        return torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        )

    return nullcontext()


# =========================================================================
# SPECIAL TOKEN IDS
# =========================================================================

_SPECIAL_IDS_B = set()

for token_id in [
    BOS_TOKEN_ID,
    EOS_TOKEN_ID,
    PAD_TOKEN_ID,
    UNK_TOKEN_ID,
]:

    if token_id is not None:

        _SPECIAL_IDS_B.add(
            int(token_id)
        )


# =========================================================================
# WORD COUNT
# =========================================================================

def _count_words_B(text):

    return len(
        text.strip().split()
    )


# =========================================================================
# TOKEN SAMPLER
# =========================================================================

def _sample_next_token_B(
    logits,
    generated_ids,
    temperature,
    top_p,
    top_k,
    repetition_penalty,
):

    logits = (
        logits
        .float()
        .clone()
    )

    # ---------------------------------------------------------------------
    # Repetition penalty
    # ---------------------------------------------------------------------

    if (
        repetition_penalty > 1.0
        and generated_ids
    ):

        for token_id in set(
            generated_ids
        ):

            if (
                0
                <= token_id
                < logits.numel()
            ):

                if logits[token_id] < 0:

                    logits[token_id] *= (
                        repetition_penalty
                    )

                else:

                    logits[token_id] /= (
                        repetition_penalty
                    )

    # ---------------------------------------------------------------------
    # Block special tokens
    # ---------------------------------------------------------------------

    for token_id in _SPECIAL_IDS_B:

        if (
            0
            <= token_id
            < logits.numel()
        ):

            logits[token_id] = (
                -float("inf")
            )

    # ---------------------------------------------------------------------
    # Temperature
    # ---------------------------------------------------------------------

    temperature = max(
        float(temperature),
        1e-5,
    )

    logits = (
        logits
        / temperature
    )

    # ---------------------------------------------------------------------
    # Top-k
    # ---------------------------------------------------------------------

    if (
        top_k > 0
        and top_k < logits.numel()
    ):

        values, indices = torch.topk(
            logits,
            k=int(top_k),
        )

        filtered = torch.full_like(
            logits,
            -float("inf"),
        )

        filtered.scatter_(
            0,
            indices,
            values,
        )

        logits = filtered

    # ---------------------------------------------------------------------
    # Top-p
    # ---------------------------------------------------------------------

    if top_p < 1.0:

        sorted_logits, sorted_indices = (
            torch.sort(
                logits,
                descending=True,
            )
        )

        sorted_probs = F.softmax(
            sorted_logits,
            dim=-1,
        )

        cumulative_probs = torch.cumsum(
            sorted_probs,
            dim=-1,
        )

        remove_mask = (
            cumulative_probs
            > top_p
        )

        # Always keep the most probable token.
        remove_mask[0] = False

        sorted_logits[
            remove_mask
        ] = -float("inf")

        filtered = torch.full_like(
            logits,
            -float("inf"),
        )

        filtered.scatter_(
            0,
            sorted_indices,
            sorted_logits,
        )

        logits = filtered

    # ---------------------------------------------------------------------
    # Final probability distribution
    # ---------------------------------------------------------------------

    probs = F.softmax(
        logits,
        dim=-1,
    )

    if (
        not torch.isfinite(
            probs
        ).all()
        or float(
            probs.sum()
        ) <= 0.0
    ):

        return int(
            torch.argmax(
                logits
            ).item()
        )

    return int(
        torch.multinomial(
            probs,
            num_samples=1,
        ).item()
    )


# =========================================================================
# WORD-ADVANCING RESCUE
# =========================================================================

def _find_word_advancing_token_B(
    logits,
    generated_ids,
    current_word_count,
    candidate_k=100,
):

    logits = (
        logits
        .float()
        .clone()
    )

    for token_id in _SPECIAL_IDS_B:

        if (
            0
            <= token_id
            < logits.numel()
        ):

            logits[token_id] = (
                -float("inf")
            )

    candidate_k = min(
        int(candidate_k),
        logits.numel(),
    )

    candidate_values, candidate_indices = (
        torch.topk(
            logits,
            k=candidate_k,
        )
    )

    best_token = None
    best_score = -float("inf")

    for score, token_tensor in zip(
        candidate_values.tolist(),
        candidate_indices.tolist(),
    ):

        token_id = int(
            token_tensor
        )

        candidate_ids = (
            generated_ids
            + [token_id]
        )

        candidate_text = TOKENIZER.decode(
            candidate_ids,
            skip_special_tokens=True,
        )

        candidate_word_count = (
            _count_words_B(
                candidate_text
            )
        )

        if (
            candidate_word_count
            > current_word_count
        ):

            if score > best_score:

                best_score = score
                best_token = token_id

    return best_token


# =========================================================================
# MAIN GENERATION FUNCTION
# =========================================================================

@torch.no_grad()
def generate_architecture_B_continuation(
    prompt,
    target_words,
    max_new_tokens,
    temperature,
    top_p,
    top_k,
    repetition_penalty,
):

    assert isinstance(
        prompt,
        str,
    )

    assert prompt.strip()

    assert target_words > 0

    assert max_new_tokens > 0

    MODEL.eval()

    # =====================================================================
    # TOKENIZE PROMPT
    # =====================================================================

    encoded_prompt = TOKENIZER.encode(
        prompt
    )

    prompt_ids = list(
        encoded_prompt.ids
    )

    assert prompt_ids, (
        "Prompt produced zero tokenizer tokens."
    )

    # =====================================================================
    # ENCODER CONTEXT
    # =====================================================================

    if (
        len(prompt_ids)
        > MODEL.src_context_length
    ):

        if RUN_MODE == "fast":

            prompt_ids = prompt_ids[
                -MODEL.src_context_length:
            ]

        else:

            raise ValueError(
                "Production Architecture-B prompt "
                "exceeds encoder context length.\n"
                f"Tokenized prompt: "
                f"{len(prompt_ids)}\n"
                f"Encoder context: "
                f"{MODEL.src_context_length}"
            )

    encoder_input_ids = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=DEVICE,
    )

    # =====================================================================
    # ENCODE PROMPT ONCE
    # =====================================================================

    with _architecture_B_autocast():

        encoder_memory = MODEL.encode(
            encoder_input_ids
        )

    # =====================================================================
    # ROLLING DECODER GENERATION LIMIT
    #
    # The old implementation limited total generation to
    # tgt_context_length - 1.
    #
    # Here, tgt_context_length remains the maximum context visible to the
    # decoder at ANY ONE TIME, but total generated tokens may be larger.
    #
    # We keep a conservative minimum safety budget of 32 tokenizer tokens
    # per requested whitespace word.
    # =====================================================================

    requested_max_new_tokens = int(
        max_new_tokens
    )

    minimum_safe_budget = max(
        1,
        int(target_words)
        * 32,
    )

    effective_max_new_tokens = max(
        requested_max_new_tokens,
        minimum_safe_budget,
    )

    decoder_context_length = int(
        MODEL.tgt_context_length
    )

    assert (
        decoder_context_length >= 2
    ), (
        "Decoder context is too small "
        "for rolling generation."
    )

    # =====================================================================
    # AUTOREGRESSIVE GENERATION
    # =====================================================================

    generated_ids = []

    stalled_tokens = 0

    last_word_count = 0

    for step in range(
        effective_max_new_tokens
    ):

        # -----------------------------------------------------------------
        # Rolling decoder window.
        #
        # Always keep:
        #
        # [START] + latest (context_length - 1) generated tokens
        #
        # This allows generation to continue beyond the model's original
        # decoder context length without passing an oversized sequence to
        # the decoder.
        # -----------------------------------------------------------------

        recent_generated_ids = (
            generated_ids[
                -(
                    decoder_context_length
                    - 1
                ):
            ]
        )

        decoder_input_ids = torch.tensor(
            [
                [
                    DECODER_START_TOKEN_ID,
                    *recent_generated_ids,
                ]
            ],
            dtype=torch.long,
            device=DEVICE,
        )

        assert (
            decoder_input_ids.size(1)
            <= decoder_context_length
        ), (
            "Rolling decoder window exceeded "
            "the configured decoder context."
        )

        # -----------------------------------------------------------------
        # Decoder forward pass.
        # -----------------------------------------------------------------

        with _architecture_B_autocast():

            decoder_hidden = MODEL.decode(
                decoder_input_ids,
                encoder_memory,
            )

            next_logits = F.linear(
                decoder_hidden[:, -1, :],
                MODEL.token_embedding.weight,
                MODEL.output_bias,
            )[0]

        # -----------------------------------------------------------------
        # Current word count.
        #
        # generated_ids retains the FULL generation history, while only
        # recent_generated_ids is supplied to the decoder.
        # -----------------------------------------------------------------

        current_text = TOKENIZER.decode(
            generated_ids,
            skip_special_tokens=True,
        )

        current_word_count = (
            _count_words_B(
                current_text
            )
        )

        # -----------------------------------------------------------------
        # Rescue if generation stalls.
        #
        # Keep the original FAST-only behavior so production decoding
        # remains probabilistic/natural unless explicitly configured for
        # FAST debugging.
        # -----------------------------------------------------------------

        use_rescue = (
            RUN_MODE == "fast"
            and stalled_tokens >= 16
            and current_word_count < target_words
        )

        if use_rescue:

            rescue_token = (
                _find_word_advancing_token_B(
                    next_logits,
                    generated_ids,
                    current_word_count,
                    candidate_k=100,
                )
            )

            if rescue_token is not None:

                next_token = rescue_token

            else:

                next_token = (
                    _sample_next_token_B(
                        next_logits,
                        generated_ids,
                        temperature,
                        top_p,
                        top_k,
                        repetition_penalty,
                    )
                )

        else:

            next_token = (
                _sample_next_token_B(
                    next_logits,
                    generated_ids,
                    temperature,
                    top_p,
                    top_k,
                    repetition_penalty,
                )
            )

        generated_ids.append(
            next_token
        )

        # -----------------------------------------------------------------
        # Check word-count progress.
        # -----------------------------------------------------------------

        updated_text = TOKENIZER.decode(
            generated_ids,
            skip_special_tokens=True,
        )

        updated_word_count = (
            _count_words_B(
                updated_text
            )
        )

        if (
            updated_word_count
            > last_word_count
        ):

            stalled_tokens = 0

        else:

            stalled_tokens += 1

        last_word_count = (
            updated_word_count
        )

        # -----------------------------------------------------------------
        # Target reached.
        # -----------------------------------------------------------------

        if (
            updated_word_count
            >= target_words
        ):

            break

    # =====================================================================
    # FINAL DECODE
    # =====================================================================

    final_text = TOKENIZER.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    final_words = final_text.split()

    # ---------------------------------------------------------------------
    # Keep exactly target_words.
    # ---------------------------------------------------------------------

    if (
        len(final_words)
        >= target_words
    ):

        final_text = " ".join(
            final_words[
                :target_words
            ]
        )

    final_word_count = (
        _count_words_B(
            final_text
        )
    )

    # =====================================================================
    # EXACT WORD COUNT
    # =====================================================================

    assert (
        final_word_count
        == target_words
    ), (
        "Architecture-B could not produce "
        f"the required {target_words} words "
        f"within {effective_max_new_tokens} "
        f"generated decoder tokens using "
        f"a rolling {decoder_context_length}-token "
        f"decoder context. "
        f"Produced {final_word_count} words."
    )

    assert final_text.strip()

    return final_text


# =========================================================================
# COMPATIBILITY ALIASES
# =========================================================================

generate_continuation_B = (
    generate_architecture_B_continuation
)

generate_continuation = (
    generate_architecture_B_continuation
)

generate_text = (
    generate_architecture_B_continuation
)


# =========================================================================
# SUMMARY
# =========================================================================

configured_max_tokens = int(
    GENERATION_CONFIG[
        "max_new_tokens"
    ]
)

minimum_safe_generation_budget = (
    int(
        GENERATION_CONFIG[
            "target_words"
        ]
    )
    * 32
)

effective_generation_max = max(
    configured_max_tokens,
    minimum_safe_generation_budget,
)

print()
print(
    "✅ Architecture-B generation function ready."
)

print(
    f"Run mode              : {RUN_MODE}"
)

print(
    f"Target words          : "
    f"{GENERATION_CONFIG['target_words']}"
)

print(
    f"Configured max tokens : "
    f"{configured_max_tokens}"
)

print(
    f"Effective max tokens  : "
    f"{effective_generation_max}"
)

print(
    f"Rolling decoder ctx   : "
    f"{MODEL.tgt_context_length}"
)

print(
    f"Temperature           : "
    f"{GENERATION_CONFIG['temperature']}"
)

print(
    f"Top-p                 : "
    f"{GENERATION_CONFIG['top_p']}"
)

print(
    f"Top-k                 : "
    f"{GENERATION_CONFIG['top_k']}"
)

print(
    f"Repetition penalty    : "
    f"{GENERATION_CONFIG['repetition_penalty']}"
)

print(
    f"FAST word rescue      : "
    f"{RUN_MODE == 'fast'}"
)

print(
    f"Encoder context       : "
    f"{MODEL.src_context_length}"
)

print(
    f"Decoder context       : "
    f"{MODEL.tgt_context_length}"
)

✍️ Building Architecture-B generation function...

✅ Architecture-B generation function ready.
Run mode              : production
Target words          : 128
Configured max tokens : 1536
Effective max tokens  : 4096
Rolling decoder ctx   : 1024
Temperature           : 0.7
Top-p                 : 0.95
Top-k                 : 50
Repetition penalty    : 1.1
FAST word rescue      : False
Encoder context       : 1024
Decoder context       : 1024


In [15]:
# Cell 12.5 — Production-safe Architecture-B generation patch
# DECISION: UPDATE

import math
from contextlib import nullcontext

import torch
import torch.nn.functional as F

print("🛠️ Installing Architecture-B production generation patch...")


# =========================================================================
# AUTOCAST
# =========================================================================

def _architecture_B_autocast_v2():

    helper = globals().get(
        "autocast_context"
    )

    if callable(helper):

        return helper()

    if (
        globals().get("USE_AMP", False)
        and DEVICE.type == "cuda"
    ):

        return torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        )

    return nullcontext()


# =========================================================================
# SPECIAL TOKENS
# =========================================================================

_SPECIAL_IDS_B_V2 = set()

for token_id in [
    BOS_TOKEN_ID,
    EOS_TOKEN_ID,
    PAD_TOKEN_ID,
    UNK_TOKEN_ID,
]:

    if token_id is not None:

        _SPECIAL_IDS_B_V2.add(
            int(token_id)
        )


# =========================================================================
# WORD COUNT
# =========================================================================

def _count_words_B_v2(text):

    return len(
        text.strip().split()
    )


# =========================================================================
# TOKEN SAMPLER
# =========================================================================

def _sample_next_token_B_v2(
    logits,
    generated_ids,
    temperature,
    top_p,
    top_k,
    repetition_penalty,
):

    logits = (
        logits
        .float()
        .clone()
    )

    # ---------------------------------------------------------------------
    # Apply repetition penalty only to recent history.
    # ---------------------------------------------------------------------

    penalty_window = min(
        len(generated_ids),
        int(MODEL.tgt_context_length) - 1,
    )

    recent_ids = (
        generated_ids[
            -penalty_window:
        ]
        if penalty_window > 0
        else []
    )

    if (
        repetition_penalty > 1.0
        and recent_ids
    ):

        for token_id in set(
            recent_ids
        ):

            if (
                0
                <= token_id
                < logits.numel()
            ):

                if logits[token_id] < 0:

                    logits[token_id] *= (
                        repetition_penalty
                    )

                else:

                    logits[token_id] /= (
                        repetition_penalty
                    )

    # ---------------------------------------------------------------------
    # Block special tokens.
    # ---------------------------------------------------------------------

    for token_id in _SPECIAL_IDS_B_V2:

        if (
            0
            <= token_id
            < logits.numel()
        ):

            logits[token_id] = (
                -float("inf")
            )

    # ---------------------------------------------------------------------
    # Temperature.
    # ---------------------------------------------------------------------

    temperature = max(
        float(temperature),
        1e-5,
    )

    logits = (
        logits
        / temperature
    )

    # ---------------------------------------------------------------------
    # Top-k.
    # ---------------------------------------------------------------------

    if (
        top_k > 0
        and top_k < logits.numel()
    ):

        values, indices = torch.topk(
            logits,
            k=int(top_k),
        )

        filtered = torch.full_like(
            logits,
            -float("inf"),
        )

        filtered.scatter_(
            0,
            indices,
            values,
        )

        logits = filtered

    # ---------------------------------------------------------------------
    # Top-p.
    # ---------------------------------------------------------------------

    if (
        top_p > 0.0
        and top_p < 1.0
    ):

        sorted_logits, sorted_indices = (
            torch.sort(
                logits,
                descending=True,
            )
        )

        sorted_probs = F.softmax(
            sorted_logits,
            dim=-1,
        )

        cumulative_probs = torch.cumsum(
            sorted_probs,
            dim=-1,
        )

        remove_mask = (
            cumulative_probs > top_p
        )

        # Keep the first token crossing the threshold.
        remove_mask[1:] = (
            remove_mask[:-1].clone()
        )

        remove_mask[0] = False

        sorted_logits[
            remove_mask
        ] = -float("inf")

        filtered = torch.full_like(
            logits,
            -float("inf"),
        )

        filtered.scatter_(
            0,
            sorted_indices,
            sorted_logits,
        )

        logits = filtered

    # ---------------------------------------------------------------------
    # Sample.
    # ---------------------------------------------------------------------

    probs = F.softmax(
        logits,
        dim=-1,
    )

    if (
        not torch.isfinite(
            probs
        ).all()
        or float(
            probs.sum()
        ) <= 0.0
    ):

        return int(
            torch.argmax(
                logits
            ).item()
        )

    return int(
        torch.multinomial(
            probs,
            num_samples=1,
        ).item()
    )


# =========================================================================
# WORD-ADVANCING RESCUE
# =========================================================================

def _find_word_advancing_token_B_v2(
    logits,
    generated_ids,
    current_word_count,
    candidate_k=512,
):

    logits = (
        logits
        .float()
        .clone()
    )

    for token_id in _SPECIAL_IDS_B_V2:

        if (
            0
            <= token_id
            < logits.numel()
        ):

            logits[token_id] = (
                -float("inf")
            )

    candidate_k = min(
        int(candidate_k),
        logits.numel(),
    )

    candidate_values, candidate_indices = (
        torch.topk(
            logits,
            k=candidate_k,
        )
    )

    best_token = None
    best_score = -float("inf")

    for score, token_tensor in zip(
        candidate_values.tolist(),
        candidate_indices.tolist(),
    ):

        token_id = int(
            token_tensor
        )

        candidate_ids = (
            generated_ids
            + [token_id]
        )

        candidate_text = TOKENIZER.decode(
            candidate_ids,
            skip_special_tokens=True,
        )

        candidate_word_count = (
            _count_words_B_v2(
                candidate_text
            )
        )

        if (
            candidate_word_count
            > current_word_count
        ):

            if score > best_score:

                best_score = score
                best_token = token_id

    return best_token


# =========================================================================
# MAIN GENERATION FUNCTION
# =========================================================================

@torch.no_grad()
def generate_architecture_B_continuation_v2(
    prompt,
    target_words,
    max_new_tokens,
    temperature,
    top_p,
    top_k,
    repetition_penalty,
):

    assert isinstance(
        prompt,
        str,
    )

    assert prompt.strip()

    target_words = int(
        target_words
    )

    assert target_words > 0

    MODEL.eval()

    # =====================================================================
    # TOKENIZE PROMPT
    # =====================================================================

    encoded_prompt = TOKENIZER.encode(
        prompt
    )

    prompt_ids = list(
        encoded_prompt.ids
    )

    assert prompt_ids, (
        "Prompt produced zero tokenizer tokens."
    )

    # =====================================================================
    # ENCODER CONTEXT
    # =====================================================================

    if (
        len(prompt_ids)
        > MODEL.src_context_length
    ):

        raise ValueError(
            "Architecture-B production prompt "
            "exceeds encoder context length.\n"
            f"Tokenized prompt: "
            f"{len(prompt_ids)}\n"
            f"Encoder context: "
            f"{MODEL.src_context_length}"
        )

    encoder_input_ids = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=DEVICE,
    )

    # =====================================================================
    # ENCODE PROMPT ONCE
    # =====================================================================

    with _architecture_B_autocast_v2():

        encoder_memory = MODEL.encode(
            encoder_input_ids
        )

    # =====================================================================
    # GENERATION BUDGET
    # =====================================================================

    requested_budget = int(
        max_new_tokens
    )

    # Safety allowance:
    # roughly 32 tokenizer tokens per whitespace word.
    minimum_safe_budget = (
        target_words * 32
    )

    generation_budget = max(
        requested_budget,
        minimum_safe_budget,
    )

    # Hard safety ceiling.
    generation_budget = min(
        generation_budget,
        4096,
    )

    decoder_context_length = int(
        MODEL.tgt_context_length
    )

    assert decoder_context_length >= 2

    # =====================================================================
    # GENERATION STATE
    # =====================================================================

    generated_ids = []

    stalled_tokens = 0
    last_word_count = 0

    RESCUE_AFTER = 12

    # =====================================================================
    # AUTOREGRESSIVE GENERATION
    # =====================================================================

    for _step in range(
        generation_budget
    ):

        # -----------------------------------------------------------------
        # Rolling decoder window.
        # -----------------------------------------------------------------

        recent_generated_ids = (
            generated_ids[
                -(
                    decoder_context_length
                    - 1
                ):
            ]
        )

        decoder_input_ids = torch.tensor(
            [
                [
                    DECODER_START_TOKEN_ID,
                    *recent_generated_ids,
                ]
            ],
            dtype=torch.long,
            device=DEVICE,
        )

        assert (
            decoder_input_ids.size(1)
            <= decoder_context_length
        )

        # -----------------------------------------------------------------
        # Decoder forward pass.
        # -----------------------------------------------------------------

        with _architecture_B_autocast_v2():

            decoder_hidden = MODEL.decode(
                decoder_input_ids,
                encoder_memory,
            )

            next_logits = F.linear(
                decoder_hidden[:, -1, :],
                MODEL.token_embedding.weight,
                MODEL.output_bias,
            )[0]

        # -----------------------------------------------------------------
        # Current word count.
        # -----------------------------------------------------------------

        current_text = TOKENIZER.decode(
            generated_ids,
            skip_special_tokens=True,
        )

        current_word_count = (
            _count_words_B_v2(
                current_text
            )
        )

        # -----------------------------------------------------------------
        # Word-advancing rescue.
        # -----------------------------------------------------------------

        rescue_token = None

        if (
            stalled_tokens
            >= RESCUE_AFTER
            and current_word_count < target_words
        ):

            rescue_token = (
                _find_word_advancing_token_B_v2(
                    next_logits,
                    generated_ids,
                    current_word_count,
                    candidate_k=512,
                )
            )

        # -----------------------------------------------------------------
        # Select next token.
        # -----------------------------------------------------------------

        if rescue_token is not None:

            next_token = rescue_token

        else:

            next_token = (
                _sample_next_token_B_v2(
                    next_logits,
                    generated_ids,
                    temperature,
                    top_p,
                    top_k,
                    repetition_penalty,
                )
            )

        generated_ids.append(
            next_token
        )

        # -----------------------------------------------------------------
        # Update word count.
        # -----------------------------------------------------------------

        updated_text = TOKENIZER.decode(
            generated_ids,
            skip_special_tokens=True,
        )

        updated_word_count = (
            _count_words_B_v2(
                updated_text
            )
        )

        if (
            updated_word_count
            > last_word_count
        ):

            stalled_tokens = 0

        else:

            stalled_tokens += 1

        last_word_count = (
            updated_word_count
        )

        # -----------------------------------------------------------------
        # Target reached.
        # -----------------------------------------------------------------

        if (
            updated_word_count
            >= target_words
        ):

            break

    # =====================================================================
    # FINAL DECODE
    # =====================================================================

    final_text = TOKENIZER.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    final_words = final_text.split()

    if (
        len(final_words)
        >= target_words
    ):

        final_text = " ".join(
            final_words[
                :target_words
            ]
        )

    final_word_count = (
        _count_words_B_v2(
            final_text
        )
    )

    # =====================================================================
    # FINAL VALIDATION
    # =====================================================================

    assert (
        final_word_count
        == target_words
    ), (
        "Architecture-B could not produce "
        f"the required {target_words} words "
        f"within {generation_budget} generated "
        f"decoder tokens using a rolling "
        f"{decoder_context_length}-token decoder "
        f"context. "
        f"Produced {final_word_count} words."
    )

    assert final_text.strip()

    return final_text


# =========================================================================
# REPLACE GENERATION ALIASES
# =========================================================================

generate_architecture_B_continuation = (
    generate_architecture_B_continuation_v2
)

generate_continuation_B = (
    generate_architecture_B_continuation_v2
)

generate_continuation = (
    generate_architecture_B_continuation_v2
)

generate_text = (
    generate_architecture_B_continuation_v2
)

if "GENERATE_B" in globals():

    GENERATE_B = (
        generate_architecture_B_continuation_v2
    )


# =========================================================================
# SUMMARY
# =========================================================================

configured_max_tokens = int(
    GENERATION_CONFIG[
        "max_new_tokens"
    ]
)

configured_target_words = int(
    GENERATION_CONFIG[
        "target_words"
    ]
)

minimum_safe_generation_budget = (
    configured_target_words * 32
)

effective_generation_max = max(
    configured_max_tokens,
    minimum_safe_generation_budget,
)

effective_generation_max = min(
    effective_generation_max,
    4096,
)

print()
print(
    "✅ Architecture-B production generation patch installed."
)

print(
    f"Decoder context     : "
    f"{MODEL.tgt_context_length}"
)

print(
    f"Target words        : "
    f"{configured_target_words}"
)

print(
    f"Configured max tokens : "
    f"{configured_max_tokens}"
)

print(
    f"Effective max tokens  : "
    f"{effective_generation_max}"
)

print(
    "Production rescue   : ENABLED"
)

print(
    "Rescue threshold    : 12 stalled tokens"
)

print(
    "Rescue candidates   : 512"
)

🛠️ Installing Architecture-B production generation patch...

✅ Architecture-B production generation patch installed.
Decoder context     : 1024
Target words        : 128
Configured max tokens : 1536
Effective max tokens  : 4096
Production rescue   : ENABLED
Rescue threshold    : 12 stalled tokens
Rescue candidates   : 512


In [16]:
# Cell 13 — Load the verified official Phase-1 test set for Architecture-B
# DECISION: UPDATE

import json
import shutil
from pathlib import Path


print("🧪 Loading Architecture-B official Phase-1 test set...")


# =========================================================================
# DIRECTORIES
# =========================================================================

TEST_DATA_ROOT = (
    ARCHITECTURE_B_ROOT / "test_data"
)

TEST_DATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# =========================================================================
# OFFICIAL DATASET
# =========================================================================

HF_REPO_ID = (
    "Exploration-Lab/CS-781-Capstone"
)


# =========================================================================
# HELPER — NORMALIZE OFFICIAL JSON
# =========================================================================

def normalize_phase1_test_json(
    test_json
):
    """
    Normalize supported official test structures into:

        [
            {
                "id": "...",
                "prompt": "..."
            },
            ...
        ]

    The official Phase-1 format documented by the course is:

        {
            "1": {"text": "..."},
            "2": {"text": "..."},
            ...
        }
    """

    records = []


    # ---------------------------------------------------------------------
    # Official dictionary format.
    # ---------------------------------------------------------------------
    if isinstance(
        test_json,
        dict,
    ):

        # Direct ID -> record mapping.
        direct_mapping = True

        for value in test_json.values():

            if not isinstance(
                value,
                dict,
            ):

                direct_mapping = False
                break

            if (
                "text" not in value
                and "prompt" not in value
                and "input" not in value
            ):

                direct_mapping = False
                break


        if direct_mapping:

            for test_id, value in (
                test_json.items()
            ):

                prompt = None

                for key in [
                    "text",
                    "prompt",
                    "input",
                ]:

                    if key in value:

                        prompt = str(
                            value[key]
                        )

                        break

                if prompt is None:
                    raise ValueError(
                        f"No prompt found for test ID "
                        f"{test_id}"
                    )

                records.append(
                    {
                        "id":
                            str(test_id),

                        "prompt":
                            prompt,
                    }
                )

            return records


        # -------------------------------------------------------------
        # Wrapper format.
        # -------------------------------------------------------------
        for wrapper_key in [
            "data",
            "records",
            "test",
            "examples",
            "items",
        ]:

            value = test_json.get(
                wrapper_key
            )

            if isinstance(
                value,
                list,
            ):

                return normalize_phase1_test_json(
                    value
                )


    # ---------------------------------------------------------------------
    # List format.
    # ---------------------------------------------------------------------
    if isinstance(
        test_json,
        list,
    ):

        for index, value in enumerate(
            test_json
        ):

            if not isinstance(
                value,
                dict,
            ):

                raise ValueError(
                    f"Test record {index} is not a "
                    "dictionary."
                )


            test_id = None

            for id_key in [
                "id",
                "ID",
                "test_id",
                "testId",
            ]:

                if id_key in value:

                    test_id = str(
                        value[id_key]
                    )

                    break


            if test_id is None:

                raise ValueError(
                    f"No test ID found in "
                    f"record {index}."
                )


            prompt = None

            for prompt_key in [
                "prompt",
                "text",
                "input",
            ]:

                if prompt_key in value:

                    prompt = str(
                        value[prompt_key]
                    )

                    break


            if prompt is None:

                raise ValueError(
                    f"No prompt found in "
                    f"record {index}."
                )


            records.append(
                {
                    "id":
                        test_id,

                    "prompt":
                        prompt,
                }
            )


        return records


    raise ValueError(
        "Unsupported Phase-1 test JSON structure."
    )


# =========================================================================
# HELPER — STRICT VALIDATION
# =========================================================================

def validate_phase1_test_records(
    records
):

    if not isinstance(
        records,
        list,
    ):

        return False

    if len(records) != 1000:

        return False

    ids = []

    for record in records:

        if not isinstance(
            record,
            dict,
        ):

            return False

        if (
            "id" not in record
            or "prompt" not in record
        ):

            return False

        test_id = str(
            record["id"]
        )

        prompt = str(
            record["prompt"]
        )

        if not test_id:
            return False

        if not prompt.strip():
            return False

        if (
            len(
                prompt.strip().split()
            )
            != 512
        ):

            return False

        ids.append(
            test_id
        )


    return (
        len(ids)
        == len(set(ids))
        == 1000
    )


# =========================================================================
# HELPER — LOAD + VALIDATE A FILE
# =========================================================================

def load_and_validate_test_file(
    path
):

    path = Path(path)

    if not path.is_file():

        return None


    try:

        with open(
            path,
            "r",
            encoding="utf-8",
        ) as f:

            raw = json.load(f)


        records = (
            normalize_phase1_test_json(
                raw
            )
        )


        if validate_phase1_test_records(
            records
        ):

            return records


    except Exception:

        return None


    return None


# =========================================================================
# 1. REUSE ARCHITECTURE-A VERIFIED TEST PATH
# =========================================================================

TEST_JSON_PATH = None
OFFICIAL_TEST_RECORDS = None


A_MANIFEST_PATH = (
    Path(
        "/kaggle/working/architecture_A"
    )
    / "configs"
    / "official_test_manifest_A.json"
)


if A_MANIFEST_PATH.is_file():

    try:

        with open(
            A_MANIFEST_PATH,
            "r",
            encoding="utf-8",
        ) as f:

            a_manifest = json.load(f)


        a_test_path = a_manifest.get(
            "test_json_path"
        )


        if a_test_path:

            a_test_path = Path(
                a_test_path
            )


            records = (
                load_and_validate_test_file(
                    a_test_path
                )
            )


            if records is not None:

                TEST_JSON_PATH = (
                    a_test_path
                )

                OFFICIAL_TEST_RECORDS = (
                    records
                )

                print(
                    "✅ Reused Architecture-A's "
                    "verified official test file:"
                )

                print(
                    f"   {TEST_JSON_PATH}"
                )

    except Exception:

        pass


# =========================================================================
# 2. CHECK COMMON LOCAL LOCATIONS
# =========================================================================

if OFFICIAL_TEST_RECORDS is None:

    local_candidates = [

        # Possible exact Phase-1 locations.
        Path(
            "/kaggle/input"
        ),

        Path(
            "/kaggle/working"
        ),

        # Architecture-A working tree.
        Path(
            "/kaggle/working/architecture_A"
        ),

        # Architecture-B working tree.
        Path(
            "/kaggle/working/architecture_B"
        ),
    ]


    candidate_files = []


    for root in local_candidates:

        if not root.exists():
            continue

        try:

            # Search only files whose names clearly indicate test data.
            for pattern in [
                "test.json",
                "Phase1- Test.json",
                "Phase1-Test.json",
                "*phase1*test*.json",
            ]:

                candidate_files.extend(
                    root.rglob(pattern)
                )

        except (
            PermissionError,
            OSError,
        ):

            continue


    # -------------------------------------------------------------
    # Prefer Phase-1 paths.
    # -------------------------------------------------------------
    unique_candidates = sorted(
        {
            path.resolve()
            for path in candidate_files
            if path.is_file()
        },
        key=lambda path: (
            0
            if "phase_1_pretraining" in (
                str(path).lower()
            )
            else 1,
            0
            if path.name.lower()
            == "test.json"
            else 1,
            len(str(path)),
        ),
    )


    print()
    print(
        f"Local test-file candidates: "
        f"{len(unique_candidates)}"
    )


    for candidate in (
        unique_candidates
    ):

        records = (
            load_and_validate_test_file(
                candidate
            )
        )


        if records is not None:

            TEST_JSON_PATH = (
                candidate
            )

            OFFICIAL_TEST_RECORDS = (
                records
            )

            print(
                "✅ Valid official test file found locally:"
            )

            print(
                f"   {TEST_JSON_PATH}"
            )

            break


# =========================================================================
# 3. EXACT OFFICIAL HUGGING FACE PATH
# =========================================================================

if OFFICIAL_TEST_RECORDS is None:

    print()
    print(
        "🌐 Downloading the exact official "
        "Phase-1 test file..."
    )


    try:

        from huggingface_hub import (
            hf_hub_download
        )

        hf_token = None


        # -------------------------------------------------------------
        # Kaggle HF token.
        # -------------------------------------------------------------
        try:

            from kaggle_secrets import (
                UserSecretsClient
            )

            secrets = UserSecretsClient()

            for secret_name in [
                "HF",
                "HUGGINGFACE_TOKEN",
                "HUGGINGFACEHUB_API_TOKEN",
            ]:

                try:

                    token = (
                        secrets.get_secret(
                            secret_name
                        )
                    )

                    if token:

                        hf_token = token
                        break

                except Exception:

                    continue

        except Exception:

            pass


        OFFICIAL_REPO_TEST_PATH = (
            "Phase_1_Pretraining/Test/test.json"
        )


        downloaded_path = (
            hf_hub_download(
                repo_id=HF_REPO_ID,
                filename=(
                    OFFICIAL_REPO_TEST_PATH
                ),
                repo_type="dataset",
                token=hf_token,
                local_dir=str(
                    TEST_DATA_ROOT
                ),
            )
        )


        downloaded_path = Path(
            downloaded_path
        )


        records = (
            load_and_validate_test_file(
                downloaded_path
            )
        )


        assert records is not None, (
            "The official Phase-1 test file was "
            "downloaded, but it did not pass "
            "the expected 1000 × 512-word validation."
        )


        TEST_JSON_PATH = (
            downloaded_path
        )

        OFFICIAL_TEST_RECORDS = (
            records
        )


        print(
            "✅ Exact official test file loaded:"
        )

        print(
            f"   {TEST_JSON_PATH}"
        )


    except Exception as exc:

        raise RuntimeError(
            "Could not load the official Phase-1 "
            "test file.\n"
            f"Error: {exc}"
        )


# =========================================================================
# FINAL GUARANTEE
# =========================================================================

assert (
    OFFICIAL_TEST_RECORDS is not None
)

assert (
    TEST_JSON_PATH is not None
)

assert (
    validate_phase1_test_records(
        OFFICIAL_TEST_RECORDS
    )
)


# =========================================================================
# COPY INTO B DIRECTORY
# =========================================================================

LOCAL_TEST_COPY = (
    TEST_DATA_ROOT
    / "official_phase1_test.json"
)


if (
    TEST_JSON_PATH.resolve()
    != LOCAL_TEST_COPY.resolve()
):

    shutil.copy2(
        TEST_JSON_PATH,
        LOCAL_TEST_COPY
    )


TEST_JSON_PATH = (
    LOCAL_TEST_COPY
)


# =========================================================================
# CANONICAL ARRAYS
# =========================================================================

OFFICIAL_TEST_IDS = [
    str(record["id"])
    for record
    in OFFICIAL_TEST_RECORDS
]

OFFICIAL_TEST_PROMPTS = [
    str(record["prompt"])
    for record
    in OFFICIAL_TEST_RECORDS
]


assert len(
    OFFICIAL_TEST_IDS
) == 1000

assert len(
    set(OFFICIAL_TEST_IDS)
) == 1000

assert all(
    len(
        prompt.strip().split()
    ) == 512
    for prompt
    in OFFICIAL_TEST_PROMPTS
)


# =========================================================================
# ACTIVE TEST SET
# =========================================================================

if RUN_MODE == "fast":

    ACTIVE_TEST_RECORDS = (
        OFFICIAL_TEST_RECORDS[:2]
    )

else:

    ACTIVE_TEST_RECORDS = (
        OFFICIAL_TEST_RECORDS
    )


EXPECTED_ACTIVE_COUNT = (
    2
    if RUN_MODE == "fast"
    else 1000
)


assert (
    len(ACTIVE_TEST_RECORDS)
    == EXPECTED_ACTIVE_COUNT
)


ACTIVE_TEST_IDS = [
    str(record["id"])
    for record
    in ACTIVE_TEST_RECORDS
]

ACTIVE_TEST_PROMPTS = [
    str(record["prompt"])
    for record
    in ACTIVE_TEST_RECORDS
]


# =========================================================================
# SAVE TEST MANIFEST
# =========================================================================

TEST_MANIFEST = {

    "project":
        "CS781 Phase 1",

    "experiment_id":
        EXPERIMENT_ID,

    "architecture_name":
        ARCHITECTURE_NAME,

    "architecture_version":
        ARCHITECTURE_VERSION,

    "run_mode":
        RUN_MODE,

    "huggingface_repo":
        HF_REPO_ID,

    "test_json_path":
        str(TEST_JSON_PATH),

    "total_official_examples":
        1000,

    "active_examples":
        len(ACTIVE_TEST_RECORDS),

    "prompt_word_count":
        512,

    "official_test_ids":
        list(OFFICIAL_TEST_IDS),

    "active_test_ids":
        list(ACTIVE_TEST_IDS),

    "validation":
        {
            "record_count":
                True,

            "unique_ids":
                True,

            "prompt_word_count":
                True,
        },
}


TEST_MANIFEST_PATH = (
    CONFIGS_DIR
    / "official_test_manifest_B.json"
)


with open(
    TEST_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        TEST_MANIFEST,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# SUMMARY
# =========================================================================

print()
print(
    "✅ Official Phase-1 test set loaded and validated."
)

print(
    f"Official examples : "
    f"{len(OFFICIAL_TEST_RECORDS):,}"
)

print(
    f"Active examples   : "
    f"{len(ACTIVE_TEST_RECORDS):,}"
)

print(
    f"Prompt words      : 512"
)

print(
    f"First active IDs  : "
    f"{ACTIVE_TEST_IDS[:5]}"
)

print(
    f"Verified source   : "
    f"{TEST_JSON_PATH}"
)

print(
    f"Test manifest     : "
    f"{TEST_MANIFEST_PATH}"
)

🧪 Loading Architecture-B official Phase-1 test set...

Local test-file candidates: 2
✅ Valid official test file found locally:
   /kaggle/input/datasets/rohitbatra2004/arch-data-1/architecture_B/test_data/Phase_1_Pretraining/Test/test.json

✅ Official Phase-1 test set loaded and validated.
Official examples : 1,000
Active examples   : 1,000
Prompt words      : 512
First active IDs  : ['1', '2', '3', '4', '5']
Verified source   : /kaggle/working/architecture_B/test_data/official_phase1_test.json
Test manifest     : /kaggle/working/architecture_B/configs/official_test_manifest_B.json


In [17]:
# Cell 14 — Generate Architecture-B official-test continuations with automatic resume
# DECISION: ADD

import hashlib
import json
import random
from pathlib import Path

import numpy as np
import torch


print("🚀 Starting Architecture-B official-test generation...")


# =========================================================================
# PATHS
# =========================================================================

METRICS_DIR = (
    ARCHITECTURE_B_ROOT / "metrics"
)

PREDICTIONS_DIR = (
    ARCHITECTURE_B_ROOT / "predictions"
)

METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PREDICTIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


GENERATION_PROGRESS_PATH = (
    METRICS_DIR
    / "generation_progress_B.json"
)

PARTIAL_PREDICTION_PATH = (
    PREDICTIONS_DIR
    / "prediction_B_partial.json"
)

FINAL_PREDICTION_PATH = (
    PREDICTIONS_DIR
    / "prediction_B.json"
)


# =========================================================================
# VERIFY BEST MODEL
# =========================================================================

assert BEST_CHECKPOINT_PATH.is_file(), (
    f"Best Architecture-B checkpoint not found:\n"
    f"{BEST_CHECKPOINT_PATH}\n"
    "Run Cell 09 and Cell 10 first."
)


BEST_CHECKPOINT = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)


assert checkpoint_is_compatible(
    BEST_CHECKPOINT
), (
    "Best Architecture-B checkpoint is incompatible "
    "with the current configuration."
)


# =========================================================================
# LOAD BEST MODEL FOR INFERENCE
# =========================================================================

MODEL.load_state_dict(
    BEST_CHECKPOINT["model_state_dict"]
)

MODEL = MODEL.to(
    DEVICE
)

MODEL.eval()


# =========================================================================
# OFFICIAL TEST DATA
# =========================================================================

assert (
    "ACTIVE_TEST_RECORDS" in globals()
), (
    "ACTIVE_TEST_RECORDS is unavailable. "
    "Run Cell 13 first."
)

assert len(
    ACTIVE_TEST_RECORDS
) > 0


# =========================================================================
# GENERATION FUNCTION
# =========================================================================

GENERATE_B = (
    generate_architecture_B_continuation
)

assert callable(
    GENERATE_B
), (
    "Architecture-B generation function "
    "is unavailable. Run Cell 12 first."
)


# =========================================================================
# TEST FIELD HELPERS
# =========================================================================

def _test_id_B(record):

    return str(
        record["id"]
    )


def _test_prompt_B(record):

    return str(
        record["prompt"]
    )


ACTIVE_TEST_IDS = [
    _test_id_B(record)
    for record in ACTIVE_TEST_RECORDS
]

ACTIVE_TEST_PROMPTS = [
    _test_prompt_B(record)
    for record in ACTIVE_TEST_RECORDS
]


assert len(
    ACTIVE_TEST_IDS
) == len(
    set(ACTIVE_TEST_IDS)
), (
    "Duplicate active test IDs detected."
)


# =========================================================================
# STABLE PER-EXAMPLE SEED
# =========================================================================

def _stable_generation_seed_B(
    test_id
):

    digest = hashlib.sha256(
        (
            f"{EXPERIMENT_ID}|"
            f"{ARCHITECTURE_NAME}|"
            f"{ARCHITECTURE_VERSION}|"
            f"{test_id}"
        ).encode("utf-8")
    ).digest()

    return (
        int.from_bytes(
            digest[:8],
            byteorder="big",
            signed=False,
        )
        % (2**31 - 1)
    )


def _set_generation_seed_B(
    seed
):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(
            seed
        )

        torch.cuda.manual_seed_all(
            seed
        )


# =========================================================================
# LOAD EXISTING PROGRESS
# =========================================================================

predictions = {}
completed_ids = set()


if (
    GENERATION_PROGRESS_PATH.is_file()
    and PARTIAL_PREDICTION_PATH.is_file()
):

    try:

        with open(
            GENERATION_PROGRESS_PATH,
            "r",
            encoding="utf-8",
        ) as f:

            generation_progress = json.load(f)

        with open(
            PARTIAL_PREDICTION_PATH,
            "r",
            encoding="utf-8",
        ) as f:

            predictions = json.load(f)

        # -------------------------------------------------------------
        # Verify identity.
        # -------------------------------------------------------------
        assert (
            generation_progress[
                "experiment_id"
            ]
            == EXPERIMENT_ID
        )

        assert (
            generation_progress[
                "architecture_name"
            ]
            == ARCHITECTURE_NAME
        )

        assert (
            generation_progress[
                "architecture_version"
            ]
            == ARCHITECTURE_VERSION
        )

        assert (
            generation_progress[
                "run_mode"
            ]
            == RUN_MODE
        )

        assert (
            generation_progress[
                "target_words"
            ]
            == GENERATION_CONFIG[
                "target_words"
            ]
        )

        assert (
            generation_progress[
                "config_fingerprint"
            ]
            == CONFIG_FINGERPRINT
        )

        # -------------------------------------------------------------
        # Keep only valid dictionary-shaped entries.
        # -------------------------------------------------------------
        if isinstance(
            predictions,
            dict,
        ):

            for test_id, value in list(
                predictions.items()
            ):

                if (
                    str(test_id)
                    not in ACTIVE_TEST_IDS
                ):

                    del predictions[
                        test_id
                    ]

                    continue

                if not isinstance(
                    value,
                    dict,
                ):

                    del predictions[
                        test_id
                    ]

                    continue

                if set(
                    value.keys()
                ) != {
                    "text"
                }:

                    del predictions[
                        test_id
                    ]

                    continue

                if not isinstance(
                    value["text"],
                    str,
                ):

                    del predictions[
                        test_id
                    ]

                    continue

                word_count = len(
                    value["text"]
                    .strip()
                    .split()
                )

                if (
                    word_count
                    != GENERATION_CONFIG[
                        "target_words"
                    ]
                ):

                    del predictions[
                        test_id
                    ]


        completed_ids = set(
            str(test_id)
            for test_id in predictions.keys()
        )

        print(
            f"♻️ Resuming generation: "
            f"{len(completed_ids):,}/"
            f"{len(ACTIVE_TEST_RECORDS):,} "
            f"complete."
        )

    except Exception:

        print(
            "ℹ️ Existing generation progress "
            "is incompatible or corrupted."
        )

        predictions = {}
        completed_ids = set()


else:

    generation_progress = {
        "project":
            "CS781 Phase 1",

        "experiment_id":
            EXPERIMENT_ID,

        "architecture_name":
            ARCHITECTURE_NAME,

        "architecture_version":
            ARCHITECTURE_VERSION,

        "run_mode":
            RUN_MODE,

        "config_fingerprint":
            CONFIG_FINGERPRINT,

        "target_words":
            GENERATION_CONFIG[
                "target_words"
            ],

        "max_new_tokens":
            GENERATION_CONFIG[
                "max_new_tokens"
            ],

        "completed_ids":
            [],

        "completed_count":
            0,

        "status":
            "started",
    }


# =========================================================================
# GENERATE EACH TEST EXAMPLE
# =========================================================================

total_examples = len(
    ACTIVE_TEST_RECORDS
)

target_words = int(
    GENERATION_CONFIG[
        "target_words"
    ]
)

max_new_tokens = int(
    GENERATION_CONFIG[
        "max_new_tokens"
    ]
)

temperature = float(
    GENERATION_CONFIG[
        "temperature"
    ]
)

top_p = float(
    GENERATION_CONFIG[
        "top_p"
    ]
)

top_k = int(
    GENERATION_CONFIG[
        "top_k"
    ]
)

repetition_penalty = float(
    GENERATION_CONFIG[
        "repetition_penalty"
    ]
)


for index, record in enumerate(
    ACTIVE_TEST_RECORDS,
    start=1,
):

    test_id = _test_id_B(
        record
    )

    prompt = _test_prompt_B(
        record
    )


    # ---------------------------------------------------------------------
    # Skip already completed examples.
    # ---------------------------------------------------------------------
    if test_id in completed_ids:

        print(
            f"[{index:>4}/{total_examples:<4}] "
            f"ID={test_id} | already complete"
        )

        continue


    # ---------------------------------------------------------------------
    # Stable deterministic seed for this test example.
    # ---------------------------------------------------------------------
    seed = _stable_generation_seed_B(
        test_id
    )

    _set_generation_seed_B(
        seed
    )


    # ---------------------------------------------------------------------
    # Generate continuation.
    # ---------------------------------------------------------------------
    continuation = GENERATE_B(
        prompt,
        target_words=target_words,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=repetition_penalty,
    )


    # ---------------------------------------------------------------------
    # Validate exact word count.
    # ---------------------------------------------------------------------
    generated_word_count = len(
        continuation.strip().split()
    )

    assert (
        generated_word_count
        == target_words
    ), (
        f"Test ID {test_id}: expected "
        f"{target_words} words, got "
        f"{generated_word_count}."
    )


    # ---------------------------------------------------------------------
    # Store ONLY the continuation.
    # ---------------------------------------------------------------------
    predictions[test_id] = {
        "text": continuation
    }

    completed_ids.add(
        test_id
    )


    # ---------------------------------------------------------------------
    # Persist immediately after every successful example.
    # ---------------------------------------------------------------------
    generation_progress[
        "completed_ids"
    ] = sorted(
        completed_ids
    )

    generation_progress[
        "completed_count"
    ] = len(
        completed_ids
    )

    generation_progress[
        "last_completed_id"
    ] = test_id

    generation_progress[
        "status"
    ] = "running"


    with open(
        PARTIAL_PREDICTION_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            predictions,
            f,
            ensure_ascii=False,
            indent=2,
        )


    with open(
        GENERATION_PROGRESS_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            generation_progress,
            f,
            ensure_ascii=False,
            indent=2,
        )


    print(
        f"[{index:>4}/{total_examples:<4}] "
        f"ID={test_id} | "
        f"{generated_word_count} words | "
        f"seed={seed}"
    )


# =========================================================================
# FINAL VALIDATION
# =========================================================================

EXPECTED_IDS = set(
    ACTIVE_TEST_IDS
)

assert (
    set(predictions.keys())
    == EXPECTED_IDS
), (
    "Prediction IDs do not exactly match "
    "the active official test IDs.\n"
    f"Missing: "
    f"{sorted(EXPECTED_IDS - set(predictions.keys()))[:10]}\n"
    f"Unexpected: "
    f"{sorted(set(predictions.keys()) - EXPECTED_IDS)[:10]}"
)


for test_id, prediction in (
    predictions.items()
):

    assert isinstance(
        prediction,
        dict,
    )

    assert set(
        prediction.keys()
    ) == {
        "text"
    }

    assert isinstance(
        prediction["text"],
        str,
    )

    assert (
        len(
            prediction["text"]
            .strip()
            .split()
        )
        == target_words
    ), (
        f"Prediction {test_id} has "
        "incorrect word count."
    )


# =========================================================================
# PRESERVE OFFICIAL TEST ORDER
# =========================================================================

ordered_predictions = {
    test_id: predictions[test_id]
    for test_id in ACTIVE_TEST_IDS
}


# =========================================================================
# SAVE FINAL PREDICTION FILE
# =========================================================================

with open(
    FINAL_PREDICTION_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        ordered_predictions,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# FINAL PROGRESS
# =========================================================================

generation_progress[
    "completed_ids"
] = sorted(
    completed_ids
)

generation_progress[
    "completed_count"
] = len(
    completed_ids
)

generation_progress[
    "status"
] = "complete"

generation_progress[
    "final_prediction_path"
] = str(
    FINAL_PREDICTION_PATH
)


with open(
    GENERATION_PROGRESS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        generation_progress,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# SUMMARY
# =========================================================================

print()
print(
    "✅ Architecture-B official-test generation complete."
)

print(
    f"Examples generated : "
    f"{len(ordered_predictions):,}"
)

print(
    f"Words/example       : "
    f"{target_words}"
)

print(
    f"Prediction format   : "
    f"ID → {{'text': continuation}}"
)

print(
    f"Prediction file     : "
    f"{FINAL_PREDICTION_PATH}"
)

print(
    f"Progress file       : "
    f"{GENERATION_PROGRESS_PATH}"
)

🚀 Starting Architecture-B official-test generation...
[   1/1000] ID=1 | 128 words | seed=156504570
[   2/1000] ID=2 | 128 words | seed=1870776602
[   3/1000] ID=3 | 128 words | seed=1022664409
[   4/1000] ID=4 | 128 words | seed=1434929785
[   5/1000] ID=5 | 128 words | seed=1988224842
[   6/1000] ID=6 | 128 words | seed=1336903723
[   7/1000] ID=7 | 128 words | seed=2039842960
[   8/1000] ID=8 | 128 words | seed=642198781
[   9/1000] ID=9 | 128 words | seed=1663885468
[  10/1000] ID=10 | 128 words | seed=1480246138
[  11/1000] ID=11 | 128 words | seed=1842702395
[  12/1000] ID=12 | 128 words | seed=310734962
[  13/1000] ID=13 | 128 words | seed=116247493
[  14/1000] ID=14 | 128 words | seed=1490007969
[  15/1000] ID=15 | 128 words | seed=1148215873
[  16/1000] ID=16 | 128 words | seed=1566547731
[  17/1000] ID=17 | 128 words | seed=1724748751
[  18/1000] ID=18 | 128 words | seed=1149179566
[  19/1000] ID=19 | 128 words | seed=176419791
[  20/1000] ID=20 | 128 words | seed=894067005
[

In [18]:
# Cell 15 — Validate and package Architecture-B submission
# DECISION: ADD

import json
import zipfile
from pathlib import Path


print("📦 Preparing Architecture-B submission package...")


# =========================================================================
# SELF-CONTAINED PATHS
# =========================================================================

ARCHITECTURE_B_ROOT = Path(
    "/kaggle/working/architecture_B"
)

PREDICTIONS_DIR = (
    ARCHITECTURE_B_ROOT / "predictions"
)

METRICS_DIR = (
    ARCHITECTURE_B_ROOT / "metrics"
)

SUBMISSION_DIR = (
    ARCHITECTURE_B_ROOT / "submission_package"
)

PREDICTION_PATH = (
    PREDICTIONS_DIR / "prediction_B.json"
)

SUBMISSION_PREDICTION_PATH = (
    SUBMISSION_DIR / "prediction.json"
)

SUBMISSION_ZIP_PATH = (
    ARCHITECTURE_B_ROOT / "submission.zip"
)

SUBMISSION_MANIFEST_PATH = (
    METRICS_DIR / "submission_manifest_B.json"
)


# =========================================================================
# BASIC PATH CHECKS
# =========================================================================

assert PREDICTION_PATH.is_file(), (
    f"Prediction file not found:\n"
    f"{PREDICTION_PATH}\n\n"
    "Run Cell 14 successfully first."
)

PREDICTIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =========================================================================
# LOAD PREDICTIONS
# =========================================================================

with open(
    PREDICTION_PATH,
    "r",
    encoding="utf-8",
) as f:

    predictions = json.load(f)


# =========================================================================
# OFFICIAL FORMAT CHECK
# =========================================================================
# Expected:
#
# {
#     "1": {
#         "text": "..."
#     },
#     "2": {
#         "text": "..."
#     }
# }
#
# The official prompt must NOT appear in the output text.

assert isinstance(
    predictions,
    dict,
), (
    "Architecture-B prediction_B.json must be "
    "a dictionary keyed by official test ID."
)


# =========================================================================
# EXPECTED COUNT
# =========================================================================

if RUN_MODE == "fast":

    expected_count = len(
        ACTIVE_TEST_RECORDS
    )

else:

    expected_count = 1000


assert len(predictions) == expected_count, (
    f"Expected {expected_count} predictions "
    f"in {RUN_MODE} mode, found "
    f"{len(predictions)}."
)


# =========================================================================
# ACTIVE OFFICIAL IDS
# =========================================================================

expected_ids = [
    str(record["id"])
    for record in ACTIVE_TEST_RECORDS
]

expected_id_set = set(
    expected_ids
)

actual_id_set = set(
    str(test_id)
    for test_id in predictions.keys()
)


assert actual_id_set == expected_id_set, (
    "Prediction IDs do not exactly match "
    "the active official test IDs.\n\n"
    f"Missing IDs: "
    f"{sorted(expected_id_set - actual_id_set)[:20]}\n\n"
    f"Unexpected IDs: "
    f"{sorted(actual_id_set - expected_id_set)[:20]}"
)


# =========================================================================
# VALIDATE EVERY PREDICTION
# =========================================================================

target_words = int(
    GENERATION_CONFIG[
        "target_words"
    ]
)


for test_id in expected_ids:

    assert test_id in predictions, (
        f"Missing prediction for test ID "
        f"{test_id}"
    )

    prediction = predictions[
        test_id
    ]

    # -------------------------------------------------------------
    # Each value must contain ONLY "text".
    # -------------------------------------------------------------
    assert isinstance(
        prediction,
        dict,
    ), (
        f"Prediction {test_id} must be "
        "a dictionary."
    )

    assert set(
        prediction.keys()
    ) == {
        "text"
    }, (
        f"Prediction {test_id} must contain "
        "only the 'text' field. "
        f"Found: {list(prediction.keys())}"
    )

    continuation = prediction[
        "text"
    ]

    assert isinstance(
        continuation,
        str,
    ), (
        f"Prediction {test_id} text must "
        "be a string."
    )

    assert continuation.strip(), (
        f"Prediction {test_id} text is empty."
    )

    # -------------------------------------------------------------
    # Exact word count.
    # -------------------------------------------------------------
    word_count = len(
        continuation.strip().split()
    )

    assert word_count == target_words, (
        f"Prediction {test_id}: expected "
        f"{target_words} words, found "
        f"{word_count}."
    )


# =========================================================================
# PRESERVE OFFICIAL TEST ORDER
# =========================================================================

ordered_predictions = {}

for test_id in expected_ids:

    ordered_predictions[
        test_id
    ] = {
        "text": predictions[
            test_id
        ]["text"]
    }


# =========================================================================
# RECREATE CLEAN SUBMISSION DIRECTORY
# =========================================================================

if SUBMISSION_DIR.exists():

    import shutil

    shutil.rmtree(
        SUBMISSION_DIR
    )

SUBMISSION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# =========================================================================
# WRITE OFFICIAL prediction.json
# =========================================================================

with open(
    SUBMISSION_PREDICTION_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        ordered_predictions,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# CREATE submission.zip
# =========================================================================

if SUBMISSION_ZIP_PATH.exists():

    SUBMISSION_ZIP_PATH.unlink()


with zipfile.ZipFile(
    SUBMISSION_ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    # IMPORTANT:
    # No folder prefix.
    zf.write(
        SUBMISSION_PREDICTION_PATH,
        arcname="prediction.json",
    )


# =========================================================================
# VERIFY ZIP STRUCTURE
# =========================================================================

with zipfile.ZipFile(
    SUBMISSION_ZIP_PATH,
    mode="r",
) as zf:

    zip_contents = zf.namelist()


assert zip_contents == [
    "prediction.json"
], (
    "Invalid submission ZIP structure.\n"
    f"Found: {zip_contents}"
)


# =========================================================================
# VERIFY ZIP CONTENT
# =========================================================================

with zipfile.ZipFile(
    SUBMISSION_ZIP_PATH,
    mode="r",
) as zf:

    with zf.open(
        "prediction.json"
    ) as f:

        zipped_predictions = json.loads(
            f.read().decode("utf-8")
        )


assert zipped_predictions == (
    ordered_predictions
), (
    "prediction.json inside submission.zip "
    "does not match the validated predictions."
)


# =========================================================================
# VERIFY FINAL FILE DIRECTLY
# =========================================================================

with open(
    SUBMISSION_PREDICTION_PATH,
    "r",
    encoding="utf-8",
) as f:

    final_submission_predictions = (
        json.load(f)
    )


assert (
    final_submission_predictions
    == ordered_predictions
)


# =========================================================================
# SAVE SUBMISSION MANIFEST
# =========================================================================

submission_manifest = {

    "project":
        "CS781 Phase 1",

    "experiment_id":
        EXPERIMENT_ID,

    "architecture_name":
        ARCHITECTURE_NAME,

    "architecture_version":
        ARCHITECTURE_VERSION,

    "run_mode":
        RUN_MODE,

    "prediction_count":
        len(ordered_predictions),

    "target_words":
        target_words,

    "prediction_format":
        "official_id_keyed_dictionary",

    "prediction_file":
        str(PREDICTION_PATH),

    "submission_prediction_file":
        str(SUBMISSION_PREDICTION_PATH),

    "submission_zip":
        str(SUBMISSION_ZIP_PATH),

    "zip_contents":
        zip_contents,

    "validation": {

        "all_active_ids_present":
            True,

        "no_unknown_ids":
            True,

        "unique_ids":
            True,

        "text_only_value":
            True,

        "exact_word_count":
            True,

        "official_test_order_preserved":
            True,

        "zip_root_structure_verified":
            True,

        "zip_content_verified":
            True,
    },
}


with open(
    SUBMISSION_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        submission_manifest,
        f,
        ensure_ascii=False,
        indent=2,
    )


# =========================================================================
# SUMMARY
# =========================================================================

zip_size_mb = (
    SUBMISSION_ZIP_PATH.stat().st_size
    / (1024 ** 2)
)

print()
print(
    "✅ Architecture-B submission packaged successfully."
)

print()
print(
    f"Run mode           : {RUN_MODE}"
)

print(
    f"Predictions        : "
    f"{len(ordered_predictions)}"
)

print(
    f"Words / prediction : "
    f"{target_words}"
)

print(
    f"Prediction format  : "
    f"ID → {{'text': continuation}}"
)

print(
    f"Prediction JSON    : "
    f"{SUBMISSION_PREDICTION_PATH}"
)

print(
    f"Submission ZIP     : "
    f"{SUBMISSION_ZIP_PATH}"
)

print(
    f"ZIP size           : "
    f"{zip_size_mb:.4f} MB"
)

print(
    f"Manifest           : "
    f"{SUBMISSION_MANIFEST_PATH}"
)

print()
print("ZIP contents:")

for item in zip_contents:

    print(
        f"  ✅ {item}"
    )

📦 Preparing Architecture-B submission package...

✅ Architecture-B submission packaged successfully.

Run mode           : production
Predictions        : 1000
Words / prediction : 128
Prediction format  : ID → {'text': continuation}
Prediction JSON    : /kaggle/working/architecture_B/submission_package/prediction.json
Submission ZIP     : /kaggle/working/architecture_B/submission.zip
ZIP size           : 0.1832 MB
Manifest           : /kaggle/working/architecture_B/metrics/submission_manifest_B.json

ZIP contents:
  ✅ prediction.json


In [19]:
# Cell 15.5 — Repair Architecture-B best-checkpoint metadata
# DECISION: REPLACE previous Cell 15.5
#
# Why:
# The recovered dataset does NOT contain best_step_*.pt files.
# Therefore, historical-best count must be recovered from the checkpoint
# state or, if unavailable, reconstructed from saved validation history.

import json
import re
from pathlib import Path
import torch

ROOT = Path("/kaggle/working/architecture_B")
CHECKPOINT_DIR = ROOT / "checkpoints"
METADATA_PATH = ROOT / "model_B_metadata.json"

assert ROOT.is_dir(), f"❌ Missing directory: {ROOT}"
assert CHECKPOINT_DIR.is_dir(), f"❌ Missing checkpoint directory: {CHECKPOINT_DIR}"
assert METADATA_PATH.is_file(), f"❌ Missing metadata file: {METADATA_PATH}"

print("🔧 Repairing Architecture-B best-checkpoint metadata...\n")

# ------------------------------------------------------------
# 1. Inspect available checkpoint files
# ------------------------------------------------------------
checkpoint_files = sorted(CHECKPOINT_DIR.glob("*.pt"))

print("Available checkpoint files:")
for p in checkpoint_files:
    print("  ", p.name)

print()

# ------------------------------------------------------------
# 2. Helper: recursively find a key inside nested dictionaries
# ------------------------------------------------------------
def find_key_recursive(obj, target_key, path="root"):
    found = []

    if isinstance(obj, dict):
        for k, v in obj.items():
            current_path = f"{path}.{k}"

            if k == target_key:
                found.append((current_path, v))

            found.extend(find_key_recursive(v, target_key, current_path))

    elif isinstance(obj, (list, tuple)):
        for i, v in enumerate(obj):
            found.extend(
                find_key_recursive(v, target_key, f"{path}[{i}]")
            )

    return found


# ------------------------------------------------------------
# 3. Try to recover best_checkpoint_count from latest.pt
# ------------------------------------------------------------
candidate_counts = []

latest_path = CHECKPOINT_DIR / "latest.pt"
best_path = CHECKPOINT_DIR / "best.pt"

for ckpt_path in [latest_path, best_path]:
    if not ckpt_path.is_file():
        continue

    try:
        checkpoint = torch.load(
            ckpt_path,
            map_location="cpu",
            weights_only=False
        )

        matches = find_key_recursive(
            checkpoint,
            "best_checkpoint_count"
        )

        if matches:
            print(f"📦 {ckpt_path.name} contains:")
            for path, value in matches:
                print(f"    {path} = {value}")

                if isinstance(value, (int, float)):
                    value = int(value)

                    if 1 <= value <= 12:
                        candidate_counts.append(
                            (ckpt_path.name, path, value)
                        )

        else:
            print(
                f"⚠️ {ckpt_path.name}: "
                "best_checkpoint_count not found."
            )

    except Exception as e:
        print(f"⚠️ Could not load {ckpt_path.name}: {e}")

print()

# ------------------------------------------------------------
# 4. Select recovered count from checkpoint
# ------------------------------------------------------------
recovered_count = None

if candidate_counts:
    # Prefer latest.pt because it represents the newest training state.
    latest_candidates = [
        x for x in candidate_counts
        if x[0] == "latest.pt"
    ]

    if latest_candidates:
        recovered_count = latest_candidates[0][2]
        recovered_source = (
            f"checkpoint:{latest_candidates[0][0]}:"
            f"{latest_candidates[0][1]}"
        )
    else:
        recovered_count = candidate_counts[0][2]
        recovered_source = (
            f"checkpoint:{candidate_counts[0][0]}:"
            f"{candidate_counts[0][1]}"
        )

# ------------------------------------------------------------
# 5. If checkpoint does not contain the count, inspect metadata
# ------------------------------------------------------------
with open(METADATA_PATH, "r") as f:
    metadata = json.load(f)

metadata_existing_count = metadata.get("best_checkpoint_count")

print(f"Metadata currently says best_checkpoint_count = "
      f"{metadata_existing_count}")

# ------------------------------------------------------------
# 6. Fallback: reconstruct from training history
# ------------------------------------------------------------
if recovered_count is None:

    print("\n🔎 Checkpoint count unavailable.")
    print("Attempting reconstruction from saved validation history...")

    # Candidate history files
    history_candidates = [
        ROOT / "training_history.json",
        ROOT / "metrics.json",
        ROOT / "training_metrics.json",
        ROOT / "history.json",
    ]

    history_path = next(
        (p for p in history_candidates if p.is_file()),
        None
    )

    if history_path is not None:

        print(f"📈 Using history file: {history_path.name}")

        try:
            with open(history_path, "r") as f:
                history = json.load(f)

            # Find every validation loss recursively.
            val_losses = find_key_recursive(
                history,
                "validation_loss"
            )

            if not val_losses:
                val_losses = find_key_recursive(
                    history,
                    "val_loss"
                )

            print(f"Validation-loss entries found: {len(val_losses)}")

            # Flatten numeric validation losses in their stored order.
            losses = []

            for _, value in val_losses:
                if isinstance(value, (int, float)):
                    losses.append(float(value))

            if losses:
                # Same concept used by the training process:
                # count successive validation improvements.
                #
                # We use a conservative 0.005 threshold, matching the
                # Architecture-B early-stopping/checkpoint improvement
                # threshold used in the production configuration.
                MIN_DELTA = 0.005

                best_loss = float("inf")
                improvement_count = 0

                for loss in losses:
                    if loss < best_loss - MIN_DELTA:
                        best_loss = loss
                        improvement_count += 1

                if 1 <= improvement_count <= 12:
                    recovered_count = improvement_count
                    recovered_source = (
                        f"reconstructed_from:{history_path.name}"
                    )

        except Exception as e:
            print(f"⚠️ History reconstruction failed: {e}")

print()

# ------------------------------------------------------------
# 7. Final safety check
# ------------------------------------------------------------
assert recovered_count is not None, (
    "❌ Could not recover best_checkpoint_count from "
    "latest.pt, best.pt, metadata, or training history.\n"
    "Do NOT manually guess the value."
)

assert 1 <= int(recovered_count) <= 12, (
    f"❌ Recovered invalid best_checkpoint_count: {recovered_count}"
)

recovered_count = int(recovered_count)

print("✅ Recovered best-checkpoint count:", recovered_count)
print("✅ Source:", recovered_source)

# ------------------------------------------------------------
# 8. Repair ONLY the incorrect metadata fields
# ------------------------------------------------------------
old_count = metadata.get("best_checkpoint_count")

metadata["best_checkpoint_count"] = recovered_count
metadata["best_checkpoint_count_source"] = recovered_source

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print()
print("🛠️ Metadata repaired successfully.")
print(f"   Old count : {old_count}")
print(f"   New count : {recovered_count}")
print(f"   Metadata  : {METADATA_PATH}")
print()
print("➡️ Now rerun Cell 16.")

🔧 Repairing Architecture-B best-checkpoint metadata...

Available checkpoint files:
   best.pt
   latest.pt

📦 latest.pt contains:
    root.best_checkpoint_count = 11
    root.training_history[0].best_checkpoint_count = 1
    root.training_history[1].best_checkpoint_count = 2
    root.training_history[2].best_checkpoint_count = 3
    root.training_history[3].best_checkpoint_count = 4
    root.training_history[4].best_checkpoint_count = 5
    root.training_history[5].best_checkpoint_count = 5
    root.training_history[6].best_checkpoint_count = 6
    root.training_history[7].best_checkpoint_count = 7
    root.training_history[8].best_checkpoint_count = 8
    root.training_history[9].best_checkpoint_count = 9
    root.training_history[10].best_checkpoint_count = 10
    root.training_history[11].best_checkpoint_count = 11
    root.training_history[12].best_checkpoint_count = 11
📦 best.pt contains:
    root.best_checkpoint_count = 11

Metadata currently says best_checkpoint_count = 0

✅ Re

In [20]:
# Cell 16 — Final Architecture-B audit and artifact inventory
# DECISION: UPDATE

import json
import math
import zipfile
from pathlib import Path

import torch


print("🔍 Running final Architecture-B audit...")


# =========================================================================
# ROOT PATHS
# =========================================================================

ROOT = Path(
    "/kaggle/working/architecture_B"
)

CHECKPOINT_DIR = (
    ROOT / "checkpoints"
)

METRICS_DIR = (
    ROOT / "metrics"
)

PREDICTIONS_DIR = (
    ROOT / "predictions"
)

CONFIG_DIR = (
    ROOT / "configs"
)

MODEL_PATH = (
    ROOT / "model_B_final.pt"
)

STATE_DICT_PATH = (
    ROOT / "model_B_state_dict.pt"
)

PREDICTION_PATH = (
    PREDICTIONS_DIR / "prediction_B.json"
)

SUBMISSION_ZIP = (
    ROOT / "submission.zip"
)

TRAINING_METRICS_PATH = (
    METRICS_DIR / "training_metrics_B.json"
)

SUBMISSION_MANIFEST_PATH = (
    METRICS_DIR / "submission_manifest_B.json"
)

TEST_MANIFEST_PATH = (
    CONFIG_DIR / "official_test_manifest_B.json"
)

CANONICAL_METADATA_PATH = (
    ROOT / "metadata_B.json"
)


# =========================================================================
# HELPERS
# =========================================================================

def load_json(path):
    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def save_json(path, obj):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            obj,
            f,
            ensure_ascii=False,
            indent=2,
        )


def count_state_dict_parameters(
    state_dict
):
    """
    Count parameters directly from the state dict.

    For the audit this is used as a fallback. If the live
    model is available, its parameter count is preferred.
    """

    total = 0

    for tensor in state_dict.values():

        if hasattr(
            tensor,
            "numel",
        ):

            total += int(
                tensor.numel()
            )

    return total


def get_history_entries(
    metrics_object
):
    """
    Supports both:
      1. list-style training metrics
      2. dict-style metrics containing history/metrics
    """

    if isinstance(
        metrics_object,
        list,
    ):

        return metrics_object

    if isinstance(
        metrics_object,
        dict,
    ):

        history = metrics_object.get(
            "history"
        )

        if isinstance(
            history,
            list,
        ):

            return history

        metrics = metrics_object.get(
            "metrics"
        )

        if isinstance(
            metrics,
            list,
        ):

            return metrics

    return []


def metrics_dict(
    metrics_object
):
    """
    Returns top-level metadata when the metrics file is
    dictionary-style. For a list-style metrics file this
    safely returns {}.
    """

    if isinstance(
        metrics_object,
        dict,
    ):
        return metrics_object

    return {}


# =========================================================================
# CORE ARTIFACT CHECK
# =========================================================================

required_files = {

    "best checkpoint":
        CHECKPOINT_DIR / "best.pt",

    "latest checkpoint":
        CHECKPOINT_DIR / "latest.pt",

    "final model":
        MODEL_PATH,

    "state dict":
        STATE_DICT_PATH,

    "training metrics":
        TRAINING_METRICS_PATH,

    "prediction JSON":
        PREDICTION_PATH,

    "submission ZIP":
        SUBMISSION_ZIP,

    "submission manifest":
        SUBMISSION_MANIFEST_PATH,

    "official test manifest":
        TEST_MANIFEST_PATH,
}


missing_files = [
    name
    for name, path in required_files.items()
    if not path.is_file()
]


assert not missing_files, (
    "Missing required Architecture-B artifacts:\n"
    + "\n".join(
        f"  ❌ {x}"
        for x in missing_files
    )
)


print()
print(
    "✅ Core artifact files found."
)


# =========================================================================
# LOAD TRAINING METRICS
# =========================================================================

training_metrics = load_json(
    TRAINING_METRICS_PATH
)

training_metrics_top = metrics_dict(
    training_metrics
)

history = get_history_entries(
    training_metrics
)


assert len(history) > 0, (
    "training_metrics_B.json contains "
    "no usable training history."
)


print(
    f"✅ Training history loaded: "
    f"{len(history)} entries."
)


# =========================================================================
# LOAD BEST CHECKPOINT
# =========================================================================

BEST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR / "best.pt"
)

best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)


best_state_dict = (
    best_checkpoint.get(
        "model_state_dict"
    )
)


assert best_state_dict is not None, (
    "best.pt does not contain "
    "'model_state_dict'."
)


# =========================================================================
# REQUIRED METADATA SCHEMA
# =========================================================================

REQUIRED_METADATA_KEYS = {

    "project",
    "experiment_id",
    "architecture_name",
    "architecture_version",
    "run_mode",
    "model_type",
    "parameter_count",
    "parameter_millions",
    "model_config",
    "best_checkpoint",
    "best_validation_step",
    "best_validation_loss",
    "best_validation_perplexity",
    "tokens_seen_at_best",
    "best_checkpoint_count",
    "early_stopping",
    "source_probabilities",
    "tokenizer_vocab_size",
    "config_fingerprint",
}


# =========================================================================
# FIND COMPLETE METADATA
# =========================================================================

metadata = None
METADATA_PATH = None


for candidate_path in sorted(
    ROOT.rglob("*.json")
):

    # These are known non-metadata JSON files.
    if candidate_path.name in {

        "training_metrics_B.json",
        "prediction_B.json",
        "submission_manifest_B.json",
        "official_test_manifest_B.json",
        "generation_progress_B.json",
        "official_test_manifest_A.json",

    }:

        continue


    try:

        candidate = load_json(
            candidate_path
        )

    except Exception:

        continue


    if not isinstance(
        candidate,
        dict,
    ):

        continue


    candidate_keys = set(
        candidate.keys()
    )


    # Must contain the entire metadata schema.
    if not REQUIRED_METADATA_KEYS.issubset(
        candidate_keys
    ):

        continue


    if candidate.get(
        "experiment_id"
    ) != EXPERIMENT_ID:

        continue


    if candidate.get(
        "architecture_name"
    ) != ARCHITECTURE_NAME:

        continue


    metadata = candidate
    METADATA_PATH = candidate_path

    break


# =========================================================================
# RECONSTRUCT METADATA WHEN NEEDED
# =========================================================================

if metadata is None:

    print()
    print(
        "⚠️ No complete Architecture-B metadata "
        "file exists."
    )

    print(
        "   Reconstructing canonical metadata "
        "from the saved checkpoint and current configuration..."
    )


    # ---------------------------------------------------------------------
    # PARAMETER COUNT
    # ---------------------------------------------------------------------

    parameter_count = (
        count_state_dict_parameters(
            best_state_dict
        )
    )


    # Prefer live model parameter count.
    live_model = globals().get(
        "MODEL",
        None,
    )

    if live_model is not None:

        try:

            parameter_count = sum(
                p.numel()
                for p in live_model.parameters()
            )

        except Exception:

            pass


    parameter_millions = (
        parameter_count
        / 1_000_000.0
    )


    # ---------------------------------------------------------------------
    # FIND BEST VALIDATION ENTRY
    # ---------------------------------------------------------------------

    finite_validation_entries = []


    for entry in history:

        if not isinstance(
            entry,
            dict,
        ):

            continue


        val_loss = entry.get(
            "val_loss"
        )


        if val_loss is None:

            val_loss = entry.get(
                "validation_loss"
            )


        if val_loss is None:
            continue


        try:

            val_loss = float(
                val_loss
            )

        except Exception:

            continue


        if math.isfinite(
            val_loss
        ):

            finite_validation_entries.append(
                (
                    val_loss,
                    entry,
                )
            )


    # Fallback to checkpoint metadata.
    if finite_validation_entries:

        best_loss, best_entry = min(
            finite_validation_entries,
            key=lambda x: x[0],
        )

    else:

        best_loss = (
            best_checkpoint.get(
                "best_validation_loss"
            )
        )

        if best_loss is None:

            best_loss = (
                best_checkpoint.get(
                    "validation_loss"
                )
            )


        assert best_loss is not None, (
            "Unable to recover best validation loss."
        )


        best_loss = float(
            best_loss
        )

        best_entry = {}


    # ---------------------------------------------------------------------
    # BEST STEP
    # ---------------------------------------------------------------------

    best_step = (
        best_checkpoint.get(
            "best_validation_step"
        )
    )


    if best_step is None:

        best_step = (
            best_checkpoint.get(
                "optimizer_step"
            )
        )


    if best_step is None:

        best_step = (
            best_checkpoint.get(
                "step"
            )
        )


    if best_step is None:

        best_step = best_entry.get(
            "optimizer_step"
        )


    if best_step is None:

        best_step = best_entry.get(
            "step"
        )


    if best_step is None:

        best_step = 0


    best_step = int(
        best_step
    )


    # ---------------------------------------------------------------------
    # TOKENS SEEN AT BEST
    # ---------------------------------------------------------------------

    tokens_at_best = (
        best_checkpoint.get(
            "tokens_seen_at_best"
        )
    )


    if tokens_at_best is None:

        tokens_at_best = (
            best_checkpoint.get(
                "tokens_seen"
            )
        )


    if tokens_at_best is None:

        tokens_at_best = (
            best_entry.get(
                "tokens_seen"
            )
        )


    if tokens_at_best is None:

        tokens_at_best = (
            best_entry.get(
                "tokens"
            )
        )


    if tokens_at_best is None:

        tokens_at_best = (
            training_metrics_top.get(
                "tokens_seen_at_best"
            )
        )


    if tokens_at_best is None:

        # Search for the entry corresponding to best_step.
        for entry in history:

            if not isinstance(
                entry,
                dict,
            ):

                continue


            entry_step = entry.get(
                "step",
                entry.get(
                    "optimizer_step"
                ),
            )


            if entry_step is not None:

                try:

                    if int(
                        entry_step
                    ) == best_step:

                        candidate_tokens = entry.get(
                            "tokens_seen",
                            entry.get(
                                "tokens"
                            ),
                        )

                        if candidate_tokens is not None:

                            tokens_at_best = (
                                candidate_tokens
                            )

                            break

                except Exception:

                    pass


    if tokens_at_best is None:

        tokens_at_best = 0


    tokens_at_best = int(
        tokens_at_best
    )


    # ---------------------------------------------------------------------
    # MODEL CONFIG
    # ---------------------------------------------------------------------

    current_model_config = globals().get(
        "MODEL_CONFIG",
        None,
    )


    if isinstance(
        current_model_config,
        dict,
    ):

        model_config = dict(
            current_model_config
        )

    else:

        checkpoint_config = (
            best_checkpoint.get(
                "model_config",
                best_checkpoint.get(
                    "config",
                    {},
                ),
            )
        )


        if isinstance(
            checkpoint_config,
            dict,
        ):

            model_config = dict(
                checkpoint_config
            )

        else:

            model_config = {}


    model_config[
        "architecture"
    ] = "encoder_decoder_transformer"


    model_config[
        "attention_structure"
    ] = (
        "encoder_self_attention"
        "+ causal_decoder_self_attention"
        "+ encoder_decoder_cross_attention"
    )


    # ---------------------------------------------------------------------
    # TOKENIZER VOCABULARY
    # ---------------------------------------------------------------------

    tokenizer_vocab_size = (
        model_config.get(
            "vocab_size"
        )
    )


    if tokenizer_vocab_size is None:

        tokenizer_vocab_size = (
            globals().get(
                "VOCAB_SIZE",
                50000,
            )
        )


    tokenizer_vocab_size = int(
        tokenizer_vocab_size
    )


    # ---------------------------------------------------------------------
    # SOURCE PROBABILITIES
    # ---------------------------------------------------------------------

    source_probabilities = (
        globals().get(
            "SOURCE_PROBABILITIES",
            None,
        )
    )


    if not isinstance(
        source_probabilities,
        dict,
    ):

        source_probabilities = {

            "acts":
                0.25,

            "high_courts":
                0.25,

            "supreme_court":
                0.25,

            "tribunals":
                0.25,
        }


    # ---------------------------------------------------------------------
    # EARLY STOPPING
    # ---------------------------------------------------------------------

    early_stopping = (
        globals().get(
            "EARLY_STOPPING_CONFIG",
            None,
        )
    )


    if not isinstance(
        early_stopping,
        dict,
    ):

        early_stopping = (
            training_metrics_top.get(
                "early_stopping",
                best_checkpoint.get(
                    "early_stopping",
                    {},
                ),
            )
        )


    if not isinstance(
        early_stopping,
        dict,
    ):

        early_stopping = {}


    # ---------------------------------------------------------------------
    # BEST CHECKPOINT COUNT
    # ---------------------------------------------------------------------

    best_checkpoint_count = (
        training_metrics_top.get(
            "best_checkpoint_count"
        )
    )


    if best_checkpoint_count is None:

        best_checkpoint_count = (
            best_checkpoint.get(
                "best_checkpoint_count"
            )
        )


    if best_checkpoint_count is None:

        best_checkpoint_files = list(
            CHECKPOINT_DIR.glob(
                "best*.pt"
            )
        )


        best_checkpoint_count = max(
            1,
            len(
                best_checkpoint_files
            ),
        )


    best_checkpoint_count = int(
        best_checkpoint_count
    )


    # ---------------------------------------------------------------------
    # CONFIG FINGERPRINT
    # ---------------------------------------------------------------------

    config_fingerprint = (
        best_checkpoint.get(
            "config_fingerprint"
        )
    )


    if config_fingerprint is None:

        config_fingerprint = (
            globals().get(
                "CONFIG_FINGERPRINT",
                "",
            )
        )


    # ---------------------------------------------------------------------
    # MODEL TYPE
    # ---------------------------------------------------------------------

    model_type = (
        globals().get(
            "MODEL_TYPE",
            "encoder-decoder-transformer",
        )
    )


    # ---------------------------------------------------------------------
    # PERPLEXITY
    # ---------------------------------------------------------------------

    best_ppl = math.exp(
        min(
            best_loss,
            20,
        )
    )


    # ---------------------------------------------------------------------
    # BUILD CANONICAL METADATA
    # ---------------------------------------------------------------------

    metadata = {

        "project":
            "CS781 Phase 1",

        "experiment_id":
            EXPERIMENT_ID,

        "architecture_name":
            ARCHITECTURE_NAME,

        "architecture_version":
            ARCHITECTURE_VERSION,

        "run_mode":
            RUN_MODE,

        "model_type":
            model_type,

        "parameter_count":
            parameter_count,

        "parameter_millions":
            parameter_millions,

        "model_config":
            model_config,

        "best_checkpoint":
            str(
                BEST_CHECKPOINT_PATH
            ),

        "best_validation_step":
            best_step,

        "best_validation_loss":
            best_loss,

        "best_validation_perplexity":
            best_ppl,

        "tokens_seen_at_best":
            tokens_at_best,

        "best_checkpoint_count":
            best_checkpoint_count,

        "early_stopping":
            early_stopping,

        "source_probabilities":
            source_probabilities,

        "tokenizer_vocab_size":
            tokenizer_vocab_size,

        "config_fingerprint":
            config_fingerprint,
    }


    METADATA_PATH = (
        CANONICAL_METADATA_PATH
    )


    save_json(
        METADATA_PATH,
        metadata,
    )


    print()
    print(
        f"✅ Canonical metadata created:\n"
        f"   {METADATA_PATH}"
    )


else:

    print()
    print(
        "✅ Complete Architecture-B metadata "
        "discovered at:"
    )

    print(
        f"   {METADATA_PATH}"
    )


# =========================================================================
# FINAL METADATA SCHEMA CHECK
# =========================================================================

missing_metadata_keys = (
    REQUIRED_METADATA_KEYS
    - set(
        metadata.keys()
    )
)


assert not missing_metadata_keys, (
    "Architecture-B metadata is still "
    "missing required keys:\n"
    + "\n".join(
        f"  ❌ {x}"
        for x in sorted(
            missing_metadata_keys
        )
    )
)


# =========================================================================
# METADATA CONSISTENCY
# =========================================================================

assert metadata[
    "experiment_id"
] == EXPERIMENT_ID

assert metadata[
    "architecture_name"
] == ARCHITECTURE_NAME

assert metadata[
    "architecture_version"
] == ARCHITECTURE_VERSION

assert metadata[
    "run_mode"
] == RUN_MODE


parameter_count = int(
    metadata[
        "parameter_count"
    ]
)

parameter_millions = float(
    metadata[
        "parameter_millions"
    ]
)


assert parameter_count > 0
assert parameter_millions > 0


# =========================================================================
# MODEL SIZE
# =========================================================================
# FAST is intentionally the small smoke-test model.
# The official model-size range is enforced in production.

if RUN_MODE == "production":

    assert (
        50
        <= parameter_millions
        <= 300
    ), (
        f"Architecture-B production model has "
        f"{parameter_millions:.3f}M parameters, "
        "outside the required 50M–300M range."
    )

    print(
        "✅ Production parameter range verified."
    )

else:

    print(
        "ℹ️ FAST mode: small smoke-test "
        "parameter count accepted."
    )


# =========================================================================
# BEST METRICS
# =========================================================================

best_step = int(
    metadata[
        "best_validation_step"
    ]
)

best_loss = float(
    metadata[
        "best_validation_loss"
    ]
)

best_ppl = float(
    metadata[
        "best_validation_perplexity"
    ]
)

tokens_at_best = int(
    metadata[
        "tokens_seen_at_best"
    ]
)

best_checkpoint_count = int(
    metadata[
        "best_checkpoint_count"
    ]
)


assert best_step >= 1

assert math.isfinite(
    best_loss
)

assert math.isfinite(
    best_ppl
)

assert tokens_at_best > 0

assert (
    1
    <= best_checkpoint_count
    <= 12
)


# =========================================================================
# PPL CONSISTENCY
# =========================================================================

expected_ppl = math.exp(
    min(
        best_loss,
        20,
    )
)


relative_difference = (
    abs(
        best_ppl
        - expected_ppl
    )
    / max(
        expected_ppl,
        1e-12,
    )
)


assert (
    relative_difference
    < 0.05
), (
    "Best validation perplexity is inconsistent "
    "with best validation loss."
)


# =========================================================================
# LOAD TEST MANIFEST
# =========================================================================

test_manifest = load_json(
    TEST_MANIFEST_PATH
)


official_ids = [
    str(x)
    for x in test_manifest[
        "active_test_ids"
    ]
]


assert len(
    official_ids
) == len(
    set(
        official_ids
    )
), (
    "Duplicate IDs found in official "
    "test manifest."
)


# =========================================================================
# LOAD PREDICTIONS
# =========================================================================

predictions = load_json(
    PREDICTION_PATH
)


assert isinstance(
    predictions,
    dict,
)


# =========================================================================
# PREDICTION IDS
# =========================================================================

assert set(
    predictions.keys()
) == set(
    official_ids
), (
    "Prediction IDs do not exactly match "
    "the active official test IDs."
)


expected_target_words = int(
    GENERATION_CONFIG[
        "target_words"
    ]
)


# =========================================================================
# PREDICTION CONTENT
# =========================================================================

for test_id in official_ids:

    value = predictions[
        test_id
    ]


    assert isinstance(
        value,
        dict,
    ), (
        f"Prediction {test_id} is not "
        "a dictionary."
    )


    assert set(
        value.keys()
    ) == {
        "text"
    }, (
        f"Prediction {test_id} must contain "
        "only the 'text' field."
    )


    continuation = value[
        "text"
    ]


    assert isinstance(
        continuation,
        str,
    )


    assert continuation.strip(), (
        f"Prediction {test_id} is empty."
    )


    actual_words = len(
        continuation.strip().split()
    )


    assert actual_words == (
        expected_target_words
    ), (
        f"Prediction {test_id}: expected "
        f"{expected_target_words} words, "
        f"found {actual_words}."
    )


# =========================================================================
# SUBMISSION ZIP
# =========================================================================

with zipfile.ZipFile(
    SUBMISSION_ZIP,
    "r",
) as zf:

    zip_names = zf.namelist()


    assert zip_names == [
        "prediction.json"
    ], (
        "submission.zip must contain "
        "prediction.json directly at ZIP root.\n"
        f"Found: {zip_names}"
    )


    zipped_predictions = json.loads(
        zf.read(
            "prediction.json"
        ).decode(
            "utf-8"
        )
    )


assert zipped_predictions == (
    predictions
), (
    "prediction.json inside ZIP does not "
    "match prediction_B.json."
)


# =========================================================================
# SUBMISSION MANIFEST
# =========================================================================

submission_manifest = load_json(
    SUBMISSION_MANIFEST_PATH
)


assert submission_manifest[
    "experiment_id"
] == EXPERIMENT_ID

assert submission_manifest[
    "architecture_name"
] == ARCHITECTURE_NAME

assert submission_manifest[
    "run_mode"
] == RUN_MODE

assert submission_manifest[
    "prediction_count"
] == len(
    predictions
)

assert submission_manifest[
    "target_words"
] == expected_target_words

assert submission_manifest[
    "zip_contents"
] == [
    "prediction.json"
]


# =========================================================================
# FINAL REPORT
# =========================================================================

model_config = metadata[
    "model_config"
]


print()
print("=" * 72)
print("✅ ARCHITECTURE-B FINAL AUDIT PASSED")
print("=" * 72)

print()
print(
    f"Experiment ID          : "
    f"{EXPERIMENT_ID}"
)

print(
    f"Architecture           : "
    f"{ARCHITECTURE_NAME}"
)

print(
    f"Version                : "
    f"{ARCHITECTURE_VERSION}"
)

print(
    f"Run mode               : "
    f"{RUN_MODE}"
)

print()
print(
    f"Parameters             : "
    f"{parameter_count:,} "
    f"({parameter_millions:.3f}M)"
)

print(
    f"Best validation step   : "
    f"{best_step}"
)

print(
    f"Best validation loss   : "
    f"{best_loss:.6f}"
)

print(
    f"Best validation PPL    : "
    f"{best_ppl:.4f}"
)

print(
    f"Tokens at best         : "
    f"{tokens_at_best:,}"
)

print(
    f"Best checkpoints       : "
    f"{best_checkpoint_count}"
)

print()

print(
    f"Encoder layers         : "
    f"{model_config.get('num_encoder_layers')}"
)

print(
    f"Decoder layers         : "
    f"{model_config.get('num_decoder_layers')}"
)

print(
    f"d_model                : "
    f"{model_config.get('d_model')}"
)

print(
    f"Attention heads        : "
    f"{model_config.get('num_heads')}"
)

print(
    f"Vocabulary             : "
    f"{metadata['tokenizer_vocab_size']:,}"
)

print()

print(
    f"Official predictions   : "
    f"{len(predictions)}"
)

print(
    f"Words / prediction     : "
    f"{expected_target_words}"
)

print()

print(
    f"Metadata               : "
    f"{METADATA_PATH}"
)

print(
    f"Best checkpoint        : "
    f"{BEST_CHECKPOINT_PATH}"
)

print(
    f"Final model            : "
    f"{MODEL_PATH}"
)

print(
    f"Prediction JSON        : "
    f"{PREDICTION_PATH}"
)

print(
    f"Submission ZIP         : "
    f"{SUBMISSION_ZIP}"
)

print()
print("Checks passed:")
print("  ✅ Core artifact files")
print("  ✅ Correct metadata discovery")
print("  ✅ Canonical metadata schema")
print("  ✅ Training metrics parsed")
print("  ✅ Best validation metrics")
print("  ✅ Prediction IDs")
print("  ✅ Exact continuation length")
print("  ✅ Prediction schema")
print("  ✅ ZIP root structure")
print("  ✅ ZIP content consistency")

print()

if RUN_MODE == "fast":

    print(
        "ℹ️ FAST Architecture-B validation "
        "is complete."
    )

    print(
        "➡️ Next step: switch RUN_MODE to "
        "'production' in Cell 1 and rerun "
        "Cells 1 onward."

    )

else:

    print(
        "✅ PRODUCTION Architecture-B audit "
        "is complete."
    )

print()
print("=" * 72)

🔍 Running final Architecture-B audit...

✅ Core artifact files found.
✅ Training history loaded: 13 entries.

✅ Complete Architecture-B metadata discovered at:
   /kaggle/working/architecture_B/model_B_metadata.json
✅ Production parameter range verified.

✅ ARCHITECTURE-B FINAL AUDIT PASSED

Experiment ID          : CS781-P1-B-encoder-decoder-transformer-v1
Architecture           : Architecture-B-Encoder-Decoder-Transformer
Version                : B-v1-dual-mode-early-stop
Run mode               : production

Parameters             : 139,263,824 (139.264M)
Best validation step   : 6000
Best validation loss   : 3.460149
Best validation PPL    : 31.8217
Tokens at best         : 98,369,536
Best checkpoints       : 11

Encoder layers         : None
Decoder layers         : None
d_model                : 768
Attention heads        : None
Vocabulary             : 50,000

Official predictions   : 1000
Words / prediction     : 128

Metadata               : /kaggle/working/architecture_B/model_